# MARS KSDD2 Benchmark — Version 1

## Purpose

This notebook adapts the original MARS FEI benchmark to the Kolektor Surface-Defect Dataset 2 (KSDD2).

The initial objective is to construct a rigorous and reproducible data pipeline before transferring the handcrafted, GP, modified-GP and EOH feature-extraction methods.

The benchmark will:

- use the official KSDD2 training and test folders;
- derive binary image-level labels from the supplied ground-truth masks;
- preserve the official test set as a locked final evaluation set;
- create a stratified internal validation split from the official training set;
- preprocess all images consistently;
- verify dataset integrity and split independence;
- prepare the data for an eight-feature controlled comparison.

## Classification task

The task is binary surface-defect classification:

- `0` — normal image
- `1` — defective image

## Experimental rules

1. The official test set must not be used during model development.
2. Only the official training set may be divided into training and validation subsets.
3. The random seed is fixed at 42.
4. Labels are derived from whether the corresponding ground-truth mask contains any non-zero pixels.
5. All processed images must have the same dimensions and numeric representation.
6. Dataset integrity checks must pass before feature extraction begins.

# 1. Dataset discovery and configuration

This section defines the experiment settings, locates the official KSDD2 folders and verifies the expected image-mask structure.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from skimage import color, io, transform
from sklearn.model_selection import train_test_split

In [2]:
# -------------------------------------------------------
# Reproducibility
# -------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# -------------------------------------------------------
# Classification labels
# -------------------------------------------------------

NORMAL = 0
DEFECTIVE = 1

CLASS_NAMES = {
    NORMAL: "Normal",
    DEFECTIVE: "Defective",
}


# -------------------------------------------------------
# Preprocessing settings
# -------------------------------------------------------

IMAGE_SIZE = (64, 64)
VALIDATION_SIZE = 0.20


# -------------------------------------------------------
# Portable project paths
# -------------------------------------------------------

def find_project_root(start: Path | None = None) -> Path:
    """Locate MARS-Summer-Research without using a machine-specific path."""
    current = Path(start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "src" / "config.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the MARS-Summer-Research repository root. "
        "Run this notebook from somewhere inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, RESULTS_DIR as PROJECT_RESULTS_DIR

# Accept either folder name, preferring the original dataset name.
KSDD2_DATASET_CANDIDATES = [
    RAW_DATA_DIR / "KolektorSDD2",
    RAW_DATA_DIR / "KSDD2",
]

DATASET_ROOT = next(
    (path for path in KSDD2_DATASET_CANDIDATES if path.exists()),
    KSDD2_DATASET_CANDIDATES[0],
)

TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

OUTPUT_DIR = PROJECT_RESULTS_DIR / "ksdd2_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EOH_RESULTS_DIR = PROJECT_RESULTS_DIR / "ksdd2_eoh_8"
EOH_DIAGNOSTICS_DIR = EOH_RESULTS_DIR / "candidate_diagnostics"


# -------------------------------------------------------
# Display configuration
# -------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Configuration loaded.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {RANDOM_STATE}")
print(f"Processed image size: {IMAGE_SIZE}")
print(f"Validation proportion: {VALIDATION_SIZE:.0%}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


Configuration loaded.
Project root: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research
Random seed: 42
Processed image size: (64, 64)
Validation proportion: 20%
Dataset root: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\data\raw\KolektorSDD2
Output directory: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark


In [4]:
def validate_directory(path: Path, description: str) -> None:
    """Raise a clear error when an expected directory is missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"{description} was not found.\n"
            f"Expected path: {path}"
        )

    if not path.is_dir():
        raise NotADirectoryError(
            f"{description} exists but is not a directory:\n{path}"
        )


validate_directory(DATASET_ROOT, "KSDD2 dataset root")
validate_directory(TRAIN_DIR, "KSDD2 training folder")
validate_directory(TEST_DIR, "KSDD2 test folder")

print("Dataset directories located successfully.")
print(f"Training folder: {TRAIN_DIR}")
print(f"Test folder:     {TEST_DIR}")

Dataset directories located successfully.
Training folder: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\data\raw\KolektorSDD2\train
Test folder:     C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\data\raw\KolektorSDD2\test


## 1.1 Image and mask discovery

Each KSDD2 image is expected to have a ground-truth mask with the same numeric identifier and the suffix `_GT`.

Example:

```text
10000.png
10000_GT.png

In [5]:
# -------------------------------------------------------
# Discover valid KSDD2 image-mask pairs
# -------------------------------------------------------

def discover_image_mask_pairs(split_dir: Path) -> pd.DataFrame:
    """
    Discover valid KSDD2 original images and pair them with their exact
    ground-truth masks.

    Valid original image:
        10301.png

    Expected mask:
        10301_GT.png

    Files such as:
        10301_GT.png
        10301_GT (copy).png
        10301 (copy).png

    are ignored as original images.
    """

    image_paths = sorted(
        path
        for path in split_dir.glob("*.png")
        if path.stem.isdigit()
    )

    records = []

    for image_path in image_paths:
        image_id = image_path.stem
        mask_path = split_dir / f"{image_id}_GT.png"

        records.append(
            {
                "split": split_dir.name,
                "image_id": image_id,
                "image_path": image_path,
                "mask_path": mask_path,
                "mask_exists": mask_path.is_file(),
            }
        )

    return pd.DataFrame(
        records,
        columns=[
            "split",
            "image_id",
            "image_path",
            "mask_path",
            "mask_exists",
        ],
    )


# Build the official split manifests
train_manifest = discover_image_mask_pairs(TRAIN_DIR)
test_manifest = discover_image_mask_pairs(TEST_DIR)


# Summary
print("=" * 60)
print("KSDD2 Dataset Discovery")
print("=" * 60)

print(f"Training images discovered: {len(train_manifest)}")
print(f"Test images discovered:     {len(test_manifest)}")

print(
    f"Training masks missing: "
    f"{int((~train_manifest['mask_exists']).sum())}"
)

print(
    f"Test masks missing:     "
    f"{int((~test_manifest['mask_exists']).sum())}"
)

print("\nFirst five training pairs:")
display(train_manifest.head())

KSDD2 Dataset Discovery
Training images discovered: 2331
Test images discovered:     1004
Training masks missing: 0
Test masks missing:     0

First five training pairs:


,split,image_id,image_path,mask_path,mask_exists
0,train,10000,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...,True
1,train,10001,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...,True
2,train,10002,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...,True
3,train,10003,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...,True
4,train,10004,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...,True


In [6]:
def validate_manifest(manifest: pd.DataFrame, split_name: str) -> None:
    """Validate that the discovered image-mask manifest is usable."""
    if manifest.empty:
        raise ValueError(
            f"No original PNG images were found in the {split_name} split."
        )

    missing_masks = manifest.loc[
        ~manifest["mask_exists"],
        ["image_id", "image_path", "mask_path"],
    ]

    duplicate_ids = manifest[
        manifest["image_id"].duplicated(keep=False)
    ]

    if not missing_masks.empty:
        raise FileNotFoundError(
            f"{len(missing_masks)} images in the {split_name} split "
            "do not have matching masks.\n"
            f"{missing_masks.head()}"
        )

    if not duplicate_ids.empty:
        raise ValueError(
            f"Duplicate image identifiers were found in the "
            f"{split_name} split.\n"
            f"{duplicate_ids.head()}"
        )

    print(
        f"{split_name.capitalize()} manifest passed: "
        f"{len(manifest)} complete image-mask pairs."
    )


validate_manifest(train_manifest, "training")
validate_manifest(test_manifest, "test")

Training manifest passed: 2331 complete image-mask pairs.
Test manifest passed: 1004 complete image-mask pairs.


In [7]:
train_ids = set(train_manifest["image_id"])
test_ids = set(test_manifest["image_id"])

overlapping_ids = train_ids.intersection(test_ids)

assert not overlapping_ids, (
    "The official training and test sets contain overlapping image IDs: "
    f"{sorted(overlapping_ids)[:10]}"
)

print("Official training and test image IDs are disjoint.")

Official training and test image IDs are disjoint.


In [8]:
def inspect_raw_pair(row: pd.Series) -> dict:
    """Read one image-mask pair and return its basic properties."""
    image = io.imread(row["image_path"])
    mask = io.imread(row["mask_path"])

    return {
        "image_id": row["image_id"],
        "image_shape": image.shape,
        "image_dtype": str(image.dtype),
        "image_min": float(np.min(image)),
        "image_max": float(np.max(image)),
        "mask_shape": mask.shape,
        "mask_dtype": str(mask.dtype),
        "mask_min": float(np.min(mask)),
        "mask_max": float(np.max(mask)),
        "mask_nonzero_pixels": int(np.count_nonzero(mask)),
    }


sample_records = []

for _, row in train_manifest.head(5).iterrows():
    sample_records.append(inspect_raw_pair(row))

raw_sample_summary = pd.DataFrame(sample_records)

display(raw_sample_summary)

,image_id,image_shape,image_dtype,image_min,image_max,mask_shape,mask_dtype,mask_min,mask_max,mask_nonzero_pixels
0,10000,"(645, 229, 3)",uint8,10.0,226.0,"(645, 229)",uint8,0.0,0.0,0
1,10001,"(633, 228, 3)",uint8,6.0,175.0,"(633, 228)",uint8,0.0,0.0,0
2,10002,"(647, 230, 3)",uint8,7.0,184.0,"(647, 230)",uint8,0.0,0.0,0
3,10003,"(634, 229, 3)",uint8,13.0,212.0,"(634, 229)",uint8,0.0,0.0,0
4,10004,"(632, 228, 3)",uint8,6.0,255.0,"(632, 228)",uint8,0.0,0.0,0


# 2. Loading and preprocessing

This section converts the raw KSDD2 images into a consistent numerical representation suitable for the benchmark methods.

Each image will be:

- converted from RGB to grayscale;
- resized to 64 × 64 pixels;
- converted to `float32`;
- normalised to the range `[0, 1]`.

Each binary label will be derived from the corresponding ground-truth mask:

- `0` — normal;
- `1` — defective.

In [9]:
# -------------------------------------------------------
# Preprocess one KSDD2 image
# -------------------------------------------------------

def preprocess_image(
    image_path: Path,
    image_size: tuple[int, int] = IMAGE_SIZE,
) -> np.ndarray:
    """
    Load and preprocess one KSDD2 image.

    Processing steps:
    1. Read the PNG image.
    2. Convert RGB/RGBA images to grayscale.
    3. Resize to the configured image size.
    4. Convert to float32.
    5. Ensure values lie within [0, 1].

    Parameters
    ----------
    image_path:
        Path to the original KSDD2 image.

    image_size:
        Output shape as (height, width).

    Returns
    -------
    np.ndarray
        A two-dimensional float32 grayscale image.
    """

    image = io.imread(image_path)

    if image.ndim == 3:
        if image.shape[-1] == 4:
            image = color.rgba2rgb(image)

        image = color.rgb2gray(image)

    elif image.ndim != 2:
        raise ValueError(
            f"Unsupported image shape for {image_path.name}: "
            f"{image.shape}"
        )

    image = transform.resize(
        image,
        image_size,
        anti_aliasing=True,
        preserve_range=False,
    )

    image = image.astype(np.float32)

    if not np.isfinite(image).all():
        raise ValueError(
            f"Non-finite values found after preprocessing "
            f"{image_path.name}"
        )

    image = np.clip(image, 0.0, 1.0)

    return image

In [10]:
# -------------------------------------------------------
# Derive binary label from one ground-truth mask
# -------------------------------------------------------

def derive_label(mask_path: Path) -> int:
    """
    Derive an image-level binary label from a KSDD2 mask.

    A mask containing at least one non-zero pixel is labelled defective.

    Returns
    -------
    int
        NORMAL (0) or DEFECTIVE (1).
    """

    mask = io.imread(mask_path)

    if mask.ndim == 3:
        mask = np.any(mask > 0, axis=-1)

    label = DEFECTIVE if np.any(mask > 0) else NORMAL

    return label

In [11]:
# -------------------------------------------------------
# Test preprocessing and label derivation
# -------------------------------------------------------

sample_row = train_manifest.iloc[0]

sample_image = preprocess_image(sample_row["image_path"])
sample_label = derive_label(sample_row["mask_path"])

print("Sample image ID:", sample_row["image_id"])
print("Processed shape:", sample_image.shape)
print("Processed dtype:", sample_image.dtype)
print("Processed minimum:", float(sample_image.min()))
print("Processed maximum:", float(sample_image.max()))
print("Derived label:", sample_label)
print("Class name:", CLASS_NAMES[sample_label])

assert sample_image.shape == IMAGE_SIZE
assert sample_image.dtype == np.float32
assert np.isfinite(sample_image).all()
assert 0.0 <= sample_image.min() <= sample_image.max() <= 1.0
assert sample_label in {NORMAL, DEFECTIVE}

print("\nSingle-image preprocessing checks passed.")

Sample image ID: 10000
Processed shape: (64, 64)
Processed dtype: float32
Processed minimum: 0.0727311372756958
Processed maximum: 0.23488593101501465
Derived label: 0
Class name: Normal

Single-image preprocessing checks passed.


In [12]:
# -------------------------------------------------------
# Find and test one defective training image
# -------------------------------------------------------

def find_first_defective_row(
    manifest: pd.DataFrame,
) -> pd.Series:
    """Return the first manifest row whose mask contains a defect."""

    for _, row in manifest.iterrows():
        if derive_label(row["mask_path"]) == DEFECTIVE:
            return row

    raise ValueError("No defective images were found in the manifest.")


defective_row = find_first_defective_row(train_manifest)

defective_image = preprocess_image(defective_row["image_path"])
defective_label = derive_label(defective_row["mask_path"])

print("Defective image ID:", defective_row["image_id"])
print("Processed shape:", defective_image.shape)
print("Processed dtype:", defective_image.dtype)
print("Derived label:", defective_label)
print("Class name:", CLASS_NAMES[defective_label])

assert defective_image.shape == IMAGE_SIZE
assert defective_image.dtype == np.float32
assert defective_label == DEFECTIVE

print("\nDefective-image preprocessing checks passed.")

Defective image ID: 10021
Processed shape: (64, 64)
Processed dtype: float32
Derived label: 1
Class name: Defective

Defective-image preprocessing checks passed.


## 2.1 Load complete dataset splits

The validated preprocessing and label functions are now applied to every image in the official KSDD2 training and test folders.

The loader retains image identifiers and file paths so that individual predictions can later be traced back to their original files.

In [14]:
# -------------------------------------------------------
# Load a complete KSDD2 split
# -------------------------------------------------------

def load_ksdd2_split(
    manifest: pd.DataFrame,
    image_size: tuple[int, int] = IMAGE_SIZE,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Load and preprocess every image-mask pair in a KSDD2 manifest.

    Parameters
    ----------
    manifest:
        DataFrame containing image IDs, image paths and mask paths.

    image_size:
        Output image shape as (height, width).

    Returns
    -------
    images:
        Array with shape (n_samples, height, width), dtype float32.

    labels:
        Binary label array with shape (n_samples,), dtype int64.

    metadata:
        DataFrame retaining file identities and derived labels.
    """

    if manifest.empty:
        raise ValueError("The supplied manifest is empty.")

    images = []
    labels = []
    metadata_records = []

    total = len(manifest)

    for position, (_, row) in enumerate(manifest.iterrows(), start=1):
        image = preprocess_image(
            row["image_path"],
            image_size=image_size,
        )

        label = derive_label(row["mask_path"])

        images.append(image)
        labels.append(label)

        metadata_records.append(
            {
                "split": row["split"],
                "image_id": row["image_id"],
                "image_path": row["image_path"],
                "mask_path": row["mask_path"],
                "label": label,
                "class_name": CLASS_NAMES[label],
            }
        )

        if position % 250 == 0 or position == total:
            print(f"Loaded {position:>4} / {total} images")

    images_array = np.stack(images).astype(np.float32)
    labels_array = np.asarray(labels, dtype=np.int64)
    metadata = pd.DataFrame(metadata_records)

    return images_array, labels_array, metadata

In [15]:
# -------------------------------------------------------
# Load official KSDD2 test data
# -------------------------------------------------------

X_official_test, y_official_test, official_test_metadata = (
    load_ksdd2_split(test_manifest)
)

print("\nOfficial test set loaded.")
print("Image array shape:", X_official_test.shape)
print("Label array shape:", y_official_test.shape)
print("Image dtype:", X_official_test.dtype)
print("Label dtype:", y_official_test.dtype)

Loaded  250 / 1004 images
Loaded  500 / 1004 images
Loaded  750 / 1004 images
Loaded 1000 / 1004 images
Loaded 1004 / 1004 images

Official test set loaded.
Image array shape: (1004, 64, 64)
Label array shape: (1004,)
Image dtype: float32
Label dtype: int64


In [18]:
# -------------------------------------------------------
# Load official KSDD2 training and test data
# -------------------------------------------------------

X_official_train, y_official_train, official_train_metadata = (
    load_ksdd2_split(train_manifest)
)

print("Official training set loaded.")
print("Image array shape:", X_official_train.shape)
print("Label array shape:", y_official_train.shape)

print()

X_official_test, y_official_test, official_test_metadata = (
    load_ksdd2_split(test_manifest)
)

print("Official test set loaded.")
print("Image array shape:", X_official_test.shape)
print("Label array shape:", y_official_test.shape)

Loaded  250 / 2331 images
Loaded  500 / 2331 images
Loaded  750 / 2331 images
Loaded 1000 / 2331 images
Loaded 1250 / 2331 images
Loaded 1500 / 2331 images
Loaded 1750 / 2331 images
Loaded 2000 / 2331 images
Loaded 2250 / 2331 images
Loaded 2331 / 2331 images
Official training set loaded.
Image array shape: (2331, 64, 64)
Label array shape: (2331,)

Loaded  250 / 1004 images
Loaded  500 / 1004 images
Loaded  750 / 1004 images
Loaded 1000 / 1004 images
Loaded 1004 / 1004 images
Official test set loaded.
Image array shape: (1004, 64, 64)
Label array shape: (1004,)


In [19]:
# -------------------------------------------------------
# Validate loaded dataset arrays
# -------------------------------------------------------

def validate_loaded_split(
    images: np.ndarray,
    labels: np.ndarray,
    metadata: pd.DataFrame,
    expected_size: tuple[int, int],
    split_name: str,
) -> None:
    """Run consistency and integrity checks on one loaded split."""

    if images.ndim != 3:
        raise ValueError(
            f"{split_name}: expected a three-dimensional image array, "
            f"received shape {images.shape}."
        )

    expected_shape = (
        len(labels),
        expected_size[0],
        expected_size[1],
    )

    if images.shape != expected_shape:
        raise ValueError(
            f"{split_name}: expected image shape {expected_shape}, "
            f"received {images.shape}."
        )

    if len(metadata) != len(labels):
        raise ValueError(
            f"{split_name}: metadata and label counts do not match."
        )

    if images.dtype != np.float32:
        raise TypeError(
            f"{split_name}: expected float32 images, "
            f"received {images.dtype}."
        )

    if labels.dtype != np.int64:
        raise TypeError(
            f"{split_name}: expected int64 labels, "
            f"received {labels.dtype}."
        )

    if not np.isfinite(images).all():
        raise ValueError(
            f"{split_name}: non-finite image values were found."
        )

    if images.min() < 0.0 or images.max() > 1.0:
        raise ValueError(
            f"{split_name}: processed values fall outside [0, 1]."
        )

    unique_labels = set(np.unique(labels).tolist())

    if not unique_labels.issubset({NORMAL, DEFECTIVE}):
        raise ValueError(
            f"{split_name}: unexpected labels found: {unique_labels}"
        )

    if metadata["image_id"].duplicated().any():
        raise ValueError(
            f"{split_name}: duplicate image IDs found in metadata."
        )

    print(f"{split_name} validation passed.")
    print(f"  Samples: {len(labels)}")
    print(f"  Shape: {images.shape}")
    print(f"  Value range: [{images.min():.4f}, {images.max():.4f}]")
    print(f"  Labels: {sorted(unique_labels)}")


validate_loaded_split(
    X_official_train,
    y_official_train,
    official_train_metadata,
    IMAGE_SIZE,
    "Official training set",
)

print()

validate_loaded_split(
    X_official_test,
    y_official_test,
    official_test_metadata,
    IMAGE_SIZE,
    "Official test set",
)

Official training set validation passed.
  Samples: 2331
  Shape: (2331, 64, 64)
  Value range: [0.0163, 0.9563]
  Labels: [0, 1]

Official test set validation passed.
  Samples: 1004
  Shape: (1004, 64, 64)
  Value range: [0.0080, 0.9865]
  Labels: [0, 1]


In [20]:
# -------------------------------------------------------
# Display class distributions
# -------------------------------------------------------

def class_distribution_table(
    labels: np.ndarray,
    split_name: str,
) -> pd.DataFrame:
    """Create a readable class-count table."""

    values, counts = np.unique(labels, return_counts=True)

    records = []

    for value, count in zip(values, counts):
        records.append(
            {
                "split": split_name,
                "label": int(value),
                "class_name": CLASS_NAMES[int(value)],
                "count": int(count),
                "proportion": float(count / len(labels)),
            }
        )

    return pd.DataFrame(records)


distribution_summary = pd.concat(
    [
        class_distribution_table(
            y_official_train,
            "Official training",
        ),
        class_distribution_table(
            y_official_test,
            "Official test",
        ),
    ],
    ignore_index=True,
)

display(distribution_summary)

,split,label,class_name,count,proportion
0,Official training,0,Normal,2085,0.894466
1,Official training,1,Defective,246,0.105534
2,Official test,0,Normal,894,0.890438
3,Official test,1,Defective,110,0.109562


# 3. Controlled train, validation and test setup

The official KSDD2 test set remains locked and is not used during model development.

The official training set is divided into:

- an internal training subset;
- an internal validation subset.

The split is stratified so that the proportion of defective images is preserved. The split is also tied to the fixed random seed, saved to disk and checked for overlap or duplicate content.

In [21]:
# -------------------------------------------------------
# Create reproducible stratified development split
# -------------------------------------------------------

official_train_indices = np.arange(len(y_official_train))

train_indices, validation_indices = train_test_split(
    official_train_indices,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_official_train,
    shuffle=True,
)

train_indices = np.sort(train_indices)
validation_indices = np.sort(validation_indices)

X_train = X_official_train[train_indices]
y_train = y_official_train[train_indices]

X_validation = X_official_train[validation_indices]
y_validation = y_official_train[validation_indices]

train_metadata = (
    official_train_metadata.iloc[train_indices]
    .copy()
    .reset_index(drop=True)
)

validation_metadata = (
    official_train_metadata.iloc[validation_indices]
    .copy()
    .reset_index(drop=True)
)

# The official test set remains unchanged.
X_test = X_official_test
y_test = y_official_test
test_metadata = official_test_metadata.copy().reset_index(drop=True)

print("Controlled split created.")
print(f"Internal training samples: {len(y_train)}")
print(f"Validation samples:        {len(y_validation)}")
print(f"Locked test samples:       {len(y_test)}")

Controlled split created.
Internal training samples: 1864
Validation samples:        467
Locked test samples:       1004


In [22]:
# -------------------------------------------------------
# Validate split shapes and metadata alignment
# -------------------------------------------------------

def validate_split_alignment(
    images: np.ndarray,
    labels: np.ndarray,
    metadata: pd.DataFrame,
    split_name: str,
) -> None:
    """Check that images, labels and metadata remain aligned."""

    n_samples = len(labels)

    assert images.shape == (
        n_samples,
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
    ), (
        f"{split_name}: unexpected image shape "
        f"{images.shape}"
    )

    assert len(metadata) == n_samples, (
        f"{split_name}: metadata length does not match "
        "the number of labels."
    )

    assert images.dtype == np.float32, (
        f"{split_name}: images must be float32."
    )

    assert labels.dtype == np.int64, (
        f"{split_name}: labels must be int64."
    )

    assert np.isfinite(images).all(), (
        f"{split_name}: non-finite image values found."
    )

    assert set(np.unique(labels)).issubset(
        {NORMAL, DEFECTIVE}
    ), (
        f"{split_name}: unexpected labels found."
    )

    metadata_labels = metadata["label"].to_numpy(
        dtype=np.int64
    )

    assert np.array_equal(labels, metadata_labels), (
        f"{split_name}: metadata labels are misaligned."
    )

    assert not metadata["image_id"].duplicated().any(), (
        f"{split_name}: duplicate image IDs found."
    )

    print(
        f"{split_name} alignment passed: "
        f"{n_samples} samples."
    )


validate_split_alignment(
    X_train,
    y_train,
    train_metadata,
    "Internal training",
)

validate_split_alignment(
    X_validation,
    y_validation,
    validation_metadata,
    "Validation",
)

validate_split_alignment(
    X_test,
    y_test,
    test_metadata,
    "Locked official test",
)

Internal training alignment passed: 1864 samples.
Validation alignment passed: 467 samples.
Locked official test alignment passed: 1004 samples.


In [23]:
# -------------------------------------------------------
# Compare class distributions across all splits
# -------------------------------------------------------

split_distribution_summary = pd.concat(
    [
        class_distribution_table(
            y_train,
            "Internal training",
        ),
        class_distribution_table(
            y_validation,
            "Validation",
        ),
        class_distribution_table(
            y_test,
            "Locked official test",
        ),
    ],
    ignore_index=True,
)

display(split_distribution_summary)

,split,label,class_name,count,proportion
0,Internal training,0,Normal,1667,0.894313
1,Internal training,1,Defective,197,0.105687
2,Validation,0,Normal,418,0.895075
3,Validation,1,Defective,49,0.104925
4,Locked official test,0,Normal,894,0.890438
5,Locked official test,1,Defective,110,0.109562


In [24]:
# -------------------------------------------------------
# Verify split identities are disjoint
# -------------------------------------------------------

train_ids = set(train_metadata["image_id"])
validation_ids = set(validation_metadata["image_id"])
test_ids = set(test_metadata["image_id"])

train_validation_overlap = train_ids & validation_ids
train_test_overlap = train_ids & test_ids
validation_test_overlap = validation_ids & test_ids

assert not train_validation_overlap, (
    "Image IDs overlap between training and validation."
)

assert not train_test_overlap, (
    "Image IDs overlap between training and test."
)

assert not validation_test_overlap, (
    "Image IDs overlap between validation and test."
)

assert len(train_ids | validation_ids) == len(
    official_train_metadata
), (
    "The internal training and validation subsets do not "
    "reconstruct the full official training set."
)

print("Split identity checks passed.")
print("No image IDs overlap across splits.")
print(
    "Internal training and validation together reconstruct "
    "the official training set."
)

Split identity checks passed.
No image IDs overlap across splits.
Internal training and validation together reconstruct the official training set.


In [25]:
# -------------------------------------------------------
# Verify deterministic split reproduction
# -------------------------------------------------------

reproduced_train_indices, reproduced_validation_indices = (
    train_test_split(
        official_train_indices,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_official_train,
        shuffle=True,
    )
)

reproduced_train_indices = np.sort(
    reproduced_train_indices
)

reproduced_validation_indices = np.sort(
    reproduced_validation_indices
)

assert np.array_equal(
    train_indices,
    reproduced_train_indices,
), "Training indices could not be reproduced."

assert np.array_equal(
    validation_indices,
    reproduced_validation_indices,
), "Validation indices could not be reproduced."

print(
    "Split reproduced exactly using the stored "
    "configuration."
)

Split reproduced exactly using the stored configuration.


In [26]:
# -------------------------------------------------------
# Check exact processed-image duplicates across splits
# -------------------------------------------------------

def image_content_hash(image: np.ndarray) -> str:
    """Return a SHA-256 hash for one processed image."""
    contiguous_image = np.ascontiguousarray(image)
    return hashlib.sha256(
        contiguous_image.tobytes()
    ).hexdigest()


def build_hash_set(images: np.ndarray) -> set[str]:
    """Build a set of content hashes for an image array."""
    return {
        image_content_hash(image)
        for image in images
    }


train_hashes = build_hash_set(X_train)
validation_hashes = build_hash_set(X_validation)
test_hashes = build_hash_set(X_test)

train_validation_duplicates = (
    train_hashes & validation_hashes
)

train_test_duplicates = train_hashes & test_hashes

validation_test_duplicates = (
    validation_hashes & test_hashes
)

print(
    "Exact processed duplicates — "
    f"train/validation: "
    f"{len(train_validation_duplicates)}"
)

print(
    "Exact processed duplicates — "
    f"train/test: {len(train_test_duplicates)}"
)

print(
    "Exact processed duplicates — "
    f"validation/test: "
    f"{len(validation_test_duplicates)}"
)

Exact processed duplicates — train/validation: 0
Exact processed duplicates — train/test: 0
Exact processed duplicates — validation/test: 0


In [27]:
# -------------------------------------------------------
# Create consolidated split manifest
# -------------------------------------------------------

train_split_manifest = train_metadata.copy()
train_split_manifest["benchmark_split"] = "train"
train_split_manifest["official_source"] = "train"

validation_split_manifest = validation_metadata.copy()
validation_split_manifest["benchmark_split"] = "validation"
validation_split_manifest["official_source"] = "train"

test_split_manifest = test_metadata.copy()
test_split_manifest["benchmark_split"] = "test"
test_split_manifest["official_source"] = "test"

split_manifest = pd.concat(
    [
        train_split_manifest,
        validation_split_manifest,
        test_split_manifest,
    ],
    ignore_index=True,
)

split_manifest = split_manifest[
    [
        "benchmark_split",
        "official_source",
        "image_id",
        "label",
        "class_name",
        "image_path",
        "mask_path",
    ]
]

display(split_manifest.head())
print(
    f"Rows in consolidated split manifest: "
    f"{len(split_manifest)}"
)

,benchmark_split,official_source,image_id,label,class_name,image_path,mask_path
0,train,train,10000,0,Normal,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...
1,train,train,10002,0,Normal,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...
2,train,train,10003,0,Normal,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...
3,train,train,10004,0,Normal,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...
4,train,train,10005,0,Normal,C:\Users\james\OneDrive - Lancaster University...,C:\Users\james\OneDrive - Lancaster University...


Rows in consolidated split manifest: 3335


In [28]:
# -------------------------------------------------------
# Save split manifest and experiment configuration
# -------------------------------------------------------

SPLIT_MANIFEST_PATH = (
    OUTPUT_DIR / "ksdd2_split_manifest.csv"
)

CONFIGURATION_PATH = (
    OUTPUT_DIR / "ksdd2_data_configuration.json"
)

split_manifest.to_csv(
    SPLIT_MANIFEST_PATH,
    index=False,
)

data_configuration = {
    "dataset": "KolektorSDD2",
    "benchmark_version": "v1",
    "classification_task": (
        "binary surface-defect classification"
    ),
    "normal_label": NORMAL,
    "defective_label": DEFECTIVE,
    "random_state": RANDOM_STATE,
    "image_size": list(IMAGE_SIZE),
    "validation_size": VALIDATION_SIZE,
    "official_training_samples": int(
        len(y_official_train)
    ),
    "internal_training_samples": int(len(y_train)),
    "validation_samples": int(len(y_validation)),
    "official_test_samples": int(len(y_test)),
    "test_set_locked": True,
    "python_version": sys.version,
    "platform": platform.platform(),
}

with CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        data_configuration,
        file,
        indent=2,
    )

print("Saved split manifest:")
print(SPLIT_MANIFEST_PATH)

print("\nSaved data configuration:")
print(CONFIGURATION_PATH)

Saved split manifest:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\ksdd2_split_manifest.csv

Saved data configuration:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\ksdd2_data_configuration.json


## 3.1 Test-set lock

The official KSDD2 test set is now designated as locked.

During model development:

- no feature extractor may be selected using test performance;
- no classifier hyperparameter may be selected using test performance;
- no feature definition may be changed after inspecting test predictions;
- validation results must guide all development decisions.

The official test set will be evaluated only after the benchmark methods and evaluation rules have been finalised.

In [29]:
# -------------------------------------------------------
# Explicit test-set lock
# -------------------------------------------------------

TEST_SET_LOCKED = True

assert TEST_SET_LOCKED is True
assert len(y_test) == len(y_official_test)
assert np.array_equal(y_test, y_official_test)
assert np.array_equal(X_test, X_official_test)

print("Official KSDD2 test set is locked.")
print(
    "Do not use X_test or y_test for feature selection, "
    "model selection or hyperparameter tuning."
)

Official KSDD2 test set is locked.
Do not use X_test or y_test for feature selection, model selection or hyperparameter tuning.


# 4. Shared feature-extraction and classification pipeline

All dimension-controlled benchmark methods must pass through the same evaluation pipeline.

Each method must:

- accept one preprocessed 64 × 64 grayscale image;
- return exactly eight finite scalar features;
- use the same training and validation images;
- use a scaler fitted only on training features;
- use the same linear SVM classifier;
- remain isolated from the locked official test set during development.

Because KSDD2 is strongly class-imbalanced, ordinary accuracy is supplemented by:

- balanced accuracy;
- macro F1 score;
- defective-class precision;
- defective-class recall;
- confusion matrix.

Inference timing is also recorded to support later real-time feasibility analysis.

In [30]:
# -------------------------------------------------------
# Shared benchmark imports and settings
# -------------------------------------------------------

import time

from typing import Any, Callable

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# -------------------------------------------------------
# Shared benchmark configuration
# -------------------------------------------------------

N_FEATURES = 8

SVM_KERNEL = "linear"
SVM_C = 1.0
SVM_CLASS_WEIGHT = "balanced"

LOCK_TEST_SET = TEST_SET_LOCKED

DATASET_NAME = "KolektorSDD2"
DATA_SOURCE_TYPE = "industrial_surface_images"

print("=" * 70)
print("Shared KSDD2 benchmark configuration")
print("=" * 70)
print(f"Dataset              : {DATASET_NAME}")
print(f"Target features      : {N_FEATURES}")
print(f"SVM kernel           : {SVM_KERNEL}")
print(f"SVM C                : {SVM_C}")
print(f"SVM class weighting  : {SVM_CLASS_WEIGHT}")
print(f"Random state         : {RANDOM_STATE}")
print(f"Test set locked      : {LOCK_TEST_SET}")
print("=" * 70)

Shared KSDD2 benchmark configuration
Dataset              : KolektorSDD2
Target features      : 8
SVM kernel           : linear
SVM C                : 1.0
SVM class weighting  : balanced
Random state         : 42
Test set locked      : True


In [31]:
# -------------------------------------------------------
# Validate one extracted feature vector
# -------------------------------------------------------

def validate_feature_vector(
    values: Any,
    *,
    image_index: int,
    expected_features: int | None,
    split_name: str,
) -> np.ndarray:
    """
    Validate the output of one feature extractor call.

    Dimension-controlled methods must return exactly eight finite
    scalar values.
    """

    vector = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    if vector.size == 0:
        raise ValueError(
            f"{split_name} image {image_index} "
            "returned no features."
        )

    if (
        expected_features is not None
        and vector.shape != (expected_features,)
    ):
        raise ValueError(
            f"{split_name} image {image_index} returned "
            f"shape {vector.shape}; expected "
            f"({expected_features},)."
        )

    if not np.all(np.isfinite(vector)):
        raise ValueError(
            f"{split_name} image {image_index} returned "
            "NaN or infinite feature values."
        )

    return vector

In [32]:
# -------------------------------------------------------
# Build feature matrix
# -------------------------------------------------------

def build_feature_matrix(
    images: np.ndarray,
    feature_extractor: Callable[[np.ndarray], np.ndarray],
    *,
    split_name: str,
    expected_features: int | None = N_FEATURES,
) -> tuple[np.ndarray, dict[str, float]]:
    """
    Apply one feature extractor to an image array.

    Returns
    -------
    feature_matrix:
        Array with shape (n_samples, n_features).

    timing:
        Dictionary containing total and per-image feature
        extraction timing.
    """

    image_array = np.asarray(images)

    if image_array.ndim != 3:
        raise ValueError(
            f"{split_name}: expected image array with shape "
            f"(n, height, width), received {image_array.shape}."
        )

    if len(image_array) == 0:
        raise ValueError(
            f"{split_name}: no images were supplied."
        )

    rows = []
    inferred_count = expected_features

    started = time.perf_counter()

    for image_index, image in enumerate(image_array):
        raw_features = feature_extractor(image)

        vector = validate_feature_vector(
            raw_features,
            image_index=image_index,
            expected_features=inferred_count,
            split_name=split_name,
        )

        if inferred_count is None:
            inferred_count = vector.size

        if vector.shape != (inferred_count,):
            raise ValueError(
                f"{split_name}: inconsistent feature "
                f"dimension at image {image_index}."
            )

        rows.append(vector)

    total_seconds = time.perf_counter() - started

    feature_matrix = np.vstack(rows).astype(
        np.float64,
        copy=False,
    )

    if not np.all(np.isfinite(feature_matrix)):
        raise ValueError(
            f"{split_name}: feature matrix contains "
            "non-finite values."
        )

    timing = {
        "total_seconds": float(total_seconds),
        "seconds_per_image": float(
            total_seconds / len(image_array)
        ),
        "milliseconds_per_image": float(
            1000.0 * total_seconds / len(image_array)
        ),
    }

    return feature_matrix, timing

In [33]:
# -------------------------------------------------------
# Validate training feature matrix
# -------------------------------------------------------

def validate_training_feature_matrix(
    features: np.ndarray,
    *,
    expected_features: int = N_FEATURES,
) -> None:
    """
    Run integrity checks on the training feature matrix.
    """

    matrix = np.asarray(
        features,
        dtype=np.float64,
    )

    if matrix.ndim != 2:
        raise ValueError(
            f"Expected a 2D training feature matrix, "
            f"received {matrix.shape}."
        )

    if matrix.shape[1] != expected_features:
        raise ValueError(
            f"Expected {expected_features} features, "
            f"received {matrix.shape[1]}."
        )

    if not np.all(np.isfinite(matrix)):
        raise ValueError(
            "Training feature matrix contains "
            "non-finite values."
        )

    standard_deviations = np.std(
        matrix,
        axis=0,
    )

    constant_columns = np.where(
        np.isclose(
            standard_deviations,
            0.0,
        )
    )[0]

    if len(constant_columns) > 0:
        raise ValueError(
            "Constant training features detected at "
            f"indices: {constant_columns.tolist()}"
        )

    print(
        "Training feature matrix passed validation: "
        f"{matrix.shape}"
    )

In [34]:
# -------------------------------------------------------
# Binary classification metrics
# -------------------------------------------------------

def calculate_binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, Any]:
    """
    Calculate KSDD2 classification metrics.

    DEFECTIVE = 1 is treated as the positive class.
    """

    true_labels = np.asarray(
        y_true,
        dtype=np.int64,
    )

    predictions = np.asarray(
        y_pred,
        dtype=np.int64,
    )

    if true_labels.shape != predictions.shape:
        raise ValueError(
            "True labels and predictions have "
            "different shapes."
        )

    matrix = confusion_matrix(
        true_labels,
        predictions,
        labels=[NORMAL, DEFECTIVE],
    )

    tn, fp, fn, tp = matrix.ravel()

    metrics = {
        "accuracy": float(
            accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "macro_f1": float(
            f1_score(
                true_labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "defect_precision": float(
            precision_score(
                true_labels,
                predictions,
                pos_label=DEFECTIVE,
                zero_division=0,
            )
        ),
        "defect_recall": float(
            recall_score(
                true_labels,
                predictions,
                pos_label=DEFECTIVE,
                zero_division=0,
            )
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "confusion_matrix": matrix,
    }

    return metrics

In [35]:
# -------------------------------------------------------
# Prediction timing
# -------------------------------------------------------

def measure_prediction_time(
    classifier,
    scaled_features: np.ndarray,
    *,
    warmup_samples: int = 20,
    repetitions: int = 5,
) -> dict[str, float]:
    """
    Measure steady-state classifier prediction time.

    Timing covers SVM prediction only. Feature extraction timing
    is recorded separately.
    """

    features = np.asarray(
        scaled_features,
        dtype=np.float64,
    )

    if len(features) == 0:
        raise ValueError(
            "No features supplied for timing."
        )

    warmup_count = min(
        warmup_samples,
        len(features),
    )

    if warmup_count > 0:
        classifier.predict(
            features[:warmup_count]
        )

    repetition_times = []

    for _ in range(repetitions):
        started = time.perf_counter()

        classifier.predict(features)

        elapsed = (
            time.perf_counter()
            - started
        )

        repetition_times.append(elapsed)

    mean_total_seconds = float(
        np.mean(repetition_times)
    )

    seconds_per_image = (
        mean_total_seconds
        / len(features)
    )

    return {
        "mean_total_seconds": mean_total_seconds,
        "seconds_per_image": float(
            seconds_per_image
        ),
        "milliseconds_per_image": float(
            seconds_per_image * 1000.0
        ),
    }

In [37]:
# -------------------------------------------------------
# Common feature-extractor evaluator
# -------------------------------------------------------

def evaluate_feature_extractor(
    feature_extractor: Callable[[np.ndarray], np.ndarray],
    *,
    method_name: str,
    expected_features: int | None = N_FEATURES,
    comparison_group: str = "dimension_controlled",
    evaluate_test: bool = False,
) -> dict[str, Any]:
    """
    Evaluate one feature extractor using the common KSDD2 pipeline.

    During development, evaluate_test must remain False.
    """

    if evaluate_test and LOCK_TEST_SET:
        raise PermissionError(
            "The official KSDD2 test set is locked. "
            "Test evaluation is not permitted during "
            "model development."
        )

    # ---------------------------------------------------
    # Training feature extraction
    # ---------------------------------------------------

    train_features, train_feature_timing = (
        build_feature_matrix(
            X_train,
            feature_extractor,
            split_name="training",
            expected_features=expected_features,
        )
    )

    # ---------------------------------------------------
    # Validation feature extraction
    # ---------------------------------------------------

    (
        validation_features,
        validation_feature_timing,
    ) = build_feature_matrix(
        X_validation,
        feature_extractor,
        split_name="validation",
        expected_features=train_features.shape[1],
    )

    validate_training_feature_matrix(
        train_features,
        expected_features=train_features.shape[1],
    )

    # ---------------------------------------------------
    # Training-only feature scaling
    # ---------------------------------------------------

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_features
    )

    validation_scaled = scaler.transform(
        validation_features
    )

    # ---------------------------------------------------
    # Common classifier
    # ---------------------------------------------------

    classifier = SVC(
        kernel=SVM_KERNEL,
        C=SVM_C,
        class_weight=SVM_CLASS_WEIGHT,
        random_state=RANDOM_STATE,
    )

    training_started = time.perf_counter()

    classifier.fit(
        train_scaled,
        y_train,
    )

    classifier_training_time = (
        time.perf_counter()
        - training_started
    )

    # ---------------------------------------------------
    # Validation prediction
    # ---------------------------------------------------

    validation_predictions = classifier.predict(
        validation_scaled
    )

    validation_metrics = (
        calculate_binary_metrics(
            y_validation,
            validation_predictions,
        )
    )

    validation_prediction_timing = (
        measure_prediction_time(
            classifier,
            validation_scaled,
        )
    )

    # ---------------------------------------------------
    # End-to-end inference estimate
    # ---------------------------------------------------

    feature_ms = (
        validation_feature_timing[
            "milliseconds_per_image"
        ]
    )

    prediction_ms = (
        validation_prediction_timing[
            "milliseconds_per_image"
        ]
    )

    estimated_inference_ms = (
        feature_ms + prediction_ms
    )

    estimated_fps = (
        1000.0 / estimated_inference_ms
        if estimated_inference_ms > 0.0
        else float("inf")
    )

    # ---------------------------------------------------
    # Store result
    # ---------------------------------------------------

    result = {
        "method": method_name,
        "comparison_group": comparison_group,
        "evaluation_pipeline": (
            "common_ksdd2_benchmark_pipeline"
        ),
        "dataset_name": DATASET_NAME,
        "data_source_type": DATA_SOURCE_TYPE,

        "feature_dimension": int(
            train_features.shape[1]
        ),
        "expected_feature_dimension": (
            expected_features
        ),

        "image_size": IMAGE_SIZE,
        "random_state": RANDOM_STATE,

        "svm_kernel": SVM_KERNEL,
        "svm_c": SVM_C,
        "svm_class_weight": (
            SVM_CLASS_WEIGHT
        ),

        "training_feature_shape": (
            train_features.shape
        ),
        "validation_feature_shape": (
            validation_features.shape
        ),
        "test_feature_shape": None,

        "validation_accuracy": (
            validation_metrics["accuracy"]
        ),
        "validation_balanced_accuracy": (
            validation_metrics[
                "balanced_accuracy"
            ]
        ),
        "validation_macro_f1": (
            validation_metrics[
                "macro_f1"
            ]
        ),
        "validation_defect_precision": (
            validation_metrics[
                "defect_precision"
            ]
        ),
        "validation_defect_recall": (
            validation_metrics[
                "defect_recall"
            ]
        ),

        "validation_confusion_matrix": (
            validation_metrics[
                "confusion_matrix"
            ]
        ),

        "test_accuracy": None,
        "test_balanced_accuracy": None,
        "test_macro_f1": None,
        "test_defect_precision": None,
        "test_defect_recall": None,
        "test_evaluated": False,

        "training_feature_extraction_time_seconds": (
            train_feature_timing[
                "total_seconds"
            ]
        ),

        "validation_feature_extraction_time_seconds": (
            validation_feature_timing[
                "total_seconds"
            ]
        ),

        "validation_feature_extraction_ms_per_image": (
            feature_ms
        ),

        "classifier_training_time_seconds": float(
            classifier_training_time
        ),

        "validation_prediction_ms_per_image": (
            prediction_ms
        ),

        "estimated_inference_ms_per_image": (
            estimated_inference_ms
        ),

        "estimated_fps": float(
            estimated_fps
        ),

        "scaler": scaler,
        "classifier": classifier,

        "training_features": train_features,
        "validation_features": validation_features,
        "test_features": None,

        "validation_predictions": (
            validation_predictions
        ),

        "test_predictions": None,
    }

    # ---------------------------------------------------
    # Optional final test evaluation
    # ---------------------------------------------------

    if evaluate_test:

        test_features, test_feature_timing = (
            build_feature_matrix(
                X_test,
                feature_extractor,
                split_name="test",
                expected_features=(
                    train_features.shape[1]
                ),
            )
        )

        test_scaled = scaler.transform(
            test_features
        )

        test_predictions = classifier.predict(
            test_scaled
        )

        test_metrics = calculate_binary_metrics(
            y_test,
            test_predictions,
        )

        result.update(
            {
                "test_feature_shape": (
                    test_features.shape
                ),
                "test_accuracy": (
                    test_metrics[
                        "accuracy"
                    ]
                ),
                "test_balanced_accuracy": (
                    test_metrics[
                        "balanced_accuracy"
                    ]
                ),
                "test_macro_f1": (
                    test_metrics[
                        "macro_f1"
                    ]
                ),
                "test_defect_precision": (
                    test_metrics[
                        "defect_precision"
                    ]
                ),
                "test_defect_recall": (
                    test_metrics[
                        "defect_recall"
                    ]
                ),
                "test_evaluated": True,
                "test_features": (
                    test_features
                ),
                "test_predictions": (
                    test_predictions
                ),
                "test_feature_extraction_time_seconds": (
                    test_feature_timing[
                        "total_seconds"
                    ]
                ),
            }
        )

    # ---------------------------------------------------
    # Console summary
    # ---------------------------------------------------

    print("=" * 70)
    print(f"Method              : {method_name}")
    print(
        f"Comparison group    : "
        f"{comparison_group}"
    )
    print(
        f"Feature dimension   : "
        f"{result['feature_dimension']}"
    )
    print(
        f"Accuracy            : "
        f"{result['validation_accuracy'] * 100:.2f}%"
    )
    print(
        f"Balanced accuracy   : "
        f"{result['validation_balanced_accuracy'] * 100:.2f}%"
    )
    print(
        f"Macro F1            : "
        f"{result['validation_macro_f1']:.4f}"
    )
    print(
        f"Defect precision    : "
        f"{result['validation_defect_precision']:.4f}"
    )
    print(
        f"Defect recall       : "
        f"{result['validation_defect_recall']:.4f}"
    )
    print(
        f"Inference estimate  : "
        f"{result['estimated_inference_ms_per_image']:.4f} ms/image"
    )
    print(
        f"Estimated FPS       : "
        f"{result['estimated_fps']:.2f}"
    )
    print(
        f"Test evaluated      : "
        f"{result['test_evaluated']}"
    )
    print("=" * 70)

    return result

In [38]:
# -------------------------------------------------------
# JSON-safe benchmark result summary
# -------------------------------------------------------

def summarise_benchmark_result(
    result: dict[str, Any],
) -> dict[str, Any]:
    """
    Return a compact JSON-serialisable summary.
    """

    return {
        "method": result["method"],
        "comparison_group": (
            result["comparison_group"]
        ),
        "evaluation_pipeline": (
            result["evaluation_pipeline"]
        ),
        "dataset_name": (
            result["dataset_name"]
        ),
        "feature_dimension": (
            result["feature_dimension"]
        ),

        "validation_accuracy": (
            result["validation_accuracy"]
        ),
        "validation_balanced_accuracy": (
            result[
                "validation_balanced_accuracy"
            ]
        ),
        "validation_macro_f1": (
            result["validation_macro_f1"]
        ),
        "validation_defect_precision": (
            result[
                "validation_defect_precision"
            ]
        ),
        "validation_defect_recall": (
            result[
                "validation_defect_recall"
            ]
        ),

        "classifier_training_time_seconds": (
            result[
                "classifier_training_time_seconds"
            ]
        ),

        "validation_feature_extraction_ms_per_image": (
            result[
                "validation_feature_extraction_ms_per_image"
            ]
        ),

        "validation_prediction_ms_per_image": (
            result[
                "validation_prediction_ms_per_image"
            ]
        ),

        "estimated_inference_ms_per_image": (
            result[
                "estimated_inference_ms_per_image"
            ]
        ),

        "estimated_fps": (
            result["estimated_fps"]
        ),

        "test_accuracy": (
            result["test_accuracy"]
        ),
        "test_evaluated": (
            result["test_evaluated"]
        ),
    }

In [40]:
# -------------------------------------------------------
# Shared pipeline smoke test
# -------------------------------------------------------

def temporary_test_extractor(
    image: np.ndarray,
) -> np.ndarray:
    """
    Simple deterministic eight-feature extractor used only
    to verify the shared benchmark infrastructure.
    """

    image = np.asarray(
        image,
        dtype=np.float64,
    )

    dx = np.diff(
        image,
        axis=1,
    )

    dy = np.diff(
        image,
        axis=0,
    )

    return np.array(
        [
            np.mean(image),
            np.std(image),
            np.min(image),
            np.max(image),
            np.median(image),
            np.mean(np.abs(dx)),
            np.mean(np.abs(dy)),
            np.mean(
                image > np.mean(image)
            ),
        ],
        dtype=np.float64,
    )


smoke_test_vector = validate_feature_vector(
    temporary_test_extractor(
        X_train[0]
    ),
    image_index=0,
    expected_features=N_FEATURES,
    split_name="pipeline smoke test",
)

print(
    "Smoke-test feature vector shape:",
    smoke_test_vector.shape,
)

print(
    "Smoke-test feature values:"
)

print(smoke_test_vector)

print(
    "\nPASS: shared feature-vector contract works."
)

Smoke-test feature vector shape: (8,)
Smoke-test feature values:
[0.15586495 0.0222256  0.07273114 0.23488593 0.1574562  0.01760342
 0.01142915 0.53515625]

PASS: shared feature-vector contract works.


In [41]:
# -------------------------------------------------------
# Verify test-set protection
# -------------------------------------------------------

try:
    evaluate_feature_extractor(
        temporary_test_extractor,
        method_name="Lock Test",
        evaluate_test=True,
    )

except PermissionError as error:
    print("PASS: test-set lock is working.")
    print(error)

else:
    raise RuntimeError(
        "Test-set lock failed."
    )

PASS: test-set lock is working.
The official KSDD2 test set is locked. Test evaluation is not permitted during model development.


# 5. Handcrafted Eight-Feature Baseline

This section defines a deterministic eight-feature baseline designed for the KSDD2 surface-defect classification task.

The baseline is intentionally simple and interpretable. It captures:

- global intensity behaviour;
- intensity variability;
- horizontal and vertical edge strength;
- local high-frequency variation;
- dark-pixel prevalence;
- bright-pixel prevalence;
- Laplacian response variability.

The same eight scalar features are extracted from every preprocessed 64 × 64 grayscale image.

The resulting feature vectors are evaluated using the shared KSDD2 benchmark pipeline defined in Section 4.

In [42]:
# -------------------------------------------------------
# Handcrafted feature definitions
# -------------------------------------------------------

HANDCRAFTED_FEATURE_METADATA = [
    {
        "index": 0,
        "name": "global_mean",
        "description": (
            "Mean grayscale intensity across the full image."
        ),
    },
    {
        "index": 1,
        "name": "global_standard_deviation",
        "description": (
            "Standard deviation of full-image intensity."
        ),
    },
    {
        "index": 2,
        "name": "horizontal_gradient_mean",
        "description": (
            "Mean absolute horizontal pixel difference."
        ),
    },
    {
        "index": 3,
        "name": "vertical_gradient_mean",
        "description": (
            "Mean absolute vertical pixel difference."
        ),
    },
    {
        "index": 4,
        "name": "local_difference_variability",
        "description": (
            "Standard deviation of neighbouring horizontal "
            "and vertical differences."
        ),
    },
    {
        "index": 5,
        "name": "dark_pixel_proportion",
        "description": (
            "Proportion of pixels below the image mean minus "
            "one standard deviation."
        ),
    },
    {
        "index": 6,
        "name": "bright_pixel_proportion",
        "description": (
            "Proportion of pixels above the image mean plus "
            "one standard deviation."
        ),
    },
    {
        "index": 7,
        "name": "laplacian_variance",
        "description": (
            "Variance of the Laplacian response, representing "
            "high-frequency edge and texture variation."
        ),
    },
]


HANDCRAFTED_FEATURE_NAMES = [
    item["name"]
    for item in HANDCRAFTED_FEATURE_METADATA
]


if len(HANDCRAFTED_FEATURE_NAMES) != N_FEATURES:
    raise ValueError(
        "Handcrafted metadata must contain exactly "
        f"{N_FEATURES} features."
    )


if len(set(HANDCRAFTED_FEATURE_NAMES)) != N_FEATURES:
    raise ValueError(
        "Handcrafted feature names must be unique."
    )


display(
    pd.DataFrame(
        HANDCRAFTED_FEATURE_METADATA
    )
)

,index,name,description
0,0,global_mean,Mean grayscale intensity across the full image.
1,1,global_standard_deviation,Standard deviation of full-image intensity.
2,2,horizontal_gradient_mean,Mean absolute horizontal pixel difference.
3,3,vertical_gradient_mean,Mean absolute vertical pixel difference.
4,4,local_difference_variability,Standard deviation of neighbouring horizontal ...
5,5,dark_pixel_proportion,Proportion of pixels below the image mean minu...
6,6,bright_pixel_proportion,Proportion of pixels above the image mean plus...
7,7,laplacian_variance,"Variance of the Laplacian response, representi..."


In [43]:
# -------------------------------------------------------
# Additional numerical operator for handcrafted features
# -------------------------------------------------------

from scipy import ndimage

print("scipy.ndimage imported successfully.")

scipy.ndimage imported successfully.


In [44]:
# -------------------------------------------------------
# KSDD2 handcrafted eight-feature extractor
# -------------------------------------------------------

def handcrafted_feature_extractor(
    image: np.ndarray,
) -> np.ndarray:
    """
    Extract eight deterministic scalar features from one
    preprocessed KSDD2 image.
    """

    image = np.asarray(
        image,
        dtype=np.float64,
    )

    if image.ndim != 2:
        raise ValueError(
            "Handcrafted extractor requires a "
            "two-dimensional grayscale image."
        )

    if image.shape != IMAGE_SIZE:
        raise ValueError(
            f"Expected image shape {IMAGE_SIZE}, "
            f"received {image.shape}."
        )

    if not np.all(np.isfinite(image)):
        raise ValueError(
            "Handcrafted extractor received "
            "non-finite pixel values."
        )

    # ---------------------------------------------------
    # Global intensity statistics
    # ---------------------------------------------------

    global_mean = float(
        np.mean(image)
    )

    global_std = float(
        np.std(image)
    )

    # ---------------------------------------------------
    # First-order directional differences
    # ---------------------------------------------------

    horizontal_difference = np.diff(
        image,
        axis=1,
    )

    vertical_difference = np.diff(
        image,
        axis=0,
    )

    horizontal_gradient_mean = float(
        np.mean(
            np.abs(
                horizontal_difference
            )
        )
    )

    vertical_gradient_mean = float(
        np.mean(
            np.abs(
                vertical_difference
            )
        )
    )

    # ---------------------------------------------------
    # Local variation
    # ---------------------------------------------------

    combined_differences = np.concatenate(
        [
            horizontal_difference.reshape(-1),
            vertical_difference.reshape(-1),
        ]
    )

    local_difference_variability = float(
        np.std(combined_differences)
    )

    # ---------------------------------------------------
    # Extreme intensity proportions
    # ---------------------------------------------------

    dark_threshold = (
        global_mean - global_std
    )

    bright_threshold = (
        global_mean + global_std
    )

    dark_pixel_proportion = float(
        np.mean(
            image < dark_threshold
        )
    )

    bright_pixel_proportion = float(
        np.mean(
            image > bright_threshold
        )
    )

    # ---------------------------------------------------
    # High-frequency texture / edge response
    # ---------------------------------------------------

    laplacian_response = ndimage.laplace(
        image
    )

    laplacian_variance = float(
        np.var(
            laplacian_response
        )
    )

    # ---------------------------------------------------
    # Final feature vector
    # ---------------------------------------------------

    features = np.array(
        [
            global_mean,
            global_std,
            horizontal_gradient_mean,
            vertical_gradient_mean,
            local_difference_variability,
            dark_pixel_proportion,
            bright_pixel_proportion,
            laplacian_variance,
        ],
        dtype=np.float64,
    )

    return validate_feature_vector(
        features,
        image_index=0,
        expected_features=N_FEATURES,
        split_name=(
            "KSDD2 handcrafted extractor"
        ),
    )

In [45]:
# -------------------------------------------------------
# Verify handcrafted extractor contract
# -------------------------------------------------------

handcrafted_sample = (
    handcrafted_feature_extractor(
        X_train[0]
    )
)

print("=" * 70)
print("Handcrafted-8 sample feature vector")
print("=" * 70)

for feature_name, feature_value in zip(
    HANDCRAFTED_FEATURE_NAMES,
    handcrafted_sample,
):
    print(
        f"{feature_name:<35}: "
        f"{feature_value:.8f}"
    )

print("-" * 70)
print(
    "Output shape:",
    handcrafted_sample.shape,
)

assert handcrafted_sample.shape == (
    N_FEATURES,
)

assert np.all(
    np.isfinite(
        handcrafted_sample
    )
)

print(
    "PASS: Handcrafted-8 extractor "
    "contract verified."
)

Handcrafted-8 sample feature vector
global_mean                        : 0.15586495
global_standard_deviation          : 0.02222560
horizontal_gradient_mean           : 0.01760342
vertical_gradient_mean             : 0.01142915
local_difference_variability       : 0.01880729
dark_pixel_proportion              : 0.17358398
bright_pixel_proportion            : 0.15283203
laplacian_variance                 : 0.00202647
----------------------------------------------------------------------
Output shape: (8,)
PASS: Handcrafted-8 extractor contract verified.


In [46]:
# -------------------------------------------------------
# Compare feature values for one normal and one defect
# -------------------------------------------------------

normal_index = np.where(
    y_train == NORMAL
)[0][0]

defective_index = np.where(
    y_train == DEFECTIVE
)[0][0]


normal_features = (
    handcrafted_feature_extractor(
        X_train[normal_index]
    )
)

defective_features = (
    handcrafted_feature_extractor(
        X_train[defective_index]
    )
)


feature_comparison = pd.DataFrame(
    {
        "feature": (
            HANDCRAFTED_FEATURE_NAMES
        ),
        "normal_sample": (
            normal_features
        ),
        "defective_sample": (
            defective_features
        ),
        "absolute_difference": np.abs(
            normal_features
            - defective_features
        ),
    }
)


display(feature_comparison)

,feature,normal_sample,defective_sample,absolute_difference
0,global_mean,0.155865,0.158335,0.002470
1,global_standard_deviation,0.022226,0.023324,0.001098
2,horizontal_gradient_mean,0.017603,0.009771,0.007833
3,vertical_gradient_mean,0.011429,0.009998,0.001431
4,local_difference_variability,0.018807,0.012744,0.006064
5,dark_pixel_proportion,0.173584,0.103760,0.069824
6,bright_pixel_proportion,0.152832,0.087646,0.065186
7,laplacian_variance,0.002026,0.001095,0.000931


In [47]:
# -------------------------------------------------------
# Evaluate Handcrafted-8
# -------------------------------------------------------

handcrafted_result = (
    evaluate_feature_extractor(
        feature_extractor=(
            handcrafted_feature_extractor
        ),
        method_name="Handcrafted-8",
        expected_features=N_FEATURES,
        comparison_group=(
            "dimension_controlled"
        ),
        evaluate_test=False,
    )
)

Training feature matrix passed validation: (1864, 8)
Method              : Handcrafted-8
Comparison group    : dimension_controlled
Feature dimension   : 8
Accuracy            : 91.22%
Balanced accuracy   : 83.39%
Macro F1            : 0.7936
Defect precision    : 0.5625
Defect recall       : 0.7347
Inference estimate  : 0.1572 ms/image
Estimated FPS       : 6363.09
Test evaluated      : False


In [48]:
# -------------------------------------------------------
# Handcrafted-8 validation confusion matrix
# -------------------------------------------------------

handcrafted_confusion_matrix = (
    handcrafted_result[
        "validation_confusion_matrix"
    ]
)


confusion_table = pd.DataFrame(
    handcrafted_confusion_matrix,
    index=[
        "Actual Normal",
        "Actual Defective",
    ],
    columns=[
        "Predicted Normal",
        "Predicted Defective",
    ],
)


display(confusion_table)

,Predicted Normal,Predicted Defective
Actual Normal,390,28
Actual Defective,13,36


In [49]:
# -------------------------------------------------------
# Handcrafted-8 validation summary
# -------------------------------------------------------

handcrafted_summary_table = pd.DataFrame(
    [
        {
            "Method": "Handcrafted-8",
            "Features": (
                handcrafted_result[
                    "feature_dimension"
                ]
            ),
            "Accuracy": (
                handcrafted_result[
                    "validation_accuracy"
                ]
            ),
            "Balanced Accuracy": (
                handcrafted_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                handcrafted_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                handcrafted_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                handcrafted_result[
                    "validation_defect_recall"
                ]
            ),
            "Feature Extraction ms/image": (
                handcrafted_result[
                    "validation_feature_extraction_ms_per_image"
                ]
            ),
            "Prediction ms/image": (
                handcrafted_result[
                    "validation_prediction_ms_per_image"
                ]
            ),
            "Total Inference ms/image": (
                handcrafted_result[
                    "estimated_inference_ms_per_image"
                ]
            ),
            "Estimated FPS": (
                handcrafted_result[
                    "estimated_fps"
                ]
            ),
        }
    ]
)


display(
    handcrafted_summary_table
)

,Method,Features,Accuracy,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall,Feature Extraction ms/image,Prediction ms/image,Total Inference ms/image,Estimated FPS
0,Handcrafted-8,8,0.912206,0.833854,0.793615,0.5625,0.734694,0.148705,0.008451,0.157156,6363.086146


In [50]:
# -------------------------------------------------------
# Save Handcrafted-8 metadata and results
# -------------------------------------------------------

HANDCRAFTED_METADATA_FILE = (
    OUTPUT_DIR
    / "handcrafted8_feature_metadata.json"
)

HANDCRAFTED_SUMMARY_FILE = (
    OUTPUT_DIR
    / "handcrafted8_validation_summary.json"
)


with HANDCRAFTED_METADATA_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        HANDCRAFTED_FEATURE_METADATA,
        file,
        indent=2,
    )


with HANDCRAFTED_SUMMARY_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        summarise_benchmark_result(
            handcrafted_result
        ),
        file,
        indent=2,
    )


print(
    "Saved handcrafted metadata:"
)
print(
    HANDCRAFTED_METADATA_FILE
)

print()

print(
    "Saved handcrafted validation summary:"
)
print(
    HANDCRAFTED_SUMMARY_FILE
)

Saved handcrafted metadata:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\handcrafted8_feature_metadata.json

Saved handcrafted validation summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\handcrafted8_validation_summary.json


# 6. Original Genetic Programming — Unrestricted Reference

This section integrates the existing Original GP-FR implementation with the KSDD2 benchmark.

The GP algorithm receives:

- the benchmark internal training split;
- the benchmark validation split;
- no access to the locked official test set.

The Original GP search remains unrestricted in feature dimensionality. Therefore, this method is treated as a high-dimensional reference rather than part of the controlled eight-feature comparison.

The feature representation discovered by GP is subsequently evaluated using the shared KSDD2 classifier settings:

- `StandardScaler` fitted only on training features;
- linear SVM;
- `class_weight="balanced"`;
- the same validation metrics used by Handcrafted-8.

The official KSDD2 test set remains locked.

In [51]:
# -------------------------------------------------------
# Original GP package loading utilities
# -------------------------------------------------------

import importlib
import inspect
import pickle
import sys


ORIGINAL_GP_PARENT = (
    PROJECT_ROOT
    / "src"
    / "legacy"
    / "gp_original"
)

ORIGINAL_GP_PACKAGE_DIR = (
    ORIGINAL_GP_PARENT
    / "gp_fr"
)


def unload_gp_fr() -> None:
    """
    Remove any previously imported gp_fr modules.

    This prevents the Original and Modified GP packages
    from colliding in the same Python kernel.
    """

    for module_name in list(sys.modules):
        if (
            module_name == "gp_fr"
            or module_name.startswith("gp_fr.")
        ):
            del sys.modules[module_name]


def load_gp_fr(
    package_parent: Path,
):
    """
    Load gp_fr from one explicitly selected parent folder.
    """

    package_parent = (
        Path(package_parent)
        .resolve()
    )

    if not package_parent.exists():
        raise FileNotFoundError(
            "GP package parent directory not found:\n"
            f"{package_parent}"
        )

    package_directory = (
        package_parent
        / "gp_fr"
    )

    if not package_directory.exists():
        raise FileNotFoundError(
            "No gp_fr package found inside:\n"
            f"{package_parent}"
        )

    unload_gp_fr()

    # Remove prior GP package directories from sys.path.
    # This prevents the legacy Original/Modified/Restricted packages
    # from colliding when the notebook switches between them.
    legacy_root = (
        PROJECT_ROOT
        / "src"
        / "legacy"
    ).resolve()

    cleaned_sys_path = []

    for path in sys.path:
        try:
            resolved_path = Path(path).resolve()
        except (TypeError, OSError):
            cleaned_sys_path.append(path)
            continue

        if resolved_path == legacy_root or legacy_root in resolved_path.parents:
            continue

        cleaned_sys_path.append(path)

    sys.path = cleaned_sys_path

    sys.path.insert(
        0,
        str(package_parent),
    )

    importlib.invalidate_caches()

    return importlib.import_module(
        "gp_fr"
    )

In [52]:
# -------------------------------------------------------
# Load and verify Original GP
# -------------------------------------------------------

if not ORIGINAL_GP_PACKAGE_DIR.exists():
    raise FileNotFoundError(
        "Original GP package not found:\n"
        f"{ORIGINAL_GP_PACKAGE_DIR}"
    )


original_gp_fr = load_gp_fr(
    ORIGINAL_GP_PARENT
)


original_gp_module_path = Path(
    original_gp_fr.__file__
).resolve()


if (
    ORIGINAL_GP_PARENT.resolve()
    not in original_gp_module_path.parents
):
    raise ImportError(
        "The wrong gp_fr package was imported:\n"
        f"{original_gp_module_path}"
    )


print("=" * 70)
print("Original GP package verification")
print("=" * 70)

print(
    "Loaded from:",
    original_gp_module_path,
)

print(
    "\nrun_gp_fr signature:"
)

print(
    inspect.signature(
        original_gp_fr.run_gp_fr
    )
)

print(
    "\nOriginal GP configuration:"
)

print(
    "Population:",
    original_gp_fr.gp_fr_main.POPULATION,
)

print(
    "Generations:",
    original_gp_fr.gp_fr_main.GENERATION,
)

print("=" * 70)

Original GP package verification
Loaded from: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_original\gp_fr\__init__.py

run_gp_fr signature:
(x_train, y_train, x_test, y_test, seed=0, verbose=True)

Original GP configuration:
Population: 250
Generations: 10


## 6.1 GP integration smoke test

Before launching the full evolutionary experiment, a small integration run is performed.

The smoke test uses:

- a small population;
- a small number of generations;
- the real KSDD2 training and validation arrays.

Its purpose is only to verify that the Original GP implementation can successfully process the KSDD2 data.

Smoke-test results are not treated as final experimental results.

In [53]:
# -------------------------------------------------------
# Original GP smoke-test settings
# -------------------------------------------------------

ORIGINAL_GP_DEFAULT_POPULATION = (
    original_gp_fr.gp_fr_main.POPULATION
)

ORIGINAL_GP_DEFAULT_GENERATIONS = (
    original_gp_fr.gp_fr_main.GENERATION
)


ORIGINAL_GP_SMOKE_POPULATION = 10
ORIGINAL_GP_SMOKE_GENERATIONS = 1


print(
    "Default population:",
    ORIGINAL_GP_DEFAULT_POPULATION,
)

print(
    "Default generations:",
    ORIGINAL_GP_DEFAULT_GENERATIONS,
)

print(
    "Smoke population:",
    ORIGINAL_GP_SMOKE_POPULATION,
)

print(
    "Smoke generations:",
    ORIGINAL_GP_SMOKE_GENERATIONS,
)

Default population: 250
Default generations: 10
Smoke population: 10
Smoke generations: 1


In [54]:
# -------------------------------------------------------
# Execute Original GP integration smoke test
# -------------------------------------------------------

original_gp_fr.gp_fr_main.POPULATION = (
    ORIGINAL_GP_SMOKE_POPULATION
)

original_gp_fr.gp_fr_main.GENERATION = (
    ORIGINAL_GP_SMOKE_GENERATIONS
)


smoke_started = time.perf_counter()


try:

    original_gp_smoke_result = (
        original_gp_fr.run_gp_fr(
            x_train=X_train,
            y_train=y_train,
            x_test=X_validation,
            y_test=y_validation,
            seed=RANDOM_STATE,
            verbose=True,
        )
    )

finally:

    # Always restore the source experiment settings.
    original_gp_fr.gp_fr_main.POPULATION = (
        ORIGINAL_GP_DEFAULT_POPULATION
    )

    original_gp_fr.gp_fr_main.GENERATION = (
        ORIGINAL_GP_DEFAULT_GENERATIONS
    )


original_gp_smoke_wall_time = (
    time.perf_counter()
    - smoke_started
)


print(
    "\nSmoke-test wall time:",
    f"{original_gp_smoke_wall_time:.2f} seconds",
)

print(
    "Default GP settings restored:",
    original_gp_fr.gp_fr_main.POPULATION,
    "population /",
    original_gp_fr.gp_fr_main.GENERATION,
    "generations",
)

gen	nevals	avg   	max  
0  	10    	89.984	94.26
1  	9     	94.117	94.42

=== GP-FR Results (seed=42) ===
Training time: 190.0s
Best train fitness: 94.42%
Test accuracy: 96.79% (classifier: RF)
Feature dim: 200
Program: FC4(Hist(GauD(Gau(Max(SobelX(Image)), 2), 1, 1, 0)), uLBP_R(LoG1_R(Sobel_R(Sobel_R(Mean_R(Mean_R(LoG2_R(Max_R(LoG2_R(Gau_R(Min_R(GauD_R(Mean_R(Max_R(GauD_R(Gau_R(Min_R(LoG1_R(RegionS(Image, 42, 49, 60))), 2), 1, 1, 2))), 3, 0, 2)), 2)))))))))), uLBP_R(Mean_R(SobelX_R(Lap_R(RegionS(Image, 0, 49, 40))))), DIF_R(Min_R(SobelY_R(LoG1_R(LoG2_R(SobelY_R(Lap_R(LoG2_R(Med_R(Mean_R(Mean_R(SobelY_R(Min_R(LoG2_R(Med_R(RegionS(GauD(Image, 1, 1, 0), 13, 42, 22)))))))))))))))))

Smoke-test wall time: 202.96 seconds
Default GP settings restored: 250 population / 10 generations


In [55]:
# -------------------------------------------------------
# Execute Original GP integration smoke test
# -------------------------------------------------------

original_gp_fr.gp_fr_main.POPULATION = (
    ORIGINAL_GP_SMOKE_POPULATION
)

original_gp_fr.gp_fr_main.GENERATION = (
    ORIGINAL_GP_SMOKE_GENERATIONS
)


smoke_started = time.perf_counter()


try:

    original_gp_smoke_result = (
        original_gp_fr.run_gp_fr(
            x_train=X_train,
            y_train=y_train,
            x_test=X_validation,
            y_test=y_validation,
            seed=RANDOM_STATE,
            verbose=True,
        )
    )

finally:

    # Always restore the source experiment settings.
    original_gp_fr.gp_fr_main.POPULATION = (
        ORIGINAL_GP_DEFAULT_POPULATION
    )

    original_gp_fr.gp_fr_main.GENERATION = (
        ORIGINAL_GP_DEFAULT_GENERATIONS
    )


original_gp_smoke_wall_time = (
    time.perf_counter()
    - smoke_started
)


print(
    "\nSmoke-test wall time:",
    f"{original_gp_smoke_wall_time:.2f} seconds",
)

print(
    "Default GP settings restored:",
    original_gp_fr.gp_fr_main.POPULATION,
    "population /",
    original_gp_fr.gp_fr_main.GENERATION,
    "generations",
)

gen	nevals	avg   	max  
0  	10    	89.984	94.26
1  	9     	94.117	94.42

=== GP-FR Results (seed=42) ===
Training time: 81.5s
Best train fitness: 94.42%
Test accuracy: 96.79% (classifier: RF)
Feature dim: 200
Program: FC4(Hist(GauD(Gau(Max(SobelX(Image)), 2), 1, 1, 0)), uLBP_R(LoG1_R(Sobel_R(Sobel_R(Mean_R(Mean_R(LoG2_R(Max_R(LoG2_R(Gau_R(Min_R(GauD_R(Mean_R(Max_R(GauD_R(Gau_R(Min_R(LoG1_R(RegionS(Image, 42, 49, 60))), 2), 1, 1, 2))), 3, 0, 2)), 2)))))))))), uLBP_R(Mean_R(SobelX_R(Lap_R(RegionS(Image, 0, 49, 40))))), DIF_R(Min_R(SobelY_R(LoG1_R(LoG2_R(SobelY_R(Lap_R(LoG2_R(Med_R(Mean_R(Mean_R(SobelY_R(Min_R(LoG2_R(Med_R(RegionS(GauD(Image, 1, 1, 0), 13, 42, 22)))))))))))))))))

Smoke-test wall time: 89.65 seconds
Default GP settings restored: 250 population / 10 generations


In [56]:
# -------------------------------------------------------
# Validate Original GP smoke-test result
# -------------------------------------------------------

if not isinstance(
    original_gp_smoke_result,
    dict,
):
    raise TypeError(
        "Original GP should return a dictionary, "
        f"received "
        f"{type(original_gp_smoke_result).__name__}."
    )


REQUIRED_ORIGINAL_GP_KEYS = {
    "test_acc",
    "classifier",
    "train_time",
    "train_fitness",
    "feature_dim",
    "program",
    "train_features",
    "test_features",
}


missing_keys = (
    REQUIRED_ORIGINAL_GP_KEYS
    - set(original_gp_smoke_result)
)


if missing_keys:
    raise KeyError(
        "Original GP result is missing keys: "
        f"{sorted(missing_keys)}"
    )


gp_smoke_train_features = np.asarray(
    original_gp_smoke_result[
        "train_features"
    ],
    dtype=np.float64,
)

gp_smoke_validation_features = np.asarray(
    original_gp_smoke_result[
        "test_features"
    ],
    dtype=np.float64,
)


assert (
    gp_smoke_train_features.shape[0]
    == len(y_train)
)

assert (
    gp_smoke_validation_features.shape[0]
    == len(y_validation)
)

assert (
    gp_smoke_train_features.shape[1]
    == gp_smoke_validation_features.shape[1]
)

assert np.isfinite(
    gp_smoke_train_features
).all()

assert np.isfinite(
    gp_smoke_validation_features
).all()


print("=" * 70)
print("Original GP integration smoke test")
print("=" * 70)

print(
    "Training feature shape:",
    gp_smoke_train_features.shape,
)

print(
    "Validation feature shape:",
    gp_smoke_validation_features.shape,
)

print(
    "Feature dimension:",
    original_gp_smoke_result[
        "feature_dim"
    ],
)

print(
    "Internal GP validation accuracy:",
    original_gp_smoke_result[
        "test_acc"
    ],
)

print(
    "Internal GP classifier:",
    original_gp_smoke_result[
        "classifier"
    ],
)

print(
    "Evolution time:",
    original_gp_smoke_result[
        "train_time"
    ],
)

print(
    "Test evaluated:",
    False,
)

print("=" * 70)

Original GP integration smoke test
Training feature shape: (1864, 200)
Validation feature shape: (467, 200)
Feature dimension: 200
Internal GP validation accuracy: 96.79
Internal GP classifier: RF
Evolution time: 81.5457980632782
Test evaluated: False


## 6.2 Shared KSDD2 evaluation of GP features

The Original GP implementation internally evaluates several classifiers and uses MinMax scaling.

Those internal results are retained only as GP diagnostics.

For the MARS comparison, the evolved feature matrices are re-evaluated using the common KSDD2 benchmark classifier:

- StandardScaler fitted on GP training features only;
- linear SVM;
- balanced class weighting;
- identical validation metrics to Handcrafted-8.

This separates:

1. GP feature discovery;
2. benchmark classification.

The resulting performance is therefore directly comparable with the other MARS benchmark methods.

In [57]:
# -------------------------------------------------------
# Evaluate precomputed feature matrices
# -------------------------------------------------------

def evaluate_precomputed_features(
    train_features: np.ndarray,
    validation_features: np.ndarray,
    *,
    method_name: str,
    comparison_group: str,
    feature_generation_time_seconds: float | None = None,
) -> dict[str, Any]:
    """
    Evaluate already-generated features using the same
    classifier pipeline as Section 4.
    """

    train_matrix = np.asarray(
        train_features,
        dtype=np.float64,
    )

    validation_matrix = np.asarray(
        validation_features,
        dtype=np.float64,
    )


    if train_matrix.ndim != 2:
        raise ValueError(
            "Training features must be 2D."
        )

    if validation_matrix.ndim != 2:
        raise ValueError(
            "Validation features must be 2D."
        )

    if (
        train_matrix.shape[1]
        != validation_matrix.shape[1]
    ):
        raise ValueError(
            "Training and validation feature "
            "dimensions differ."
        )

    if (
        train_matrix.shape[0]
        != len(y_train)
    ):
        raise ValueError(
            "Training feature row count does "
            "not match y_train."
        )

    if (
        validation_matrix.shape[0]
        != len(y_validation)
    ):
        raise ValueError(
            "Validation feature row count does "
            "not match y_validation."
        )

    if not np.isfinite(
        train_matrix
    ).all():
        raise ValueError(
            "Training features contain "
            "non-finite values."
        )

    if not np.isfinite(
        validation_matrix
    ).all():
        raise ValueError(
            "Validation features contain "
            "non-finite values."
        )


    # Reject constant columns.
    training_std = np.std(
        train_matrix,
        axis=0,
    )

    constant_columns = np.where(
        np.isclose(
            training_std,
            0.0,
        )
    )[0]


    if len(constant_columns) > 0:

        print(
            "Warning: removing constant GP "
            "feature columns:",
            constant_columns.tolist(),
        )

        keep_columns = ~np.isclose(
            training_std,
            0.0,
        )

        train_matrix = (
            train_matrix[
                :,
                keep_columns,
            ]
        )

        validation_matrix = (
            validation_matrix[
                :,
                keep_columns,
            ]
        )


    if train_matrix.shape[1] == 0:
        raise ValueError(
            "No usable GP features remain."
        )


    # ---------------------------------------------------
    # Training-only scaling
    # ---------------------------------------------------

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_matrix
    )

    validation_scaled = scaler.transform(
        validation_matrix
    )


    # ---------------------------------------------------
    # Common linear SVM
    # ---------------------------------------------------

    classifier = SVC(
        kernel=SVM_KERNEL,
        C=SVM_C,
        class_weight=SVM_CLASS_WEIGHT,
        random_state=RANDOM_STATE,
    )


    classifier_started = (
        time.perf_counter()
    )

    classifier.fit(
        train_scaled,
        y_train,
    )

    classifier_training_time = (
        time.perf_counter()
        - classifier_started
    )


    validation_predictions = (
        classifier.predict(
            validation_scaled
        )
    )


    validation_metrics = (
        calculate_binary_metrics(
            y_validation,
            validation_predictions,
        )
    )


    prediction_timing = (
        measure_prediction_time(
            classifier,
            validation_scaled,
        )
    )


    result = {
        "method": method_name,
        "comparison_group": comparison_group,
        "evaluation_pipeline": (
            "common_ksdd2_precomputed_feature_pipeline"
        ),

        "feature_dimension": int(
            train_matrix.shape[1]
        ),

        "validation_accuracy": (
            validation_metrics[
                "accuracy"
            ]
        ),

        "validation_balanced_accuracy": (
            validation_metrics[
                "balanced_accuracy"
            ]
        ),

        "validation_macro_f1": (
            validation_metrics[
                "macro_f1"
            ]
        ),

        "validation_defect_precision": (
            validation_metrics[
                "defect_precision"
            ]
        ),

        "validation_defect_recall": (
            validation_metrics[
                "defect_recall"
            ]
        ),

        "validation_confusion_matrix": (
            validation_metrics[
                "confusion_matrix"
            ]
        ),

        "classifier_training_time_seconds": float(
            classifier_training_time
        ),

        "validation_prediction_ms_per_image": (
            prediction_timing[
                "milliseconds_per_image"
            ]
        ),

        "feature_generation_time_seconds": (
            None
            if feature_generation_time_seconds
            is None
            else float(
                feature_generation_time_seconds
            )
        ),

        "test_accuracy": None,
        "test_evaluated": False,

        "scaler": scaler,
        "classifier": classifier,

        "training_features": (
            train_matrix
        ),

        "validation_features": (
            validation_matrix
        ),

        "validation_predictions": (
            validation_predictions
        ),
    }


    return result

In [58]:
# -------------------------------------------------------
# Common benchmark evaluation of GP smoke-test features
# -------------------------------------------------------

original_gp_smoke_benchmark_result = (
    evaluate_precomputed_features(
        train_features=(
            gp_smoke_train_features
        ),
        validation_features=(
            gp_smoke_validation_features
        ),
        method_name=(
            "Original GP — Smoke Test"
        ),
        comparison_group=(
            "integration_smoke_test"
        ),
        feature_generation_time_seconds=(
            original_gp_smoke_result[
                "train_time"
            ]
        ),
    )
)


print("=" * 70)
print("Original GP — Common KSDD2 Evaluation")
print("=" * 70)

print(
    "Feature dimension:",
    original_gp_smoke_benchmark_result[
        "feature_dimension"
    ],
)

print(
    "Accuracy:",
    f"{original_gp_smoke_benchmark_result['validation_accuracy'] * 100:.2f}%",
)

print(
    "Balanced accuracy:",
    f"{original_gp_smoke_benchmark_result['validation_balanced_accuracy'] * 100:.2f}%",
)

print(
    "Macro F1:",
    f"{original_gp_smoke_benchmark_result['validation_macro_f1']:.4f}",
)

print(
    "Defect precision:",
    f"{original_gp_smoke_benchmark_result['validation_defect_precision']:.4f}",
)

print(
    "Defect recall:",
    f"{original_gp_smoke_benchmark_result['validation_defect_recall']:.4f}",
)

print(
    "Prediction time:",
    f"{original_gp_smoke_benchmark_result['validation_prediction_ms_per_image']:.6f} ms/image",
)

print(
    "Test evaluated:",
    original_gp_smoke_benchmark_result[
        "test_evaluated"
    ],
)

print("=" * 70)

Original GP — Common KSDD2 Evaluation
Feature dimension: 162
Accuracy: 85.44%
Balanced accuracy: 81.96%
Macro F1: 0.7209
Defect precision: 0.4000
Defect recall: 0.7755
Prediction time: 0.016455 ms/image
Test evaluated: False


# 7. Restricted Original GP-8

This section evaluates the dimension-controlled Restricted Original GP implementation.

Unlike unrestricted Original GP, every valid Restricted GP individual must return exactly eight finite scalar features.

The method receives:

- `X_train` and `y_train` for evolutionary feature discovery;
- `X_validation` and `y_validation` for development validation;
- no access to the locked official KSDD2 test set.

The GP implementation uses stratified cross-validation internally for evolutionary fitness.

After evolution, its eight-dimensional feature matrices are evaluated using the same shared KSDD2 benchmark classifier as Handcrafted-8:

- StandardScaler fitted on training features only;
- linear SVM;
- balanced class weighting;
- identical validation metrics.

The first run is a small integration smoke test.

In [59]:
# -------------------------------------------------------
# Restricted Original GP package paths
# -------------------------------------------------------

RESTRICTED_ORIGINAL_GP_PARENT = (
    PROJECT_ROOT
    / "src"
    / "legacy"
    / "gp_restricted_original"
)

RESTRICTED_ORIGINAL_GP_PACKAGE_DIR = (
    RESTRICTED_ORIGINAL_GP_PARENT
    / "gp_fr"
)


if not RESTRICTED_ORIGINAL_GP_PACKAGE_DIR.exists():
    raise FileNotFoundError(
        "Restricted Original GP package not found:\n"
        f"{RESTRICTED_ORIGINAL_GP_PACKAGE_DIR}\n\n"
        "If your folder has a different name, update "
        "RESTRICTED_ORIGINAL_GP_PARENT."
    )


print("Restricted Original GP parent:")
print(RESTRICTED_ORIGINAL_GP_PARENT)

print("\nPackage directory:")
print(RESTRICTED_ORIGINAL_GP_PACKAGE_DIR)

Restricted Original GP parent:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_original

Package directory:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_original\gp_fr


In [60]:
# -------------------------------------------------------
# Load Restricted Original GP
# -------------------------------------------------------

restricted_original_gp_fr = load_gp_fr(
    RESTRICTED_ORIGINAL_GP_PARENT
)


restricted_original_module_path = Path(
    restricted_original_gp_fr.__file__
).resolve()


if (
    RESTRICTED_ORIGINAL_GP_PARENT.resolve()
    not in restricted_original_module_path.parents
):
    raise ImportError(
        "Wrong gp_fr package imported:\n"
        f"{restricted_original_module_path}"
    )


print("=" * 70)
print("Restricted Original GP-8 verification")
print("=" * 70)

print(
    "Loaded from:",
    restricted_original_module_path,
)

print(
    "\nrun_gp_fr signature:"
)

print(
    inspect.signature(
        restricted_original_gp_fr.run_gp_fr
    )
)

print(
    "\nConfigured population:",
    restricted_original_gp_fr.gp_fr_main.POPULATION,
)

print(
    "Configured generations:",
    restricted_original_gp_fr.gp_fr_main.GENERATION,
)

print(
    "Target feature dimension:",
    restricted_original_gp_fr.gp_fr_main.TARGET_FEATURE_DIMENSION,
)

print(
    "Internal CV folds:",
    restricted_original_gp_fr.gp_fr_main.CV_SPLITS,
)

print("=" * 70)

Restricted Original GP-8 verification
Loaded from: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_original\gp_fr\__init__.py

run_gp_fr signature:
(x_train, y_train, x_test, y_test, seed=0, verbose=True, population_size=None, generations=None)

Configured population: 250
Configured generations: 10
Target feature dimension: 8
Internal CV folds: 5


In [61]:
# -------------------------------------------------------
# Restricted GP-8 configuration checks
# -------------------------------------------------------

assert (
    restricted_original_gp_fr
    .gp_fr_main
    .TARGET_FEATURE_DIMENSION
    == N_FEATURES
), (
    "Restricted GP target dimension does not match "
    "the benchmark eight-feature requirement."
)


assert (
    restricted_original_gp_fr
    .gp_fr_main
    .CV_SPLITS
    >= 2
)


print(
    "PASS: Restricted Original GP target "
    f"dimension = {N_FEATURES}."
)

print(
    "PASS: internal stratified CV is configured."
)

PASS: Restricted Original GP target dimension = 8.
PASS: internal stratified CV is configured.


## 7.1 Restricted Original GP-8 smoke test

A small evolutionary run is used first to verify integration with KSDD2.

Smoke-test settings:

- population size: 10;
- generations: 1;
- seed: 42.

These settings are for software validation only and are not treated as the final GP experiment.

In [66]:
# -------------------------------------------------------
# Smoke-test settings
# -------------------------------------------------------

RESTRICTED_ORIGINAL_SMOKE_POPULATION = 10
RESTRICTED_ORIGINAL_SMOKE_GENERATIONS = 1


print(
    "Smoke-test population:",
    RESTRICTED_ORIGINAL_SMOKE_POPULATION,
)

print(
    "Smoke-test generations:",
    RESTRICTED_ORIGINAL_SMOKE_GENERATIONS,
)

print(
    "Seed:",
    RANDOM_STATE,
)

print(
    "Training images:",
    len(y_train),
)

print(
    "Validation images:",
    len(y_validation),
)

print(
    "Locked test images supplied to GP: 0"
)

Smoke-test population: 10
Smoke-test generations: 1
Seed: 42
Training images: 1864
Validation images: 467
Locked test images supplied to GP: 0


In [67]:
# -------------------------------------------------------
# Run Restricted Original GP-8 smoke test
# -------------------------------------------------------

restricted_original_started = (
    time.perf_counter()
)


restricted_original_smoke_raw = (
    restricted_original_gp_fr.run_gp_fr(
        x_train=X_train,
        y_train=y_train,

        # These are the benchmark VALIDATION arrays.
        # The locked official test set is not supplied.
        x_test=X_validation,
        y_test=y_validation,

        seed=RANDOM_STATE,
        verbose=True,

        population_size=(
            RESTRICTED_ORIGINAL_SMOKE_POPULATION
        ),

        generations=(
            RESTRICTED_ORIGINAL_SMOKE_GENERATIONS
        ),
    )
)


restricted_original_smoke_wall_time = (
    time.perf_counter()
    - restricted_original_started
)


print(
    "\nTotal benchmark wall time:",
    f"{restricted_original_smoke_wall_time:.2f} seconds",
)

gen	nevals	avg   	max  
0  	10    	65.005	93.35
1  	9     	93.35 	93.35

=== Restricted Original GP-8 Results (seed=42) ===
Training time: 237.9s
Best CV fitness: 93.35%
Validation accuracy: 95.07% (classifier: ERF)
Feature dimension: 8
Program: FC8(GradY(Image), Mean(Image), GradY(Image), Symmetry(Image), Symmetry(Image), Std(Image), MaxVal(Image), GradY_R(LoG2_R(Lap_R(Sobel_R(SobelX_R(RegionR(MinFilter(Image), 5, 39, 31, 14)))))))

Total benchmark wall time: 249.46 seconds


In [68]:
# -------------------------------------------------------
# Validate Restricted Original GP result
# -------------------------------------------------------

if not isinstance(
    restricted_original_smoke_raw,
    dict,
):
    raise TypeError(
        "Restricted Original GP should return "
        "a dictionary."
    )


REQUIRED_RESTRICTED_GP_KEYS = {
    "test_acc",
    "classifier",
    "train_time",
    "train_fitness",
    "feature_dim",
    "target_feature_dim",
    "program",
    "train_features",
    "test_features",
    "population_size",
    "generations",
    "seed",
}


missing_keys = (
    REQUIRED_RESTRICTED_GP_KEYS
    - set(restricted_original_smoke_raw)
)


if missing_keys:
    raise KeyError(
        "Restricted Original GP result is missing: "
        f"{sorted(missing_keys)}"
    )


restricted_original_train_features = np.asarray(
    restricted_original_smoke_raw[
        "train_features"
    ],
    dtype=np.float64,
)

restricted_original_validation_features = np.asarray(
    restricted_original_smoke_raw[
        "test_features"
    ],
    dtype=np.float64,
)


print(
    "Training feature shape:",
    restricted_original_train_features.shape,
)

print(
    "Validation feature shape:",
    restricted_original_validation_features.shape,
)

print(
    "Reported feature dimension:",
    restricted_original_smoke_raw[
        "feature_dim"
    ],
)

Training feature shape: (1864, 8)
Validation feature shape: (467, 8)
Reported feature dimension: 8


In [65]:
# -------------------------------------------------------
# Strict eight-feature validation
# -------------------------------------------------------

assert (
    restricted_original_train_features.shape
    == (len(y_train), N_FEATURES)
), (
    "Restricted Original GP training features "
    "violate the eight-feature contract."
)


assert (
    restricted_original_validation_features.shape
    == (len(y_validation), N_FEATURES)
), (
    "Restricted Original GP validation features "
    "violate the eight-feature contract."
)


assert np.isfinite(
    restricted_original_train_features
).all()


assert np.isfinite(
    restricted_original_validation_features
).all()


assert (
    restricted_original_smoke_raw[
        "feature_dim"
    ]
    == N_FEATURES
)


assert (
    restricted_original_smoke_raw[
        "target_feature_dim"
    ]
    == N_FEATURES
)


print(
    "PASS: Restricted Original GP returned "
    "exactly eight features for every sample."
)

PASS: Restricted Original GP returned exactly eight features for every sample.


## 7.2 Common KSDD2 evaluation

The Restricted GP implementation reports its own internal validation result.

For the controlled MARS comparison, the returned eight-dimensional feature matrices are evaluated again using the common benchmark pipeline.

This ensures that Handcrafted-8 and Restricted Original GP-8 use the same:

- feature scaling procedure;
- linear SVM;
- class weighting;
- validation labels;
- performance metrics.

In [69]:
# -------------------------------------------------------
# Shared benchmark evaluation
# -------------------------------------------------------

restricted_original_result = (
    evaluate_precomputed_features(
        train_features=(
            restricted_original_train_features
        ),

        validation_features=(
            restricted_original_validation_features
        ),

        method_name=(
            "Restricted Original GP-8"
        ),

        comparison_group=(
            "dimension_controlled"
        ),

        feature_generation_time_seconds=(
            restricted_original_smoke_raw[
                "train_time"
            ]
        ),
    )
)

In [70]:
# -------------------------------------------------------
# Restricted Original GP-8 result summary
# -------------------------------------------------------

print("=" * 70)
print("Restricted Original GP-8 — KSDD2")
print("=" * 70)

print(
    "Feature dimension :",
    restricted_original_result[
        "feature_dimension"
    ],
)

print(
    "Accuracy          :",
    f"{restricted_original_result['validation_accuracy'] * 100:.2f}%",
)

print(
    "Balanced accuracy :",
    f"{restricted_original_result['validation_balanced_accuracy'] * 100:.2f}%",
)

print(
    "Macro F1          :",
    f"{restricted_original_result['validation_macro_f1']:.4f}",
)

print(
    "Defect precision  :",
    f"{restricted_original_result['validation_defect_precision']:.4f}",
)

print(
    "Defect recall     :",
    f"{restricted_original_result['validation_defect_recall']:.4f}",
)

print(
    "SVM prediction    :",
    f"{restricted_original_result['validation_prediction_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "GP search time    :",
    f"{restricted_original_smoke_raw['train_time']:.2f}",
    "seconds",
)

print(
    "Population        :",
    restricted_original_smoke_raw[
        "population_size"
    ],
)

print(
    "Generations       :",
    restricted_original_smoke_raw[
        "generations"
    ],
)

print(
    "Test evaluated    :",
    restricted_original_result[
        "test_evaluated"
    ],
)

print("=" * 70)

Restricted Original GP-8 — KSDD2
Feature dimension : 8
Accuracy          : 89.94%
Balanced accuracy : 83.57%
Macro F1          : 0.7769
Defect precision  : 0.5139
Defect recall     : 0.7551
SVM prediction    : 0.007445 ms/image
GP search time    : 237.95 seconds
Population        : 10
Generations       : 1
Test evaluated    : False


In [71]:
# -------------------------------------------------------
# Restricted Original GP-8 confusion matrix
# -------------------------------------------------------

restricted_original_confusion_table = (
    pd.DataFrame(
        restricted_original_result[
            "validation_confusion_matrix"
        ],
        index=[
            "Actual Normal",
            "Actual Defective",
        ],
        columns=[
            "Predicted Normal",
            "Predicted Defective",
        ],
    )
)


display(
    restricted_original_confusion_table
)

,Predicted Normal,Predicted Defective
Actual Normal,383,35
Actual Defective,12,37


In [76]:
# -------------------------------------------------------
# First controlled eight-feature comparison
# -------------------------------------------------------

first_controlled_comparison = pd.DataFrame(
    [
        {
            "Method": "Handcrafted-8",
            "Features": (
                handcrafted_result[
                    "feature_dimension"
                ]
            ),
            "Accuracy": (
                handcrafted_result[
                    "validation_accuracy"
                ]
            ),
            "Balanced Accuracy": (
                handcrafted_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                handcrafted_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                handcrafted_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                handcrafted_result[
                    "validation_defect_recall"
                ]
            ),
        },

        {
            "Method": (
                "Restricted Original GP-8"
            ),
            "Features": (
                restricted_original_result[
                    "feature_dimension"
                ]
            ),
            "Accuracy": (
                restricted_original_result[
                    "validation_accuracy"
                ]
            ),
            "Balanced Accuracy": (
                restricted_original_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                restricted_original_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                restricted_original_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                restricted_original_result[
                    "validation_defect_recall"
                ]
            ),
        },
    ]
)


first_controlled_comparison = (
    first_controlled_comparison
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    first_controlled_comparison
)

,Method,Features,Accuracy,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall
0,Restricted Original GP-8,8,0.899358,0.835685,0.776880,0.513889,0.755102
1,Handcrafted-8,8,0.912206,0.833854,0.793615,0.562500,0.734694


In [77]:
# -------------------------------------------------------
# Save Restricted Original GP-8 result
# -------------------------------------------------------

RESTRICTED_ORIGINAL_SUMMARY_FILE = (
    OUTPUT_DIR
    / "restricted_original_gp8_smoke_summary.json"
)


restricted_original_summary = {
    "method": "Restricted Original GP-8",
    "run_type": "smoke_test",

    "feature_dimension": int(
        restricted_original_result[
            "feature_dimension"
        ]
    ),

    "population_size": int(
        restricted_original_smoke_raw[
            "population_size"
        ]
    ),

    "generations": int(
        restricted_original_smoke_raw[
            "generations"
        ]
    ),

    "seed": int(
        restricted_original_smoke_raw[
            "seed"
        ]
    ),

    "training_fitness": float(
        restricted_original_smoke_raw[
            "train_fitness"
        ]
    ),

    "gp_training_time_seconds": float(
        restricted_original_smoke_raw[
            "train_time"
        ]
    ),

    "validation_accuracy": float(
        restricted_original_result[
            "validation_accuracy"
        ]
    ),

    "validation_balanced_accuracy": float(
        restricted_original_result[
            "validation_balanced_accuracy"
        ]
    ),

    "validation_macro_f1": float(
        restricted_original_result[
            "validation_macro_f1"
        ]
    ),

    "validation_defect_precision": float(
        restricted_original_result[
            "validation_defect_precision"
        ]
    ),

    "validation_defect_recall": float(
        restricted_original_result[
            "validation_defect_recall"
        ]
    ),

    "program": (
        restricted_original_smoke_raw[
            "program"
        ]
    ),

    "test_evaluated": False,
}


with RESTRICTED_ORIGINAL_SUMMARY_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        restricted_original_summary,
        file,
        indent=2,
    )


print(
    "Saved Restricted Original GP-8 summary:"
)

print(
    RESTRICTED_ORIGINAL_SUMMARY_FILE
)

Saved Restricted Original GP-8 summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\restricted_original_gp8_smoke_summary.json


# 8. Modified Genetic Programming — Unrestricted Reference

This section integrates the existing Modified GP-FR implementation with the KSDD2 benchmark.

The Modified GP method is unrestricted in feature dimensionality. It is therefore treated as a reference method rather than part of the controlled eight-feature comparison.

The method receives:

- `X_train` and `y_train` for feature discovery;
- `X_validation` and `y_validation` for development validation;
- no access to the locked official KSDD2 test set.

After GP evolution, the returned feature matrices are re-evaluated using the shared KSDD2 benchmark classifier:

- StandardScaler fitted only on training features;
- linear SVM;
- balanced class weighting;
- identical validation metrics to the other benchmark methods.

The first run is an integration smoke test using a small population and generation count.

In [78]:
# -------------------------------------------------------
# Unrestricted Modified GP package paths
# -------------------------------------------------------

MODIFIED_GP_PARENT = (
    PROJECT_ROOT
    / "src"
    / "legacy"
    / "gp_modified"
)

MODIFIED_GP_PACKAGE_DIR = (
    MODIFIED_GP_PARENT
    / "gp_fr"
)


if not MODIFIED_GP_PACKAGE_DIR.exists():
    raise FileNotFoundError(
        "Modified GP package not found:\n"
        f"{MODIFIED_GP_PACKAGE_DIR}\n\n"
        "Update MODIFIED_GP_PARENT if your folder "
        "has a different name."
    )


print("Modified GP parent:")
print(MODIFIED_GP_PARENT)

print("\nPackage directory:")
print(MODIFIED_GP_PACKAGE_DIR)

Modified GP parent:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_modified

Package directory:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_modified\gp_fr


In [79]:
# -------------------------------------------------------
# Load Unrestricted Modified GP
# -------------------------------------------------------

unrestricted_modified_gp_fr = load_gp_fr(
    MODIFIED_GP_PARENT
)


unrestricted_modified_module_path = Path(
    unrestricted_modified_gp_fr.__file__
).resolve()


if (
    MODIFIED_GP_PARENT.resolve()
    not in unrestricted_modified_module_path.parents
):
    raise ImportError(
        "Wrong gp_fr package imported:\n"
        f"{unrestricted_modified_module_path}"
    )


print("=" * 70)
print("Unrestricted Modified GP verification")
print("=" * 70)

print(
    "Loaded from:",
    unrestricted_modified_module_path,
)

print("\nrun_gp_fr signature:")

print(
    inspect.signature(
        unrestricted_modified_gp_fr.run_gp_fr
    )
)

print(
    "\nConfigured population:",
    unrestricted_modified_gp_fr.gp_fr_main.POPULATION,
)

print(
    "Configured generations:",
    unrestricted_modified_gp_fr.gp_fr_main.GENERATION,
)

print("=" * 70)

Unrestricted Modified GP verification
Loaded from: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_modified\gp_fr\__init__.py

run_gp_fr signature:
(x_train, y_train, x_test, y_test, seed=0, verbose=True)

Configured population: 250
Configured generations: 10


## 8.1 Modified GP integration smoke test

Before running a larger experiment, a reduced evolutionary search is used to verify that the Modified GP implementation works correctly with KSDD2.

Smoke-test settings:

- population size: 10;
- generations: 1;
- random seed: 42.

This run is intended only to validate integration and estimate runtime.

In [80]:
# -------------------------------------------------------
# Modified GP smoke-test settings
# -------------------------------------------------------

MODIFIED_GP_DEFAULT_POPULATION = (
    unrestricted_modified_gp_fr
    .gp_fr_main
    .POPULATION
)

MODIFIED_GP_DEFAULT_GENERATIONS = (
    unrestricted_modified_gp_fr
    .gp_fr_main
    .GENERATION
)


MODIFIED_GP_SMOKE_POPULATION = 10
MODIFIED_GP_SMOKE_GENERATIONS = 1


print(
    "Default population:",
    MODIFIED_GP_DEFAULT_POPULATION,
)

print(
    "Default generations:",
    MODIFIED_GP_DEFAULT_GENERATIONS,
)

print(
    "Smoke population:",
    MODIFIED_GP_SMOKE_POPULATION,
)

print(
    "Smoke generations:",
    MODIFIED_GP_SMOKE_GENERATIONS,
)

print(
    "Training images:",
    len(y_train),
)

print(
    "Validation images:",
    len(y_validation),
)

print(
    "Locked test images supplied to GP: 0"
)

Default population: 250
Default generations: 10
Smoke population: 10
Smoke generations: 1
Training images: 1864
Validation images: 467
Locked test images supplied to GP: 0


In [81]:
# -------------------------------------------------------
# Execute Unrestricted Modified GP smoke test
# -------------------------------------------------------

unrestricted_modified_gp_fr.gp_fr_main.POPULATION = (
    MODIFIED_GP_SMOKE_POPULATION
)

unrestricted_modified_gp_fr.gp_fr_main.GENERATION = (
    MODIFIED_GP_SMOKE_GENERATIONS
)


modified_gp_started = time.perf_counter()


try:

    unrestricted_modified_smoke_raw = (
        unrestricted_modified_gp_fr.run_gp_fr(
            x_train=X_train,
            y_train=y_train,

            # Benchmark validation split only.
            # The official test set remains locked.
            x_test=X_validation,
            y_test=y_validation,

            seed=RANDOM_STATE,
            verbose=True,
        )
    )

finally:

    unrestricted_modified_gp_fr.gp_fr_main.POPULATION = (
        MODIFIED_GP_DEFAULT_POPULATION
    )

    unrestricted_modified_gp_fr.gp_fr_main.GENERATION = (
        MODIFIED_GP_DEFAULT_GENERATIONS
    )


unrestricted_modified_smoke_wall_time = (
    time.perf_counter()
    - modified_gp_started
)


print(
    "\nSmoke-test wall time:",
    f"{unrestricted_modified_smoke_wall_time:.2f} seconds",
)

print(
    "Default Modified GP settings restored:",
    unrestricted_modified_gp_fr.gp_fr_main.POPULATION,
    "population /",
    unrestricted_modified_gp_fr.gp_fr_main.GENERATION,
    "generations",
)

gen	nevals	avg   	max  
0  	10    	88.063	94.37
1  	9     	93.512	94.37

=== GP-FR Results (seed=42) ===
Training time: 319.1s
Best train fitness: 94.37%
Test accuracy: 95.72% (classifier: RF)
Feature dim: 274
Program: FC4(DIF_R(Max_R(Max_R(Min_R(SobelY_R(Mean_R(SobelX_R(SobelY_R(LoG2_R(Mean_R(Sobel_R(Lap_R(Med_R(Med_R(Mean_R(Max_R(Max_R(LoG2_R(Sobel_R(Med_R(LoG2_R(Max_R(SobelY_R(SobelX_R(SobelY_R(LoG2_R(Lap_R(Max_R(LoG1_R(Sobel_R(Gau_R(RegionR(Med(Image), 21, 9, 61, 6), 2))))))))))))))))))))))))))))))), SIFT_R(GauD_R(RegionS(Image, 51, 46, 34), 2, 2, 2)), Hist(Image), Hist(Image))

Smoke-test wall time: 351.20 seconds
Default Modified GP settings restored: 250 population / 10 generations


In [82]:
# -------------------------------------------------------
# Validate Modified GP smoke-test result
# -------------------------------------------------------

if not isinstance(
    unrestricted_modified_smoke_raw,
    dict,
):
    raise TypeError(
        "Modified GP should return a dictionary."
    )


REQUIRED_MODIFIED_GP_KEYS = {
    "test_acc",
    "classifier",
    "train_time",
    "train_fitness",
    "feature_dim",
    "program",
    "train_features",
    "test_features",
}


missing_keys = (
    REQUIRED_MODIFIED_GP_KEYS
    - set(unrestricted_modified_smoke_raw)
)


if missing_keys:
    raise KeyError(
        "Modified GP result is missing keys: "
        f"{sorted(missing_keys)}"
    )


modified_gp_train_features = np.asarray(
    unrestricted_modified_smoke_raw[
        "train_features"
    ],
    dtype=np.float64,
)

modified_gp_validation_features = np.asarray(
    unrestricted_modified_smoke_raw[
        "test_features"
    ],
    dtype=np.float64,
)


assert (
    modified_gp_train_features.shape[0]
    == len(y_train)
)

assert (
    modified_gp_validation_features.shape[0]
    == len(y_validation)
)

assert (
    modified_gp_train_features.shape[1]
    == modified_gp_validation_features.shape[1]
)

assert np.isfinite(
    modified_gp_train_features
).all()

assert np.isfinite(
    modified_gp_validation_features
).all()


print("=" * 70)
print("Unrestricted Modified GP smoke test")
print("=" * 70)

print(
    "Training feature shape:",
    modified_gp_train_features.shape,
)

print(
    "Validation feature shape:",
    modified_gp_validation_features.shape,
)

print(
    "Reported feature dimension:",
    unrestricted_modified_smoke_raw[
        "feature_dim"
    ],
)

print(
    "Internal validation accuracy:",
    unrestricted_modified_smoke_raw[
        "test_acc"
    ],
)

print(
    "Internal classifier:",
    unrestricted_modified_smoke_raw[
        "classifier"
    ],
)

print(
    "Evolution time:",
    unrestricted_modified_smoke_raw[
        "train_time"
    ],
)

print(
    "Test evaluated:",
    False,
)

print("=" * 70)

Unrestricted Modified GP smoke test
Training feature shape: (1864, 274)
Validation feature shape: (467, 274)
Reported feature dimension: 274
Internal validation accuracy: 95.72
Internal classifier: RF
Evolution time: 319.1420178413391
Test evaluated: False


## 8.2 Shared KSDD2 evaluation of Modified GP features

The Modified GP implementation performs its own internal classifier evaluation.

For the MARS comparison, those internal results are retained only as diagnostic information.

The evolved feature matrices are re-evaluated using the common KSDD2 benchmark pipeline so that unrestricted Original GP and unrestricted Modified GP use the same:

- StandardScaler;
- linear SVM;
- balanced class weighting;
- validation labels;
- performance metrics.

This isolates the quality of the learned feature representation from differences in downstream classifier configuration.

In [83]:
# -------------------------------------------------------
# Common KSDD2 evaluation of Modified GP features
# -------------------------------------------------------

unrestricted_modified_result = (
    evaluate_precomputed_features(
        train_features=(
            modified_gp_train_features
        ),

        validation_features=(
            modified_gp_validation_features
        ),

        method_name=(
            "Modified GP — Unrestricted"
        ),

        comparison_group=(
            "unrestricted_reference"
        ),

        feature_generation_time_seconds=(
            unrestricted_modified_smoke_raw[
                "train_time"
            ]
        ),
    )
)

In [84]:
# -------------------------------------------------------
# Unrestricted Modified GP result summary
# -------------------------------------------------------

print("=" * 70)
print("Modified GP — Unrestricted — KSDD2")
print("=" * 70)

print(
    "Feature dimension :",
    unrestricted_modified_result[
        "feature_dimension"
    ],
)

print(
    "Accuracy          :",
    f"{unrestricted_modified_result['validation_accuracy'] * 100:.2f}%",
)

print(
    "Balanced accuracy :",
    f"{unrestricted_modified_result['validation_balanced_accuracy'] * 100:.2f}%",
)

print(
    "Macro F1          :",
    f"{unrestricted_modified_result['validation_macro_f1']:.4f}",
)

print(
    "Defect precision  :",
    f"{unrestricted_modified_result['validation_defect_precision']:.4f}",
)

print(
    "Defect recall     :",
    f"{unrestricted_modified_result['validation_defect_recall']:.4f}",
)

print(
    "Prediction time   :",
    f"{unrestricted_modified_result['validation_prediction_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "GP search time    :",
    f"{unrestricted_modified_smoke_raw['train_time']:.2f}",
    "seconds",
)

print(
    "Test evaluated    :",
    unrestricted_modified_result[
        "test_evaluated"
    ],
)

print("=" * 70)

Modified GP — Unrestricted — KSDD2
Feature dimension : 266
Accuracy          : 88.22%
Balanced accuracy : 80.81%
Macro F1          : 0.7460
Defect precision  : 0.4605
Defect recall     : 0.7143
Prediction time   : 0.029176 ms/image
GP search time    : 319.14 seconds
Test evaluated    : False


In [85]:
# -------------------------------------------------------
# Modified GP unrestricted confusion matrix
# -------------------------------------------------------

modified_gp_confusion_table = pd.DataFrame(
    unrestricted_modified_result[
        "validation_confusion_matrix"
    ],
    index=[
        "Actual Normal",
        "Actual Defective",
    ],
    columns=[
        "Predicted Normal",
        "Predicted Defective",
    ],
)


display(
    modified_gp_confusion_table
)

,Predicted Normal,Predicted Defective
Actual Normal,377,41
Actual Defective,14,35


In [86]:
# -------------------------------------------------------
# Compare unrestricted Original and Modified GP
# -------------------------------------------------------

unrestricted_comparison = pd.DataFrame(
    [
        {
            "Method": "Original GP — Unrestricted",
            "Features": (
                original_gp_smoke_benchmark_result[
                    "feature_dimension"
                ]
            ),
            "Accuracy": (
                original_gp_smoke_benchmark_result[
                    "validation_accuracy"
                ]
            ),
            "Balanced Accuracy": (
                original_gp_smoke_benchmark_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                original_gp_smoke_benchmark_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                original_gp_smoke_benchmark_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                original_gp_smoke_benchmark_result[
                    "validation_defect_recall"
                ]
            ),
        },

        {
            "Method": "Modified GP — Unrestricted",
            "Features": (
                unrestricted_modified_result[
                    "feature_dimension"
                ]
            ),
            "Accuracy": (
                unrestricted_modified_result[
                    "validation_accuracy"
                ]
            ),
            "Balanced Accuracy": (
                unrestricted_modified_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                unrestricted_modified_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                unrestricted_modified_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                unrestricted_modified_result[
                    "validation_defect_recall"
                ]
            ),
        },
    ]
)


unrestricted_comparison = (
    unrestricted_comparison
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    unrestricted_comparison
)

,Method,Features,Accuracy,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall
0,Original GP — Unrestricted,162,0.854390,0.819573,0.720851,0.400000,0.775510
1,Modified GP — Unrestricted,266,0.882227,0.808100,0.746007,0.460526,0.714286


In [87]:
# -------------------------------------------------------
# Save Unrestricted Modified GP summary
# -------------------------------------------------------

UNRESTRICTED_MODIFIED_SUMMARY_FILE = (
    OUTPUT_DIR
    / "unrestricted_modified_gp_smoke_summary.json"
)


unrestricted_modified_summary = {
    "method": "Modified GP — Unrestricted",
    "run_type": "smoke_test",

    "feature_dimension": int(
        unrestricted_modified_result[
            "feature_dimension"
        ]
    ),

    "training_fitness": float(
        unrestricted_modified_smoke_raw[
            "train_fitness"
        ]
    ),

    "gp_training_time_seconds": float(
        unrestricted_modified_smoke_raw[
            "train_time"
        ]
    ),

    "validation_accuracy": float(
        unrestricted_modified_result[
            "validation_accuracy"
        ]
    ),

    "validation_balanced_accuracy": float(
        unrestricted_modified_result[
            "validation_balanced_accuracy"
        ]
    ),

    "validation_macro_f1": float(
        unrestricted_modified_result[
            "validation_macro_f1"
        ]
    ),

    "validation_defect_precision": float(
        unrestricted_modified_result[
            "validation_defect_precision"
        ]
    ),

    "validation_defect_recall": float(
        unrestricted_modified_result[
            "validation_defect_recall"
        ]
    ),

    "program": (
        unrestricted_modified_smoke_raw[
            "program"
        ]
    ),

    "test_evaluated": False,
}


with UNRESTRICTED_MODIFIED_SUMMARY_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        unrestricted_modified_summary,
        file,
        indent=2,
    )


print(
    "Saved Unrestricted Modified GP summary:"
)

print(
    UNRESTRICTED_MODIFIED_SUMMARY_FILE
)

Saved Unrestricted Modified GP summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\unrestricted_modified_gp_smoke_summary.json


# 9. Restricted Modified GP-8

This section evaluates the dimension-controlled Restricted Modified GP implementation on KSDD2.

The Modified GP search space contains the additional feature-generation primitives introduced by the modified method, while the output representation is constrained to exactly eight features.

The method receives:

- `X_train` and `y_train` for evolutionary feature discovery;
- `X_validation` and `y_validation` for development validation;
- no access to the locked official KSDD2 test set.

The resulting eight-dimensional feature matrices are evaluated using the same benchmark classifier as:

- Handcrafted-8;
- Restricted Original GP-8.

This provides a direct comparison between the Original and Modified GP methods under the same eight-feature constraint.

The first run uses:

- population size: 10;
- generations: 1;
- random seed: 42.

In [88]:
# -------------------------------------------------------
# Restricted Modified GP package paths
# -------------------------------------------------------

RESTRICTED_MODIFIED_GP_PARENT = (
    PROJECT_ROOT
    / "src"
    / "legacy"
    / "gp_restricted_modified"
)

RESTRICTED_MODIFIED_GP_PACKAGE_DIR = (
    RESTRICTED_MODIFIED_GP_PARENT
    / "gp_fr"
)


if not RESTRICTED_MODIFIED_GP_PACKAGE_DIR.exists():
    raise FileNotFoundError(
        "Restricted Modified GP package not found:\n"
        f"{RESTRICTED_MODIFIED_GP_PACKAGE_DIR}\n\n"
        "Update RESTRICTED_MODIFIED_GP_PARENT if your "
        "folder uses a different name."
    )


print("Restricted Modified GP parent:")
print(RESTRICTED_MODIFIED_GP_PARENT)

print("\nPackage directory:")
print(RESTRICTED_MODIFIED_GP_PACKAGE_DIR)

Restricted Modified GP parent:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_modified

Package directory:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_modified\gp_fr


In [89]:
# -------------------------------------------------------
# Load Restricted Modified GP
# -------------------------------------------------------

restricted_modified_gp_fr = load_gp_fr(
    RESTRICTED_MODIFIED_GP_PARENT
)


restricted_modified_module_path = Path(
    restricted_modified_gp_fr.__file__
).resolve()


if (
    RESTRICTED_MODIFIED_GP_PARENT.resolve()
    not in restricted_modified_module_path.parents
):
    raise ImportError(
        "Wrong gp_fr package imported:\n"
        f"{restricted_modified_module_path}"
    )


print("=" * 70)
print("Restricted Modified GP-8 verification")
print("=" * 70)

print(
    "Loaded from:",
    restricted_modified_module_path,
)

print("\nrun_gp_fr signature:")

print(
    inspect.signature(
        restricted_modified_gp_fr.run_gp_fr
    )
)

print("\nConfiguration:")

print(
    "Population:",
    restricted_modified_gp_fr
    .gp_fr_main
    .POPULATION,
)

print(
    "Generations:",
    restricted_modified_gp_fr
    .gp_fr_main
    .GENERATION,
)

print(
    "Target feature dimension:",
    restricted_modified_gp_fr
    .gp_fr_main
    .TARGET_FEATURE_DIMENSION,
)

print(
    "Internal CV folds:",
    restricted_modified_gp_fr
    .gp_fr_main
    .CV_SPLITS,
)

print("=" * 70)

Restricted Modified GP-8 verification
Loaded from: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\gp_restricted_modified\gp_fr\__init__.py

run_gp_fr signature:
(x_train, y_train, x_test, y_test, seed=0, verbose=True, population_size=None, generations=None)

Configuration:
Population: 250
Generations: 10
Target feature dimension: 8
Internal CV folds: 5


In [90]:
# -------------------------------------------------------
# Restricted Modified GP configuration checks
# -------------------------------------------------------

assert (
    restricted_modified_gp_fr
    .gp_fr_main
    .TARGET_FEATURE_DIMENSION
    == N_FEATURES
), (
    "Restricted Modified GP does not target "
    "exactly eight features."
)


assert (
    restricted_modified_gp_fr
    .gp_fr_main
    .CV_SPLITS
    >= 2
), (
    "Restricted Modified GP internal CV "
    "is not configured correctly."
)


print(
    "PASS: Restricted Modified GP target "
    f"dimension = {N_FEATURES}."
)

print(
    "PASS: internal stratified CV "
    "is configured."
)

PASS: Restricted Modified GP target dimension = 8.
PASS: internal stratified CV is configured.


## 9.1 Restricted Modified GP-8 integration smoke test

A small evolutionary experiment is performed before any larger search.

The purpose of this run is to confirm:

- compatibility with the KSDD2 arrays;
- successful evolutionary search;
- exactly eight returned features;
- finite feature values;
- compatibility with the common KSDD2 benchmark pipeline.

The smoke-test result is a development result and is not yet treated as a fully tuned GP experiment.

In [94]:
# -------------------------------------------------------
# Restricted Modified GP smoke-test settings
# -------------------------------------------------------

RESTRICTED_MODIFIED_SMOKE_POPULATION = 10
RESTRICTED_MODIFIED_SMOKE_GENERATIONS = 1


print("=" * 70)
print("Restricted Modified GP-8 smoke-test settings")
print("=" * 70)

print(
    "Population:",
    RESTRICTED_MODIFIED_SMOKE_POPULATION,
)

print(
    "Generations:",
    RESTRICTED_MODIFIED_SMOKE_GENERATIONS,
)

print(
    "Seed:",
    RANDOM_STATE,
)

print(
    "Training images:",
    len(y_train),
)

print(
    "Validation images:",
    len(y_validation),
)

print(
    "Locked official test images supplied: 0"
)

print("=" * 70)

Restricted Modified GP-8 smoke-test settings
Population: 10
Generations: 1
Seed: 42
Training images: 1864
Validation images: 467
Locked official test images supplied: 0


In [95]:
# -------------------------------------------------------
# Run Restricted Modified GP-8 smoke test
# -------------------------------------------------------

restricted_modified_started = (
    time.perf_counter()
)


restricted_modified_smoke_raw = (
    restricted_modified_gp_fr.run_gp_fr(
        x_train=X_train,
        y_train=y_train,

        # Development validation data only.
        x_test=X_validation,
        y_test=y_validation,

        seed=RANDOM_STATE,
        verbose=True,

        population_size=(
            RESTRICTED_MODIFIED_SMOKE_POPULATION
        ),

        generations=(
            RESTRICTED_MODIFIED_SMOKE_GENERATIONS
        ),
    )
)


restricted_modified_smoke_wall_time = (
    time.perf_counter()
    - restricted_modified_started
)


print(
    "\nTotal benchmark wall time:",
    f"{restricted_modified_smoke_wall_time:.2f} seconds",
)

print(
    "Official test set evaluated:",
    False,
)

gen	nevals	avg   	max  
0  	10    	54.438	92.33
1  	9     	91.714	92.7 

=== Restricted Modified GP-8 Results (seed=42) ===
Training time: 269.3s
Best CV fitness: 92.70%
Validation accuracy: 94.65% (classifier: RF)
Feature dimension: 8
Program: FC8(MinVal(Med(SobelY(GauD(Image, 3, 0, 1)))), Median(SobelY(SobelX(MinFilter(Image)))), Median_R(RegionS(Sobel(SobelX(Image)), 8, 15, 23)), Mean(Sobel(Med(Sobel(Image)))), Symmetry(LoG1(GauD(GauD(Image, 3, 2, 1), 3, 2, 0))), RIFPeak(MinFilter(LoG1(LoG2(Image)))), Energy(MeanFilter(Gau(Gau(Image, 3), 3))), MaxVal(MaxFilter(MeanFilter(MinFilter(Image)))))

Total benchmark wall time: 278.91 seconds
Official test set evaluated: False


In [96]:
# -------------------------------------------------------
# Validate Restricted Modified GP output
# -------------------------------------------------------

if not isinstance(
    restricted_modified_smoke_raw,
    dict,
):
    raise TypeError(
        "Restricted Modified GP should return "
        "a dictionary."
    )


REQUIRED_RESTRICTED_MODIFIED_KEYS = {
    "test_acc",
    "classifier",
    "train_time",
    "train_fitness",
    "feature_dim",
    "target_feature_dim",
    "program",
    "train_features",
    "test_features",
    "population_size",
    "generations",
    "seed",
}


missing_keys = (
    REQUIRED_RESTRICTED_MODIFIED_KEYS
    - set(restricted_modified_smoke_raw)
)


if missing_keys:
    raise KeyError(
        "Restricted Modified GP result is "
        "missing keys: "
        f"{sorted(missing_keys)}"
    )


restricted_modified_train_features = (
    np.asarray(
        restricted_modified_smoke_raw[
            "train_features"
        ],
        dtype=np.float64,
    )
)

restricted_modified_validation_features = (
    np.asarray(
        restricted_modified_smoke_raw[
            "test_features"
        ],
        dtype=np.float64,
    )
)


print("=" * 70)
print("Restricted Modified GP-8 raw output")
print("=" * 70)

print(
    "Training feature shape:",
    restricted_modified_train_features.shape,
)

print(
    "Validation feature shape:",
    restricted_modified_validation_features.shape,
)

print(
    "Reported feature dimension:",
    restricted_modified_smoke_raw[
        "feature_dim"
    ],
)

print(
    "Target feature dimension:",
    restricted_modified_smoke_raw[
        "target_feature_dim"
    ],
)

print(
    "Internal validation accuracy:",
    restricted_modified_smoke_raw[
        "test_acc"
    ],
)

print(
    "GP training fitness:",
    restricted_modified_smoke_raw[
        "train_fitness"
    ],
)

print(
    "GP search time:",
    restricted_modified_smoke_raw[
        "train_time"
    ],
)

print("=" * 70)

Restricted Modified GP-8 raw output
Training feature shape: (1864, 8)
Validation feature shape: (467, 8)
Reported feature dimension: 8
Target feature dimension: 8
Internal validation accuracy: 94.65
GP training fitness: 92.7
GP search time: 269.2640156999696


In [97]:
# -------------------------------------------------------
# Strict Restricted Modified GP-8 contract
# -------------------------------------------------------

expected_train_shape = (
    len(y_train),
    N_FEATURES,
)

expected_validation_shape = (
    len(y_validation),
    N_FEATURES,
)


assert (
    restricted_modified_train_features.shape
    == expected_train_shape
), (
    "Restricted Modified GP training features "
    f"have shape "
    f"{restricted_modified_train_features.shape}; "
    f"expected {expected_train_shape}."
)


assert (
    restricted_modified_validation_features.shape
    == expected_validation_shape
), (
    "Restricted Modified GP validation features "
    f"have shape "
    f"{restricted_modified_validation_features.shape}; "
    f"expected {expected_validation_shape}."
)


assert np.isfinite(
    restricted_modified_train_features
).all(), (
    "Non-finite training features detected."
)


assert np.isfinite(
    restricted_modified_validation_features
).all(), (
    "Non-finite validation features detected."
)


assert (
    restricted_modified_smoke_raw[
        "feature_dim"
    ]
    == N_FEATURES
)


assert (
    restricted_modified_smoke_raw[
        "target_feature_dim"
    ]
    == N_FEATURES
)


print(
    "PASS: Restricted Modified GP returned "
    "exactly eight finite features for every sample."
)

PASS: Restricted Modified GP returned exactly eight finite features for every sample.


## 9.2 Common benchmark evaluation

The Restricted Modified GP implementation performs its own internal fitness and validation calculations during evolution.

For the controlled MARS comparison, its final eight-dimensional feature matrices are evaluated again using the shared benchmark classifier.

Therefore, Handcrafted-8, Restricted Original GP-8 and Restricted Modified GP-8 all use:

- exactly eight features;
- the same training images;
- the same validation images;
- StandardScaler fitted only to training features;
- the same linear SVM;
- balanced class weighting;
- identical performance metrics.

In [98]:
# -------------------------------------------------------
# Common benchmark evaluation
# -------------------------------------------------------

restricted_modified_result = (
    evaluate_precomputed_features(
        train_features=(
            restricted_modified_train_features
        ),

        validation_features=(
            restricted_modified_validation_features
        ),

        method_name=(
            "Restricted Modified GP-8"
        ),

        comparison_group=(
            "dimension_controlled"
        ),

        feature_generation_time_seconds=(
            restricted_modified_smoke_raw[
                "train_time"
            ]
        ),
    )
)

In [99]:
# -------------------------------------------------------
# Restricted Modified GP-8 result summary
# -------------------------------------------------------

print("=" * 70)
print("Restricted Modified GP-8 — KSDD2")
print("=" * 70)

print(
    "Feature dimension :",
    restricted_modified_result[
        "feature_dimension"
    ],
)

print(
    "Accuracy          :",
    f"{restricted_modified_result['validation_accuracy'] * 100:.2f}%",
)

print(
    "Balanced accuracy :",
    f"{restricted_modified_result['validation_balanced_accuracy'] * 100:.2f}%",
)

print(
    "Macro F1          :",
    f"{restricted_modified_result['validation_macro_f1']:.4f}",
)

print(
    "Defect precision  :",
    f"{restricted_modified_result['validation_defect_precision']:.4f}",
)

print(
    "Defect recall     :",
    f"{restricted_modified_result['validation_defect_recall']:.4f}",
)

print(
    "SVM prediction    :",
    f"{restricted_modified_result['validation_prediction_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "GP search time    :",
    f"{restricted_modified_smoke_raw['train_time']:.2f}",
    "seconds",
)

print(
    "Population        :",
    restricted_modified_smoke_raw[
        "population_size"
    ],
)

print(
    "Generations       :",
    restricted_modified_smoke_raw[
        "generations"
    ],
)

print(
    "Test evaluated    :",
    restricted_modified_result[
        "test_evaluated"
    ],
)

print("=" * 70)

Restricted Modified GP-8 — KSDD2
Feature dimension : 8
Accuracy          : 88.22%
Balanced accuracy : 79.01%
Macro F1          : 0.7389
Defect precision  : 0.4583
Defect recall     : 0.6735
SVM prediction    : 0.007622 ms/image
GP search time    : 269.26 seconds
Population        : 10
Generations       : 1
Test evaluated    : False


In [100]:
# -------------------------------------------------------
# Restricted Modified GP-8 confusion matrix
# -------------------------------------------------------

restricted_modified_confusion_table = (
    pd.DataFrame(
        restricted_modified_result[
            "validation_confusion_matrix"
        ],

        index=[
            "Actual Normal",
            "Actual Defective",
        ],

        columns=[
            "Predicted Normal",
            "Predicted Defective",
        ],
    )
)


display(
    restricted_modified_confusion_table
)

,Predicted Normal,Predicted Defective
Actual Normal,379,39
Actual Defective,16,33


In [101]:
# -------------------------------------------------------
# Controlled eight-feature comparison
# -------------------------------------------------------

controlled_comparison_three = pd.DataFrame(
    [
        {
            "Method": "Handcrafted-8",
            "Features": handcrafted_result[
                "feature_dimension"
            ],
            "Accuracy": handcrafted_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": handcrafted_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": handcrafted_result[
                "validation_macro_f1"
            ],
            "Defect Precision": handcrafted_result[
                "validation_defect_precision"
            ],
            "Defect Recall": handcrafted_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Original GP-8",
            "Features": restricted_original_result[
                "feature_dimension"
            ],
            "Accuracy": restricted_original_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": restricted_original_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_original_result[
                "validation_macro_f1"
            ],
            "Defect Precision": restricted_original_result[
                "validation_defect_precision"
            ],
            "Defect Recall": restricted_original_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Modified GP-8",
            "Features": restricted_modified_result[
                "feature_dimension"
            ],
            "Accuracy": restricted_modified_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": restricted_modified_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_modified_result[
                "validation_macro_f1"
            ],
            "Defect Precision": restricted_modified_result[
                "validation_defect_precision"
            ],
            "Defect Recall": restricted_modified_result[
                "validation_defect_recall"
            ],
        },
    ]
)


controlled_comparison_three = (
    controlled_comparison_three
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    controlled_comparison_three
)

,Method,Features,Accuracy,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall
0,Restricted Original GP-8,8,0.899358,0.835685,0.776880,0.513889,0.755102
1,Handcrafted-8,8,0.912206,0.833854,0.793615,0.562500,0.734694
2,Restricted Modified GP-8,8,0.882227,0.790084,0.738902,0.458333,0.673469


In [102]:
# -------------------------------------------------------
# Direct Restricted Original vs Modified GP comparison
# -------------------------------------------------------

restricted_gp_pair_comparison = pd.DataFrame(
    [
        {
            "Method": "Restricted Original GP-8",
            "Balanced Accuracy": (
                restricted_original_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                restricted_original_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                restricted_original_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                restricted_original_result[
                    "validation_defect_recall"
                ]
            ),
            "Search Time (s)": (
                restricted_original_smoke_raw[
                    "train_time"
                ]
            ),
        },

        {
            "Method": "Restricted Modified GP-8",
            "Balanced Accuracy": (
                restricted_modified_result[
                    "validation_balanced_accuracy"
                ]
            ),
            "Macro F1": (
                restricted_modified_result[
                    "validation_macro_f1"
                ]
            ),
            "Defect Precision": (
                restricted_modified_result[
                    "validation_defect_precision"
                ]
            ),
            "Defect Recall": (
                restricted_modified_result[
                    "validation_defect_recall"
                ]
            ),
            "Search Time (s)": (
                restricted_modified_smoke_raw[
                    "train_time"
                ]
            ),
        },
    ]
)


display(
    restricted_gp_pair_comparison
)

,Method,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall,Search Time (s)
0,Restricted Original GP-8,0.835685,0.776880,0.513889,0.755102,237.948556
1,Restricted Modified GP-8,0.790084,0.738902,0.458333,0.673469,269.264016


In [103]:
# -------------------------------------------------------
# Save Restricted Modified GP-8 result
# -------------------------------------------------------

RESTRICTED_MODIFIED_SUMMARY_FILE = (
    OUTPUT_DIR
    / "restricted_modified_gp8_smoke_summary.json"
)


restricted_modified_summary = {
    "method": "Restricted Modified GP-8",
    "run_type": "smoke_test",

    "feature_dimension": int(
        restricted_modified_result[
            "feature_dimension"
        ]
    ),

    "population_size": int(
        restricted_modified_smoke_raw[
            "population_size"
        ]
    ),

    "generations": int(
        restricted_modified_smoke_raw[
            "generations"
        ]
    ),

    "seed": int(
        restricted_modified_smoke_raw[
            "seed"
        ]
    ),

    "training_fitness": float(
        restricted_modified_smoke_raw[
            "train_fitness"
        ]
    ),

    "gp_training_time_seconds": float(
        restricted_modified_smoke_raw[
            "train_time"
        ]
    ),

    "validation_accuracy": float(
        restricted_modified_result[
            "validation_accuracy"
        ]
    ),

    "validation_balanced_accuracy": float(
        restricted_modified_result[
            "validation_balanced_accuracy"
        ]
    ),

    "validation_macro_f1": float(
        restricted_modified_result[
            "validation_macro_f1"
        ]
    ),

    "validation_defect_precision": float(
        restricted_modified_result[
            "validation_defect_precision"
        ]
    ),

    "validation_defect_recall": float(
        restricted_modified_result[
            "validation_defect_recall"
        ]
    ),

    "program": (
        restricted_modified_smoke_raw[
            "program"
        ]
    ),

    "test_evaluated": False,
}


with RESTRICTED_MODIFIED_SUMMARY_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        restricted_modified_summary,
        file,
        indent=2,
    )


print(
    "Saved Restricted Modified GP-8 summary:"
)

print(
    RESTRICTED_MODIFIED_SUMMARY_FILE
)

Saved Restricted Modified GP-8 summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\restricted_modified_gp8_smoke_summary.json


# 10. EOH-8 — LLM-Generated Eight-Feature Representation

This section adapts the Evolution of Heuristics (EOH) feature-generation approach to the KSDD2 benchmark.

EOH uses a Large Language Model to generate candidate Python feature-extraction functions. Every candidate must:

- accept one preprocessed 64 × 64 grayscale image;
- return exactly eight finite scalar features;
- use deterministic NumPy-only computation;
- avoid access to labels, datasets, files, networks and classifiers;
- remain interpretable as an image-processing representation.

Candidate programs are evaluated using only the benchmark training and validation subsets.

Because KSDD2 is strongly class-imbalanced, EOH fitness is based on balanced validation accuracy rather than ordinary accuracy.

The official KSDD2 test set remains locked.

The first EOH configuration is intentionally minimal:

- population size: 2;
- populations: 1;
- evolutionary operator: `e1`;
- samplers: 1;
- evaluators: 1.

The API-consuming search remains disabled until all local preflight checks pass.

In [107]:
# -------------------------------------------------------
# Locate and import the repository copy of EOH
# -------------------------------------------------------

import importlib
import inspect
import sys

from pathlib import Path


# -------------------------------------------------------
# EOH repository locations
# -------------------------------------------------------

EOH_ROOT = (
    PROJECT_ROOT
    / "src"
    / "legacy"
    / "eoh"
    / "EoH-main"
)

EOH_SRC_ROOT = (
    EOH_ROOT
    / "eoh"
    / "src"
)

EOH_PACKAGE_DIR = (
    EOH_SRC_ROOT
    / "eoh"
)

EOH_INIT_FILE = (
    EOH_PACKAGE_DIR
    / "__init__.py"
)


# -------------------------------------------------------
# Validate repository structure
# -------------------------------------------------------

if not EOH_ROOT.exists():
    raise FileNotFoundError(
        "EOH repository was not found:\n"
        f"{EOH_ROOT}"
    )

if not EOH_SRC_ROOT.exists():
    raise FileNotFoundError(
        "EOH src directory was not found:\n"
        f"{EOH_SRC_ROOT}"
    )

if not EOH_PACKAGE_DIR.exists():
    raise FileNotFoundError(
        "EOH package directory was not found:\n"
        f"{EOH_PACKAGE_DIR}"
    )

if not EOH_INIT_FILE.exists():
    raise FileNotFoundError(
        "EOH package __init__.py was not found:\n"
        f"{EOH_INIT_FILE}"
    )


# -------------------------------------------------------
# Remove any previously imported EOH package
# -------------------------------------------------------

for module_name in list(sys.modules):
    if (
        module_name == "eoh"
        or module_name.startswith("eoh.")
    ):
        del sys.modules[module_name]


# -------------------------------------------------------
# Force repository source tree to highest priority
# -------------------------------------------------------

eoh_src_string = str(EOH_SRC_ROOT)

if eoh_src_string in sys.path:
    sys.path.remove(eoh_src_string)

sys.path.insert(
    0,
    eoh_src_string,
)

importlib.invalidate_caches()


# -------------------------------------------------------
# Import EOH
# -------------------------------------------------------

import eoh
from eoh import EoH, LLMConfig


# -------------------------------------------------------
# Verify correct package was imported
# -------------------------------------------------------

eoh_module_path = Path(
    inspect.getfile(eoh)
).resolve()

if EOH_PACKAGE_DIR.resolve() not in eoh_module_path.parents:
    raise ImportError(
        "EOH was imported from the wrong location:\n"
        f"{eoh_module_path}\n"
        "Expected it underneath:\n"
        f"{EOH_PACKAGE_DIR}"
    )


print("EOH repository import successful.")
print("EOH imported from:")
print(eoh_module_path)

EOH repository import successful.
EOH imported from:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\src\legacy\eoh\EoH-main\eoh\src\eoh\__init__.py


## 10.1 Create an importable KSDD2 EOH-8 problem

EOH candidate evaluation may occur in worker processes.

To avoid Windows multiprocessing problems, the KSDD2 problem class is written to a real Python module before being imported.

The problem enforces:

- exactly eight output features;
- finite numerical values;
- no constant training feature columns;
- fixed training and validation arrays;
- balanced validation accuracy as the EOH fitness objective.

Lower EOH fitness is better:

`fitness = 1 - balanced validation accuracy`.

In [108]:
# -------------------------------------------------------
# Write importable KSDD2 EOH-8 problem module
# -------------------------------------------------------

KSDD2_EOH_MODULE_NAME = "ksdd2_eoh_problem_8"

EOH_RUNTIME_DIR = OUTPUT_DIR / "eoh_runtime"
EOH_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

KSDD2_EOH_MODULE_PATH = (
    EOH_RUNTIME_DIR
    / f"{KSDD2_EOH_MODULE_NAME}.py"
)


KSDD2_EOH_MODULE_SOURCE = r'''
from __future__ import annotations

import time
import traceback

import numpy as np

from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from eoh import BaseProblem


class KSDD2FeatureExtractionProblem8(BaseProblem):
    """
    EOH problem for KSDD2 binary defect classification.

    Every valid candidate must return exactly eight finite
    scalar features from one 64 x 64 grayscale image.
    """

    N_FEATURES = 8

    template_program = """
import numpy as np

def extract_features(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=float)

    dx = np.diff(image, axis=1)
    dy = np.diff(image, axis=0)

    mean_value = np.mean(image)
    std_value = np.std(image)

    dark_threshold = mean_value - std_value
    bright_threshold = mean_value + std_value

    laplacian_like = (
        -4.0 * image[1:-1, 1:-1]
        + image[:-2, 1:-1]
        + image[2:, 1:-1]
        + image[1:-1, :-2]
        + image[1:-1, 2:]
    )

    return np.array([
        mean_value,
        std_value,
        np.mean(np.abs(dx)),
        np.mean(np.abs(dy)),
        np.std(
            np.concatenate([
                dx.reshape(-1),
                dy.reshape(-1),
            ])
        ),
        np.mean(image < dark_threshold),
        np.mean(image > bright_threshold),
        np.var(laplacian_like),
    ], dtype=float)
"""


    task_description = (
        "Improve the NumPy-only extract_features function for binary "
        "industrial surface-defect classification on the KolektorSDD2 "
        "dataset. The input is one preprocessed 64 by 64 grayscale NumPy "
        "array with values between 0 and 1. The function must be named "
        "extract_features, accept exactly one image argument, and return "
        "exactly eight finite scalar values in a one-dimensional NumPy "
        "array with shape (8,). Each feature should have a stable and "
        "interpretable image-processing meaning. Use only deterministic "
        "NumPy computation. Do not access files, networks, random number "
        "generators, classifiers, labels, masks, global datasets or "
        "external libraries. Prefer interpretable measurements relevant "
        "to surface defects, including intensity statistics, gradients, "
        "local contrast, edge strength, texture variation, bright or dark "
        "pixel proportions, regional statistics and simple moments. "
        "Do not return flattened images, long histograms, arbitrary pixel "
        "collections or high-dimensional vectors. Candidate quality is "
        "measured using balanced validation accuracy from a fixed "
        "standardised class-balanced linear SVM. Fitness equals one minus "
        "balanced validation accuracy, so lower fitness is better."
    )


    def __init__(
        self,
        X_train,
        y_train,
        X_validation,
        y_validation,
        *,
        timeout=180,
        n_processes=1,
        max_seconds_per_evaluation=300.0,
    ):
        super().__init__(
            timeout=timeout,
            n_processes=n_processes,
        )

        self.X_train = np.asarray(
            X_train,
            dtype=np.float32,
        )

        self.y_train = np.asarray(
            y_train,
            dtype=np.int64,
        )

        self.X_validation = np.asarray(
            X_validation,
            dtype=np.float32,
        )

        self.y_validation = np.asarray(
            y_validation,
            dtype=np.int64,
        )

        self.max_seconds_per_evaluation = float(
            max_seconds_per_evaluation
        )


    @classmethod
    def _validate_vector(
        cls,
        values,
        image_index,
    ):
        vector = np.asarray(
            values,
            dtype=np.float64,
        ).reshape(-1)

        if vector.shape != (cls.N_FEATURES,):
            raise ValueError(
                f"Image {image_index} produced "
                f"{vector.shape}; exactly "
                f"({cls.N_FEATURES},) is required."
            )

        if not np.all(np.isfinite(vector)):
            raise ValueError(
                f"Image {image_index} produced "
                "NaN or infinite values."
            )

        return vector


    def _build_feature_matrix(
        self,
        images,
        feature_function,
    ):
        rows = []

        started = time.perf_counter()

        for image_index, image in enumerate(images):

            if (
                time.perf_counter() - started
                > self.max_seconds_per_evaluation
            ):
                raise TimeoutError(
                    "Feature extraction exceeded "
                    "the local evaluation limit."
                )

            rows.append(
                self._validate_vector(
                    feature_function(image),
                    image_index,
                )
            )

        matrix = np.vstack(rows)

        expected_shape = (
            len(images),
            self.N_FEATURES,
        )

        if matrix.shape != expected_shape:
            raise ValueError(
                f"Expected feature matrix "
                f"{expected_shape}, got "
                f"{matrix.shape}."
            )

        return matrix


    def _validation_balanced_accuracy(
        self,
        feature_function,
    ):
        training_features = (
            self._build_feature_matrix(
                self.X_train,
                feature_function,
            )
        )

        validation_features = (
            self._build_feature_matrix(
                self.X_validation,
                feature_function,
            )
        )

        constant_columns = np.where(
            np.isclose(
                training_features.std(axis=0),
                0.0,
            )
        )[0]

        if len(constant_columns) > 0:
            raise ValueError(
                "Constant training features in "
                f"columns {constant_columns.tolist()}."
            )

        scaler = StandardScaler()

        training_scaled = scaler.fit_transform(
            training_features
        )

        validation_scaled = scaler.transform(
            validation_features
        )

        classifier = SVC(
            kernel="linear",
            C=1.0,
            class_weight="balanced",
            random_state=42,
        )

        classifier.fit(
            training_scaled,
            self.y_train,
        )

        predictions = classifier.predict(
            validation_scaled
        )

        return float(
            balanced_accuracy_score(
                self.y_validation,
                predictions,
            )
        )


    def evaluate_program(
        self,
        program_str,
        callable_func,
    ):
        started = time.perf_counter()

        try:
            if callable_func is None:
                raise TypeError(
                    "EOH did not provide a "
                    "callable function."
                )

            self._validate_vector(
                callable_func(
                    self.X_train[0]
                ),
                0,
            )

            balanced_accuracy = (
                self._validation_balanced_accuracy(
                    callable_func
                )
            )

            if not (
                0.0
                <= balanced_accuracy
                <= 1.0
            ):
                raise ValueError(
                    "Balanced validation accuracy "
                    "is outside [0, 1]."
                )

            fitness = float(
                1.0 - balanced_accuracy
            )

            self._write_diagnostic(
                "success",
                program_str,
                (
                    "feature_dim=8, "
                    f"balanced_accuracy="
                    f"{balanced_accuracy:.6f}, "
                    f"fitness={fitness:.6f}, "
                    f"runtime="
                    f"{time.perf_counter() - started:.3f}s"
                ),
            )

            return fitness

        except Exception as error:

            self._write_diagnostic(
                "failure",
                program_str,
                (
                    f"{type(error).__name__}: "
                    f"{error}\n"
                    f"{traceback.format_exc()}"
                ),
            )

            return None


    def _write_diagnostic(
        self,
        status,
        program_str,
        message,
    ):
        try:
            from datetime import datetime
            from pathlib import Path
            import os

            folder = (
                Path(__file__).resolve().parents[2]
                / "ksdd2_eoh_8"
                / "candidate_diagnostics"
            )

            folder.mkdir(
                parents=True,
                exist_ok=True,
            )

            stamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S_%f"
            )

            path = (
                folder
                / f"{stamp}_{os.getpid()}_"
                  f"{status}.txt"
            )

            path.write_text(
                (
                    f"STATUS: {status}\n\n"
                    f"MESSAGE:\n{message}\n\n"
                    f"PROGRAM:\n{program_str}\n"
                ),
                encoding="utf-8",
            )

        except Exception:
            pass
'''


KSDD2_EOH_MODULE_PATH.write_text(
    textwrap.dedent(
        KSDD2_EOH_MODULE_SOURCE
    ),
    encoding="utf-8",
)


print(
    "KSDD2 EOH-8 problem module written:"
)

print(
    KSDD2_EOH_MODULE_PATH.resolve()
)

KSDD2 EOH-8 problem module written:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\eoh_runtime\ksdd2_eoh_problem_8.py


## 10.2 Local EOH-8 preflight

Before any API request is made, the local candidate-evaluation pipeline is tested.

The preflight verifies:

1. the template program compiles;
2. it returns exactly eight features;
3. all features are finite;
4. the template can be evaluated on the full KSDD2 development benchmark;
5. an invalid nine-feature candidate is rejected.

No LLM API request is made in these cells.

In [110]:
# -------------------------------------------------------
# Import KSDD2 EOH-8 problem
# -------------------------------------------------------

EOH8_TARGET_FEATURES = 8

EOH8_TIMEOUT_SECONDS = 180
EOH8_MAX_LOCAL_EVALUATION_SECONDS = 300.0

EOH8_POPULATION_SIZE = 2
EOH8_N_POPULATIONS = 1
EOH8_OPERATORS = ["e1"]

EOH8_NUM_SAMPLERS = 1
EOH8_NUM_EVALUATORS = 1


if str(EOH_RUNTIME_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(EOH_RUNTIME_DIR),
    )

if KSDD2_EOH_MODULE_NAME in sys.modules:
    del sys.modules[
        KSDD2_EOH_MODULE_NAME
    ]

importlib.invalidate_caches()

ksdd2_eoh_module = importlib.import_module(
    KSDD2_EOH_MODULE_NAME
)

KSDD2FeatureExtractionProblem8 = (
    ksdd2_eoh_module
    .KSDD2FeatureExtractionProblem8
)

eoh8_problem = (
    KSDD2FeatureExtractionProblem8(
        X_train=X_train,
        y_train=y_train,

        X_validation=X_validation,
        y_validation=y_validation,

        timeout=EOH8_TIMEOUT_SECONDS,

        n_processes=1,

        max_seconds_per_evaluation=(
            EOH8_MAX_LOCAL_EVALUATION_SECONDS
        ),
    )
)

print("=" * 70)
print("KSDD2 EOH-8 problem")
print("=" * 70)

print(
    "Module:",
    ksdd2_eoh_module.__file__,
)

print(
    "Problem class:",
    type(eoh8_problem).__name__,
)

print(
    "Required features:",
    eoh8_problem.N_FEATURES,
)

print(
    "Training array:",
    eoh8_problem.X_train.shape,
)

print(
    "Validation array:",
    eoh8_problem.X_validation.shape,
)

print("=" * 70)

assert (
    eoh8_problem.N_FEATURES
    == EOH8_TARGET_FEATURES
)

assert (
    eoh8_problem.X_train.shape
    == X_train.shape
)

assert (
    eoh8_problem.X_validation.shape
    == X_validation.shape
)

KSDD2 EOH-8 problem
Module: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\eoh_runtime\ksdd2_eoh_problem_8.py
Problem class: KSDD2FeatureExtractionProblem8
Required features: 8
Training array: (1864, 64, 64)
Validation array: (467, 64, 64)


In [111]:
# -------------------------------------------------------
# Compile EOH-8 template locally
# -------------------------------------------------------

template_namespace = {
    "np": np,
}


exec(
    compile(
        eoh8_problem.template_program,
        "<ksdd2_eoh8_template>",
        "exec",
    ),
    template_namespace,
)


template_function = (
    template_namespace[
        "extract_features"
    ]
)


template_vector = np.asarray(
    template_function(
        X_train[0]
    ),
    dtype=np.float64,
).reshape(-1)


template_fitness = (
    eoh8_problem.evaluate_program(
        eoh8_problem.template_program,
        template_function,
    )
)


print("=" * 70)
print("EOH-8 template preflight")
print("=" * 70)

print(
    "Template output shape:",
    template_vector.shape,
)

print(
    "Template vector:",
    template_vector,
)

print(
    "Template fitness:",
    template_fitness,
)


if template_fitness is not None:

    template_balanced_accuracy = (
        1.0 - template_fitness
    )

    print(
        "Template balanced accuracy:",
        f"{template_balanced_accuracy * 100:.2f}%",
    )


assert (
    template_vector.shape
    == (N_FEATURES,)
)

assert np.all(
    np.isfinite(template_vector)
)

assert isinstance(
    template_fitness,
    float,
)

assert np.isfinite(
    template_fitness
)

print("=" * 70)
print(
    "PASS: template candidate is valid."
)

EOH-8 template preflight
Template output shape: (8,)
Template vector: [0.15586495 0.0222256  0.01760342 0.01142915 0.01880729 0.17358398
 0.15283203 0.00205731]
Template fitness: 0.14573772092569093
Template balanced accuracy: 85.43%
PASS: template candidate is valid.


In [112]:
# -------------------------------------------------------
# Verify exact eight-feature enforcement
# -------------------------------------------------------

def invalid_nine_feature_function(
    image,
):
    image = np.asarray(
        image,
        dtype=float,
    )

    return np.array(
        [
            np.mean(image),
            np.std(image),
            np.min(image),
            np.max(image),
            np.median(image),
            np.percentile(image, 25),
            np.percentile(image, 75),
            np.mean(
                np.abs(
                    np.diff(
                        image,
                        axis=0,
                    )
                )
            ),
            np.mean(
                np.abs(
                    np.diff(
                        image,
                        axis=1,
                    )
                )
            ),
        ],
        dtype=float,
    )


invalid_fitness = (
    eoh8_problem.evaluate_program(
        (
            "Invalid local test candidate "
            "returning nine features."
        ),
        invalid_nine_feature_function,
    )
)


print(
    "Nine-feature candidate fitness:",
    invalid_fitness,
)


assert invalid_fitness is None


print(
    "PASS: candidates not returning exactly "
    "eight features are rejected."
)

Nine-feature candidate fitness: None
PASS: candidates not returning exactly eight features are rejected.


## 10.3 Configure DeepSeek securely

The DeepSeek API key is read only from the environment variable:

`DEEPSEEK_API_KEY`

The API key must never be pasted into the notebook, committed to GitHub or written to experiment output files.

Constructing the configuration and EOH runner does not itself make an API request.

In [113]:
# -------------------------------------------------------
# Secure DeepSeek configuration
# -------------------------------------------------------

deepseek_api_key = os.getenv(
    "DEEPSEEK_API_KEY"
)


if not deepseek_api_key:

    raise RuntimeError(
        "DEEPSEEK_API_KEY is not available in "
        "this notebook session.\n\n"
        "Set it as an environment variable, "
        "restart VS Code / the notebook kernel, "
        "and rerun this cell."
    )


eoh8_llm_config = LLMConfig(
    api_endpoint="api.deepseek.com",

    api_key=deepseek_api_key,

    model="deepseek-v4-flash",

    timeout=EOH8_TIMEOUT_SECONDS,
)


print("=" * 70)
print("EOH-8 LLM configuration")
print("=" * 70)

print(
    "Endpoint:",
    eoh8_llm_config.api_endpoint,
)

print(
    "Model:",
    eoh8_llm_config.model,
)

print(
    "Timeout:",
    eoh8_llm_config.timeout,
)

print(
    "API key loaded:",
    bool(eoh8_llm_config.api_key),
)

print(
    "API key value printed:",
    False,
)

print("=" * 70)

EOH-8 LLM configuration
Endpoint: api.deepseek.com
Model: deepseek-v4-flash
Timeout: 180
API key loaded: True
API key value printed: False


In [114]:
# -------------------------------------------------------
# Configure minimal EOH-8 runner
# -------------------------------------------------------

eoh8_runner = EoH(
    llm=eoh8_llm_config,

    problem=eoh8_problem,

    pop_size=EOH8_POPULATION_SIZE,

    n_pop=EOH8_N_POPULATIONS,

    operators=EOH8_OPERATORS,

    num_samplers=EOH8_NUM_SAMPLERS,

    num_evaluators=EOH8_NUM_EVALUATORS,
)


print("=" * 70)
print("Minimal KSDD2 EOH-8 runner")
print("=" * 70)

print(
    "Population size:",
    eoh8_runner._config.pop_size,
)

print(
    "Populations:",
    eoh8_runner._config.n_pop,
)

print(
    "Operators:",
    eoh8_runner._config.operators,
)

print(
    "Samplers:",
    EOH8_NUM_SAMPLERS,
)

print(
    "Evaluators:",
    EOH8_NUM_EVALUATORS,
)

print(
    "Correct problem attached:",
    eoh8_runner._problem
    is eoh8_problem,
)

print(
    "API request made:",
    False,
)

print("=" * 70)

Minimal KSDD2 EOH-8 runner
Population size: 2
Populations: 1
Operators: ['e1']
Samplers: 1
Evaluators: 1
Correct problem attached: True
API request made: False


In [115]:
# -------------------------------------------------------
# Final local EOH-8 preflight
# -------------------------------------------------------

eoh8_preflight_checks = {

    "API key loaded":
        bool(
            os.getenv(
                "DEEPSEEK_API_KEY"
            )
        ),

    "Correct endpoint":
        (
            eoh8_runner
            ._config
            .llm
            .api_endpoint
            == "api.deepseek.com"
        ),

    "Model configured":
        bool(
            eoh8_runner
            ._config
            .llm
            .model
        ),

    "Population size is 2":
        (
            eoh8_runner
            ._config
            .pop_size
            == 2
        ),

    "One population":
        (
            eoh8_runner
            ._config
            .n_pop
            == 1
        ),

    "Only E1 enabled":
        (
            eoh8_runner
            ._config
            .operators
            == ["e1"]
        ),

    "Correct problem attached":
        (
            eoh8_runner._problem
            is eoh8_problem
        ),

    "Required dimension is 8":
        (
            eoh8_problem.N_FEATURES
            == N_FEATURES
        ),

    "Template program parses":
        bool(
            ast.parse(
                eoh8_problem
                .template_program
            )
        ),

    "Problem module importable":
        (
            KSDD2FeatureExtractionProblem8
            .__module__
            == KSDD2_EOH_MODULE_NAME
        ),

    "Run method exists":
        callable(
            getattr(
                eoh8_runner,
                "run",
                None,
            )
        ),

    "Template candidate valid":
        (
            isinstance(
                template_fitness,
                float,
            )
            and np.isfinite(
                template_fitness
            )
        ),

    "Nine-feature candidate rejected":
        (
            invalid_fitness
            is None
        ),

    "Official test set locked":
        (
            TEST_SET_LOCKED
            is True
        ),
}


print("=" * 70)
print("EOH-8 FINAL LOCAL PREFLIGHT")
print("=" * 70)


for check_name, passed in (
    eoh8_preflight_checks.items()
):

    print(
        f"{check_name:<42}"
        f"{'PASS' if passed else 'FAIL'}"
    )


failed_checks = [
    name
    for name, passed
    in eoh8_preflight_checks.items()
    if not passed
]


if failed_checks:

    raise RuntimeError(
        "Do not start the EOH API run. "
        "Failed checks: "
        + ", ".join(
            failed_checks
        )
    )


print("=" * 70)

print(
    "All local EOH-8 checks passed."
)

print(
    "The paid search remains disabled."
)

EOH-8 FINAL LOCAL PREFLIGHT
API key loaded                            PASS
Correct endpoint                          PASS
Model configured                          PASS
Population size is 2                      PASS
One population                            PASS
Only E1 enabled                           PASS
Correct problem attached                  PASS
Required dimension is 8                   PASS
Template program parses                   PASS
Problem module importable                 PASS
Run method exists                         PASS
Template candidate valid                  PASS
Nine-feature candidate rejected           PASS
Official test set locked                  PASS
All local EOH-8 checks passed.
The paid search remains disabled.


## 10.4 Paid EOH-8 smoke test — disabled by default

The following cell is the first cell in this section that may make API requests and incur charges.

Leave:

`RUN_EOH8_PAID_SMOKE_TEST = False`

until all preceding preflight checks pass.

When deliberately ready to run EOH, change only this flag to `True`.

The official KSDD2 test set is not involved in the EOH search.

In [116]:
# -------------------------------------------------------
# PAID EOH-8 smoke test
# -------------------------------------------------------

RUN_EOH8_PAID_SMOKE_TEST = False


eoh8_search_result = None
eoh8_search_runtime_seconds = None


if RUN_EOH8_PAID_SMOKE_TEST:

    if not all(
        eoh8_preflight_checks.values()
    ):
        raise RuntimeError(
            "EOH preflight did not pass."
        )

    print("=" * 70)
    print("STARTING PAID KSDD2 EOH-8 SEARCH")
    print("=" * 70)

    print(
        "Population size:",
        eoh8_runner._config.pop_size,
    )

    print(
        "Populations:",
        eoh8_runner._config.n_pop,
    )

    print(
        "Operators:",
        eoh8_runner._config.operators,
    )

    print(
        "Official test images supplied: 0"
    )

    started = time.perf_counter()

    try:

        eoh8_search_result = (
            eoh8_runner.run()
        )

        eoh8_search_runtime_seconds = (
            time.perf_counter()
            - started
        )

        print(
            "\nEOH-8 search completed."
        )

        print(
            "Runtime:",
            f"{eoh8_search_runtime_seconds:.2f}",
            "seconds",
        )

        print(
            "Result type:",
            type(
                eoh8_search_result
            ).__name__,
        )

        print(
            "\nResult preview:"
        )

        print(
            repr(
                eoh8_search_result
            )[:5000]
        )

    finally:

        # Reset to safe state in memory.
        RUN_EOH8_PAID_SMOKE_TEST = False

        print(
            "\nSafety flag reset:"
        )

        print(
            "RUN_EOH8_PAID_SMOKE_TEST = False"
        )

else:

    print(
        "EOH-8 paid smoke test not started."
    )

    print(
        "No API request was made."
    )

STARTING PAID KSDD2 EOH-8 SEARCH
Population size: 2
Populations: 1
Operators: ['e1']
Official test images supplied: 0
[2026-08-21 00:55:02] LLM: deepseek-v4-flash @ api.deepseek.com
[2026-08-21 00:55:03] LLM connection verified.
[2026-08-21 00:55:03] ======================================================
[2026-08-21 00:55:03]   EoH
[2026-08-21 00:55:03]   LLM      : deepseek-v4-flash @ api.deepseek.com
[2026-08-21 00:55:03]   EC       : gen=1  pop=2  ops=[e1]
[2026-08-21 00:55:03]   Sampling : init=4 (2×pop)  evo_budget=2
[2026-08-21 00:55:03]   Pipeline : samplers=1  evaluators=1 (async)
[2026-08-21 00:55:03]   Timeout  : llm=180s  eval=180s
[2026-08-21 00:55:03] ======================================================
[2026-08-21 00:55:03] 
[Init]  (4 samples → pop=2)  samplers=1  evaluators=1
[2026-08-21 00:57:12]   #1    [i1]  0.1793            best=0.1793  *
[2026-08-21 00:59:09]   #2    [i1]  0.18113           best=0.1793
[2026-08-21 01:01:07]   #3    [i1]  0.18233           best=0

In [117]:
# -------------------------------------------------------
# Recover successful EOH-8 candidates from diagnostics
# -------------------------------------------------------

import re


EOH8_DIAGNOSTICS_DIR = EOH_DIAGNOSTICS_DIR


if not EOH8_DIAGNOSTICS_DIR.exists():
    raise FileNotFoundError(
        "EOH-8 diagnostics directory was not found:\n"
        f"{EOH8_DIAGNOSTICS_DIR}"
    )


successful_diagnostic_files = sorted(
    EOH8_DIAGNOSTICS_DIR.glob(
        "*_success.txt"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)


print("=" * 70)
print("EOH-8 DIAGNOSTIC RECOVERY")
print("=" * 70)

print(
    "Diagnostics directory:",
    EOH8_DIAGNOSTICS_DIR.resolve(),
)

print(
    "Successful diagnostic files found:",
    len(successful_diagnostic_files),
)

print("\nMost recent successful files:")

for path in successful_diagnostic_files[:10]:
    print(" ", path.name)

print("=" * 70)

EOH-8 DIAGNOSTIC RECOVERY
Diagnostics directory: C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_eoh_8\candidate_diagnostics
Successful diagnostic files found: 7

Most recent successful files:
  20260821_010631_489610_16640_success.txt
  20260821_010504_639620_3024_success.txt
  20260821_010249_972666_23548_success.txt
  20260821_010107_468448_24728_success.txt
  20260821_005908_918062_25372_success.txt
  20260821_005712_198537_21452_success.txt
  20260821_005420_686836_30468_success.txt


## 10.5 Load the selected EOH-8 program

After the EOH search:

1. inspect the generated candidates;
2. identify the best valid candidate using validation performance only;
3. paste its complete Python program into `BEST_EOH8_PROGRAM_TEXT`;
4. rerun the local validation cells below.

Do not use the official test set to choose the program.

In [118]:
# -------------------------------------------------------
# Parse successful EOH-8 diagnostic files
# -------------------------------------------------------

FITNESS_PATTERN = re.compile(
    r"fitness=([0-9]*\.?[0-9]+)"
)

BALANCED_ACCURACY_PATTERN = re.compile(
    r"balanced_accuracy=([0-9]*\.?[0-9]+)"
)


def parse_ksdd2_eoh_diagnostic(
    path: Path,
) -> dict:
    """
    Parse one successful KSDD2 EOH-8 diagnostic.
    """

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    if "STATUS: success" not in text:
        raise ValueError(
            f"{path.name} is not a successful candidate."
        )

    if "PROGRAM:" not in text:
        raise ValueError(
            f"No PROGRAM section found in {path.name}."
        )

    message, program_text = text.split(
        "PROGRAM:",
        maxsplit=1,
    )

    fitness_match = FITNESS_PATTERN.search(
        message
    )

    balanced_accuracy_match = (
        BALANCED_ACCURACY_PATTERN.search(
            message
        )
    )

    if fitness_match is None:
        raise ValueError(
            f"No fitness value found in {path.name}."
        )

    fitness = float(
        fitness_match.group(1)
    )

    if balanced_accuracy_match is not None:
        balanced_accuracy = float(
            balanced_accuracy_match.group(1)
        )
    else:
        balanced_accuracy = (
            1.0 - fitness
        )

    program_text = program_text.strip()

    if (
        "def extract_features"
        not in program_text
    ):
        raise ValueError(
            f"{path.name} does not contain "
            "extract_features()."
        )

    return {
        "diagnostic_path": path,
        "fitness": fitness,
        "balanced_accuracy": (
            balanced_accuracy
        ),
        "program_text": program_text,
        "modified_time": (
            path.stat().st_mtime
        ),
    }


parsed_eoh8_candidates = []


for path in successful_diagnostic_files:

    try:
        candidate = (
            parse_ksdd2_eoh_diagnostic(
                path
            )
        )

        parsed_eoh8_candidates.append(
            candidate
        )

    except ValueError as error:
        print(
            f"Skipping {path.name}: {error}"
        )


print(
    "Valid successful candidates parsed:",
    len(parsed_eoh8_candidates),
)

Valid successful candidates parsed: 7


In [119]:
# -------------------------------------------------------
# Isolate candidates from the most recent paid EOH run
# -------------------------------------------------------

NUMBER_OF_CANDIDATES_IN_PAID_RUN = 4


if (
    len(parsed_eoh8_candidates)
    < NUMBER_OF_CANDIDATES_IN_PAID_RUN
):
    raise RuntimeError(
        "Fewer successful candidates were found "
        "than expected from the paid run."
    )


recent_paid_candidates = sorted(
    parsed_eoh8_candidates,
    key=lambda item:
        item["modified_time"],
    reverse=True,
)[:NUMBER_OF_CANDIDATES_IN_PAID_RUN]


recent_paid_candidates = sorted(
    recent_paid_candidates,
    key=lambda item:
        item["fitness"],
)


paid_candidate_table = pd.DataFrame(
    [
        {
            "Candidate": index + 1,
            "Fitness": candidate[
                "fitness"
            ],
            "Balanced Accuracy": candidate[
                "balanced_accuracy"
            ],
            "Diagnostic File": candidate[
                "diagnostic_path"
            ].name,
        }

        for index, candidate
        in enumerate(
            recent_paid_candidates
        )
    ]
)


display(
    paid_candidate_table
)

,Candidate,Fitness,Balanced Accuracy,Diagnostic File
0,1,0.139684,0.860316,20260821_010249_972666_23548_success.txt
1,2,0.149326,0.850674,20260821_010631_489610_16640_success.txt
2,3,0.182331,0.817669,20260821_010107_468448_24728_success.txt
3,4,0.191900,0.808100,20260821_010504_639620_3024_success.txt


In [120]:
# -------------------------------------------------------
# Select best candidate using validation fitness only
# -------------------------------------------------------

best_eoh8_candidate = min(
    recent_paid_candidates,
    key=lambda item:
        item["fitness"],
)


BEST_EOH8_PROGRAM_TEXT = (
    best_eoh8_candidate[
        "program_text"
    ]
)


selected_eoh8_fitness_recorded = (
    best_eoh8_candidate[
        "fitness"
    ]
)


selected_eoh8_balanced_accuracy_recorded = (
    best_eoh8_candidate[
        "balanced_accuracy"
    ]
)


print("=" * 70)
print("SELECTED EOH-8 CANDIDATE")
print("=" * 70)

print(
    "Diagnostic file:",
    best_eoh8_candidate[
        "diagnostic_path"
    ].name,
)

print(
    "Recorded fitness:",
    selected_eoh8_fitness_recorded,
)

print(
    "Recorded balanced accuracy:",
    f"{selected_eoh8_balanced_accuracy_recorded * 100:.2f}%",
)

print(
    "Program characters:",
    len(BEST_EOH8_PROGRAM_TEXT),
)

print(
    "\nProgram preview:"
)

print(
    BEST_EOH8_PROGRAM_TEXT[:1500]
)

print("=" * 70)

SELECTED EOH-8 CANDIDATE
Diagnostic file: 20260821_010249_972666_23548_success.txt
Recorded fitness: 0.139684
Recorded balanced accuracy: 86.03%
Program characters: 1675

Program preview:
import numpy as np

def extract_features(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=float)

    mean_value = np.mean(image)
    std_value = np.std(image)

    dx = np.empty_like(image)
    dy = np.empty_like(image)

    dx[:, 1:-1] = image[:, 2:] - image[:, :-2]
    dx[:, 0] = image[:, 1] - image[:, 0]
    dx[:, -1] = image[:, -1] - image[:, -2]

    dy[1:-1, :] = image[2:, :] - image[:-2, :]
    dy[0, :] = image[1, :] - image[0, :]
    dy[-1, :] = image[-1, :] - image[-2, :]

    grad_magnitude = np.sqrt(dx * dx + dy * dy)
    mean_gradient = np.mean(grad_magnitude)
    gradient_std = np.std(grad_magnitude)

    dark_threshold = mean_value - std_value
    bright_threshold = mean_value + std_value

    dark_proportion = np.mean(image < dark_threshold)
    bright_proportion =

In [121]:
# -------------------------------------------------------
# Compile and validate recovered EOH-8 candidate
# -------------------------------------------------------

best_eoh8_namespace = {
    "np": np,
}


exec(
    compile(
        BEST_EOH8_PROGRAM_TEXT,
        "<best_ksdd2_eoh8_program>",
        "exec",
    ),
    best_eoh8_namespace,
)


best_eoh8_function = (
    best_eoh8_namespace.get(
        "extract_features"
    )
)


if not callable(
    best_eoh8_function
):
    raise RuntimeError(
        "Recovered EOH candidate does not "
        "define extract_features()."
    )


selected_vector = (
    validate_feature_vector(
        best_eoh8_function(
            X_train[0]
        ),
        image_index=0,
        expected_features=N_FEATURES,
        split_name="Recovered EOH-8 program",
    )
)


print("=" * 70)
print("RECOVERED EOH-8 PROGRAM")
print("=" * 70)

print(
    "Feature vector shape:",
    selected_vector.shape,
)

print(
    "Feature values:"
)

print(
    selected_vector
)

assert (
    selected_vector.shape
    == (N_FEATURES,)
)

assert np.all(
    np.isfinite(
        selected_vector
    )
)

print(
    "\nPASS: recovered EOH-8 program "
    "satisfies the eight-feature contract."
)

print("=" * 70)

RECOVERED EOH-8 PROGRAM
Feature vector shape: (8,)
Feature values:
[0.15586495 0.0222256  0.03112472 0.01768354 0.17358398 0.15283203
 0.00847447 0.00992847]

PASS: recovered EOH-8 program satisfies the eight-feature contract.


## 10.6 Reproduce the selected EOH-8 candidate locally

The recovered winning EOH program is re-evaluated using the same KSDD2 EOH problem used during the paid search.

This checks that:

- the recovered program is executable;
- it still returns exactly eight features;
- its saved search fitness can be reproduced;
- the correct candidate was recovered.

The official KSDD2 test set remains locked.

In [122]:
# -------------------------------------------------------
# Reproduce selected EOH-8 fitness
# -------------------------------------------------------

selected_eoh8_fitness = (
    eoh8_problem.evaluate_program(
        BEST_EOH8_PROGRAM_TEXT,
        best_eoh8_function,
    )
)


if selected_eoh8_fitness is None:
    raise RuntimeError(
        "Recovered EOH-8 program failed "
        "local re-evaluation."
    )


selected_eoh8_balanced_accuracy = (
    1.0
    - selected_eoh8_fitness
)


print("=" * 70)
print("EOH-8 LOCAL FITNESS REPRODUCTION")
print("=" * 70)

print(
    "Recorded search fitness:",
    selected_eoh8_fitness_recorded,
)

print(
    "Reproduced fitness:",
    selected_eoh8_fitness,
)

print(
    "Recorded balanced accuracy:",
    f"{selected_eoh8_balanced_accuracy_recorded * 100:.2f}%",
)

print(
    "Reproduced balanced accuracy:",
    f"{selected_eoh8_balanced_accuracy * 100:.2f}%",
)


fitness_difference = abs(
    selected_eoh8_fitness
    - selected_eoh8_fitness_recorded
)


print(
    "Absolute fitness difference:",
    fitness_difference,
)

print("=" * 70)

EOH-8 LOCAL FITNESS REPRODUCTION
Recorded search fitness: 0.139684
Reproduced fitness: 0.13968362464603068
Recorded balanced accuracy: 86.03%
Reproduced balanced accuracy: 86.03%
Absolute fitness difference: 3.7535396932297793e-07


In [123]:
# -------------------------------------------------------
# Validate EOH-8 feature matrices
# -------------------------------------------------------

eoh8_train_features, eoh8_train_timing = (
    build_feature_matrix(
        X_train,
        best_eoh8_function,
        split_name="EOH-8 training",
        expected_features=N_FEATURES,
    )
)


eoh8_validation_features, eoh8_validation_timing = (
    build_feature_matrix(
        X_validation,
        best_eoh8_function,
        split_name="EOH-8 validation",
        expected_features=N_FEATURES,
    )
)


validate_training_feature_matrix(
    eoh8_train_features,
    expected_features=N_FEATURES,
)


assert (
    eoh8_train_features.shape
    == (len(y_train), N_FEATURES)
)

assert (
    eoh8_validation_features.shape
    == (len(y_validation), N_FEATURES)
)

assert np.isfinite(
    eoh8_train_features
).all()

assert np.isfinite(
    eoh8_validation_features
).all()


print("=" * 70)
print("EOH-8 FEATURE MATRIX VALIDATION")
print("=" * 70)

print(
    "Training feature shape:",
    eoh8_train_features.shape,
)

print(
    "Validation feature shape:",
    eoh8_validation_features.shape,
)

print(
    "Training extraction:",
    f"{eoh8_train_timing['milliseconds_per_image']:.6f}",
    "ms/image",
)

print(
    "Validation extraction:",
    f"{eoh8_validation_timing['milliseconds_per_image']:.6f}",
    "ms/image",
)

print(
    "PASS: full EOH-8 feature matrices are valid."
)

print("=" * 70)

Training feature matrix passed validation: (1864, 8)
EOH-8 FEATURE MATRIX VALIDATION
Training feature shape: (1864, 8)
Validation feature shape: (467, 8)
Training extraction: 0.204181 ms/image
Validation extraction: 0.206867 ms/image
PASS: full EOH-8 feature matrices are valid.


## 10.7 Common KSDD2 benchmark evaluation

The selected EOH-8 extractor is now passed through the shared MARS benchmark pipeline.

This makes EOH-8 directly comparable with the other controlled methods because all methods use:

- exactly eight features;
- the same training and validation images;
- StandardScaler fitted only on training features;
- the same linear SVM;
- balanced class weighting;
- identical validation metrics;
- identical inference-timing methodology.

In [124]:
# -------------------------------------------------------
# Common benchmark evaluation of EOH-8
# -------------------------------------------------------

eoh8_result = (
    evaluate_feature_extractor(
        feature_extractor=best_eoh8_function,
        method_name="EOH-8",
        expected_features=N_FEATURES,
        comparison_group="dimension_controlled",
        evaluate_test=False,
    )
)

Training feature matrix passed validation: (1864, 8)
Method              : EOH-8
Comparison group    : dimension_controlled
Feature dimension   : 8
Accuracy            : 89.51%
Balanced accuracy   : 86.03%
Macro F1            : 0.7796
Defect precision    : 0.5000
Defect recall       : 0.8163
Inference estimate  : 0.1808 ms/image
Estimated FPS       : 5532.00
Test evaluated      : False


In [125]:
# -------------------------------------------------------
# EOH-8 result summary
# -------------------------------------------------------

print("=" * 70)
print("EOH-8 — KSDD2")
print("=" * 70)

print(
    "Feature dimension :",
    eoh8_result[
        "feature_dimension"
    ],
)

print(
    "Accuracy          :",
    f"{eoh8_result['validation_accuracy'] * 100:.2f}%",
)

print(
    "Balanced accuracy :",
    f"{eoh8_result['validation_balanced_accuracy'] * 100:.2f}%",
)

print(
    "Macro F1          :",
    f"{eoh8_result['validation_macro_f1']:.4f}",
)

print(
    "Defect precision  :",
    f"{eoh8_result['validation_defect_precision']:.4f}",
)

print(
    "Defect recall     :",
    f"{eoh8_result['validation_defect_recall']:.4f}",
)

print(
    "Feature extraction:",
    f"{eoh8_result['validation_feature_extraction_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "SVM prediction    :",
    f"{eoh8_result['validation_prediction_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "Total inference   :",
    f"{eoh8_result['estimated_inference_ms_per_image']:.6f}",
    "ms/image",
)

print(
    "Estimated FPS     :",
    f"{eoh8_result['estimated_fps']:.2f}",
)

print(
    "EOH search fitness:",
    f"{selected_eoh8_fitness:.6f}",
)

print(
    "Test evaluated    :",
    eoh8_result[
        "test_evaluated"
    ],
)

print("=" * 70)

EOH-8 — KSDD2
Feature dimension : 8
Accuracy          : 89.51%
Balanced accuracy : 86.03%
Macro F1          : 0.7796
Defect precision  : 0.5000
Defect recall     : 0.8163
Feature extraction: 0.173545 ms/image
SVM prediction    : 0.007222 ms/image
Total inference   : 0.180766 ms/image
Estimated FPS     : 5532.00
EOH search fitness: 0.139684
Test evaluated    : False


In [126]:
# -------------------------------------------------------
# EOH-8 validation confusion matrix
# -------------------------------------------------------

eoh8_confusion_table = pd.DataFrame(
    eoh8_result[
        "validation_confusion_matrix"
    ],
    index=[
        "Actual Normal",
        "Actual Defective",
    ],
    columns=[
        "Predicted Normal",
        "Predicted Defective",
    ],
)


display(
    eoh8_confusion_table
)

,Predicted Normal,Predicted Defective
Actual Normal,378,40
Actual Defective,9,40


## 10.8 Complete controlled eight-feature comparison

EOH-8 completes the main dimension-controlled experiment.

The four primary methods are:

1. Handcrafted-8;
2. Restricted Original GP-8;
3. Restricted Modified GP-8;
4. EOH-8.

All four produce exactly eight final features and are evaluated using the same downstream classification pipeline.

The unrestricted Original GP and Modified GP methods remain separate high-dimensional reference experiments.

In [127]:
# -------------------------------------------------------
# Complete controlled eight-feature comparison
# -------------------------------------------------------

controlled_comparison_four = pd.DataFrame(
    [
        {
            "Method": "Handcrafted-8",
            "Features": handcrafted_result[
                "feature_dimension"
            ],
            "Accuracy": handcrafted_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": handcrafted_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": handcrafted_result[
                "validation_macro_f1"
            ],
            "Defect Precision": handcrafted_result[
                "validation_defect_precision"
            ],
            "Defect Recall": handcrafted_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Original GP-8",
            "Features": restricted_original_result[
                "feature_dimension"
            ],
            "Accuracy": restricted_original_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": restricted_original_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_original_result[
                "validation_macro_f1"
            ],
            "Defect Precision": restricted_original_result[
                "validation_defect_precision"
            ],
            "Defect Recall": restricted_original_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Modified GP-8",
            "Features": restricted_modified_result[
                "feature_dimension"
            ],
            "Accuracy": restricted_modified_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": restricted_modified_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_modified_result[
                "validation_macro_f1"
            ],
            "Defect Precision": restricted_modified_result[
                "validation_defect_precision"
            ],
            "Defect Recall": restricted_modified_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "EOH-8",
            "Features": eoh8_result[
                "feature_dimension"
            ],
            "Accuracy": eoh8_result[
                "validation_accuracy"
            ],
            "Balanced Accuracy": eoh8_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": eoh8_result[
                "validation_macro_f1"
            ],
            "Defect Precision": eoh8_result[
                "validation_defect_precision"
            ],
            "Defect Recall": eoh8_result[
                "validation_defect_recall"
            ],
        },
    ]
)


controlled_comparison_four = (
    controlled_comparison_four
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    controlled_comparison_four
)

,Method,Features,Accuracy,Balanced Accuracy,Macro F1,Defect Precision,Defect Recall
0,EOH-8,8,0.895075,0.860316,0.779643,0.500000,0.816327
1,Restricted Original GP-8,8,0.899358,0.835685,0.776880,0.513889,0.755102
2,Handcrafted-8,8,0.912206,0.833854,0.793615,0.562500,0.734694
3,Restricted Modified GP-8,8,0.882227,0.790084,0.738902,0.458333,0.673469


In [128]:
# -------------------------------------------------------
# Full development-stage comparison
# -------------------------------------------------------

full_development_comparison = pd.DataFrame(
    [
        {
            "Method": "Handcrafted-8",
            "Group": "Controlled 8-feature",
            "Features": handcrafted_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": handcrafted_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": handcrafted_result[
                "validation_macro_f1"
            ],
            "Defect Recall": handcrafted_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Original GP-8",
            "Group": "Controlled 8-feature",
            "Features": restricted_original_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": restricted_original_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_original_result[
                "validation_macro_f1"
            ],
            "Defect Recall": restricted_original_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Restricted Modified GP-8",
            "Group": "Controlled 8-feature",
            "Features": restricted_modified_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": restricted_modified_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": restricted_modified_result[
                "validation_macro_f1"
            ],
            "Defect Recall": restricted_modified_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "EOH-8",
            "Group": "Controlled 8-feature",
            "Features": eoh8_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": eoh8_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": eoh8_result[
                "validation_macro_f1"
            ],
            "Defect Recall": eoh8_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Original GP — Unrestricted",
            "Group": "Unrestricted reference",
            "Features": original_gp_smoke_benchmark_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": original_gp_smoke_benchmark_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": original_gp_smoke_benchmark_result[
                "validation_macro_f1"
            ],
            "Defect Recall": original_gp_smoke_benchmark_result[
                "validation_defect_recall"
            ],
        },

        {
            "Method": "Modified GP — Unrestricted",
            "Group": "Unrestricted reference",
            "Features": unrestricted_modified_result[
                "feature_dimension"
            ],
            "Balanced Accuracy": unrestricted_modified_result[
                "validation_balanced_accuracy"
            ],
            "Macro F1": unrestricted_modified_result[
                "validation_macro_f1"
            ],
            "Defect Recall": unrestricted_modified_result[
                "validation_defect_recall"
            ],
        },
    ]
)


full_development_comparison = (
    full_development_comparison
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    full_development_comparison
)

,Method,Group,Features,Balanced Accuracy,Macro F1,Defect Recall
0,EOH-8,Controlled 8-feature,8,0.860316,0.779643,0.816327
1,Restricted Original GP-8,Controlled 8-feature,8,0.835685,0.776880,0.755102
2,Handcrafted-8,Controlled 8-feature,8,0.833854,0.793615,0.734694
3,Original GP — Unrestricted,Unrestricted reference,162,0.819573,0.720851,0.775510
4,Modified GP — Unrestricted,Unrestricted reference,266,0.808100,0.746007,0.714286
5,Restricted Modified GP-8,Controlled 8-feature,8,0.790084,0.738902,0.673469


In [129]:
# -------------------------------------------------------
# Save selected EOH-8 program
# -------------------------------------------------------

EOH8_PROGRAM_FILE = (
    OUTPUT_DIR
    / "eoh8_selected_program.py"
)


EOH8_PROGRAM_FILE.write_text(
    BEST_EOH8_PROGRAM_TEXT,
    encoding="utf-8",
)


print(
    "Saved selected EOH-8 program:"
)

print(
    EOH8_PROGRAM_FILE
)

Saved selected EOH-8 program:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\eoh8_selected_program.py


In [130]:
# -------------------------------------------------------
# Save EOH-8 validation summary
# -------------------------------------------------------

EOH8_SUMMARY_FILE = (
    OUTPUT_DIR
    / "eoh8_validation_summary.json"
)


eoh8_summary = (
    summarise_benchmark_result(
        eoh8_result
    )
)


eoh8_summary.update(
    {
        "run_type":
            "minimal_paid_eoh_search",

        "search_fitness_recorded":
            float(
                selected_eoh8_fitness_recorded
            ),

        "search_fitness_reproduced":
            float(
                selected_eoh8_fitness
            ),

        "search_balanced_accuracy":
            float(
                selected_eoh8_balanced_accuracy
            ),

        "population_size":
            EOH8_POPULATION_SIZE,

        "n_populations":
            EOH8_N_POPULATIONS,

        "operators":
            EOH8_OPERATORS,

        "num_samplers":
            EOH8_NUM_SAMPLERS,

        "num_evaluators":
            EOH8_NUM_EVALUATORS,

        "llm_model":
            eoh8_llm_config.model,

        "llm_endpoint":
            eoh8_llm_config.api_endpoint,

        "selected_diagnostic_file":
            best_eoh8_candidate[
                "diagnostic_path"
            ].name,

        "official_test_evaluated":
            False,
    }
)


with EOH8_SUMMARY_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        eoh8_summary,
        file,
        indent=2,
    )


print(
    "Saved EOH-8 validation summary:"
)

print(
    EOH8_SUMMARY_FILE
)

Saved EOH-8 validation summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\eoh8_validation_summary.json


In [131]:
# -------------------------------------------------------
# Save development comparison tables
# -------------------------------------------------------

CONTROLLED_COMPARISON_FILE = (
    OUTPUT_DIR
    / "controlled_8_feature_comparison.csv"
)

FULL_COMPARISON_FILE = (
    OUTPUT_DIR
    / "full_development_comparison.csv"
)


controlled_comparison_four.to_csv(
    CONTROLLED_COMPARISON_FILE,
    index=False,
)

full_development_comparison.to_csv(
    FULL_COMPARISON_FILE,
    index=False,
)


print(
    "Saved controlled comparison:"
)

print(
    CONTROLLED_COMPARISON_FILE
)

print()

print(
    "Saved full development comparison:"
)

print(
    FULL_COMPARISON_FILE
)

Saved controlled comparison:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\controlled_8_feature_comparison.csv

Saved full development comparison:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\full_development_comparison.csv


## 10.9 Development-stage conclusion

The complete KSDD2 development benchmark now contains four controlled eight-feature methods and two unrestricted GP reference methods.

The official KSDD2 test set has remained locked throughout feature development and model comparison.

At this stage:

- Handcrafted-8 provides the deterministic baseline;
- Restricted Original GP-8 provides the original GP comparison under the eight-feature constraint;
- Restricted Modified GP-8 tests the modified GP search space under the same constraint;
- EOH-8 tests LLM-generated feature extraction under the same eight-feature budget;
- unrestricted Original and Modified GP results provide high-dimensional reference performance.

All current performance values are development-validation results rather than final generalisation estimates.

The next stage is to standardise inference timing across all methods, decide whether additional stochastic runs or cross-validation are feasible, and only then perform final evaluation on the locked official KSDD2 test set.

# 11. Unified Inference Timing Benchmark

This section measures the runtime required to classify one previously unseen
preprocessed image after each feature extractor and classifier have already
been trained.

The timing benchmark is designed to address real-time deployment feasibility.

For each image, the timed inference pipeline is:

1. feature extraction;
2. removal of any training-constant GP feature columns;
3. StandardScaler transformation;
4. linear SVM prediction.

Evolutionary search, LLM generation and classifier training are excluded
because these are offline operations rather than per-frame inference costs.

All methods are benchmarked:

- on the same KSDD2 validation images;
- sequentially, one image at a time;
- after warm-up;
- across repeated timing passes;
- using the already-selected development-stage feature extractors.

The benchmark reports:

- mean feature-extraction time per image;
- mean scaling time per image;
- mean SVM prediction time per image;
- mean total inference time per image;
- median and standard deviation of total inference time;
- estimated frames per second (FPS).

Disk I/O is excluded. The input to this benchmark is the already-preprocessed
64 × 64 grayscale frame.

In [132]:
# -------------------------------------------------------
# Unified timing configuration
# -------------------------------------------------------

from deap import gp


TIMING_WARMUP_IMAGES = 25

TIMING_REPETITIONS = 5

TIMING_IMAGES = X_validation


assert TIMING_IMAGES.ndim == 3
assert TIMING_IMAGES.shape[1:] == IMAGE_SIZE
assert len(TIMING_IMAGES) > TIMING_WARMUP_IMAGES


print("=" * 70)
print("UNIFIED INFERENCE TIMING CONFIGURATION")
print("=" * 70)

print(
    "Validation images:",
    len(TIMING_IMAGES),
)

print(
    "Image size:",
    IMAGE_SIZE,
)

print(
    "Warm-up images:",
    TIMING_WARMUP_IMAGES,
)

print(
    "Timing repetitions:",
    TIMING_REPETITIONS,
)

print(
    "Total timed image inferences per method:",
    len(TIMING_IMAGES)
    * TIMING_REPETITIONS,
)

print(
    "Disk I/O included:",
    False,
)

print(
    "Training/search time included:",
    False,
)

print("=" * 70)

UNIFIED INFERENCE TIMING CONFIGURATION
Validation images: 467
Image size: (64, 64)
Warm-up images: 25
Timing repetitions: 5
Total timed image inferences per method: 2335
Disk I/O included: False
Training/search time included: False


In [133]:
# -------------------------------------------------------
# Compile a saved GP program without reparsing typed terminals
# -------------------------------------------------------

def compile_gp_program(
    program_text: str,
    pset,
):
    """
    Reconstruct a saved GP program as a callable function.

    The GP result stores the winning individual as text. Re-parsing that
    text with gp.PrimitiveTree.from_string() can fail for typed ephemeral
    constants because numeric literals such as 4 are read back as ordinary
    Python ints rather than the custom Int1/Int2/Int3 terminal types.

    Instead, evaluate the already-valid GP expression directly using the
    primitive set's runtime context. This mirrors the final evaluation step
    performed internally by DEAP's gp.compile().
    """

    if not isinstance(
        program_text,
        str,
    ):
        raise TypeError(
            "GP program must be supplied as text."
        )

    program_text = program_text.strip()

    if not program_text:
        raise ValueError(
            "GP program text is empty."
        )

    if not hasattr(
        pset,
        "context",
    ):
        raise TypeError(
            "The supplied primitive set does not "
            "contain a runtime context."
        )

    if not hasattr(
        pset,
        "arguments",
    ):
        raise TypeError(
            "The supplied primitive set does not "
            "define GP arguments."
        )

    arguments = ", ".join(
        pset.arguments
    )

    lambda_source = (
        f"lambda {arguments}: {program_text}"
    )

    try:
        function = eval(
            lambda_source,
            pset.context,
            {},
        )

    except Exception as error:
        raise RuntimeError(
            "Failed to compile saved GP program.\n"
            f"Error: {type(error).__name__}: {error}"
        ) from error

    if not callable(function):
        raise TypeError(
            "Compiled GP program is not callable."
        )

    return function

In [134]:
# -------------------------------------------------------
# GP inference wrapper
# -------------------------------------------------------

def make_gp_feature_extractor(
    gp_function,
    raw_training_features,
    expected_final_dimension,
    *,
    method_name,
):
    """
    Wrap one compiled GP program for deployment-style inference.

    Constant columns are identified using TRAINING features only.
    The same fixed column mask is subsequently used for every unseen image.
    """

    raw_training_matrix = np.asarray(
        raw_training_features,
        dtype=np.float64,
    )

    if raw_training_matrix.ndim != 2:
        raise ValueError(
            f"{method_name}: raw training features must be 2D."
        )

    training_std = np.std(
        raw_training_matrix,
        axis=0,
    )

    keep_columns = ~np.isclose(
        training_std,
        0.0,
    )

    retained_dimension = int(
        np.sum(keep_columns)
    )

    if (
        retained_dimension
        != expected_final_dimension
    ):
        raise ValueError(
            f"{method_name}: retained dimension "
            f"{retained_dimension} does not match "
            f"benchmark dimension "
            f"{expected_final_dimension}."
        )

    def extractor(
        image: np.ndarray,
    ) -> np.ndarray:

        raw_vector = np.asarray(
            gp_function(image),
            dtype=np.float64,
        ).reshape(-1)

        if (
            raw_vector.shape[0]
            != raw_training_matrix.shape[1]
        ):
            raise ValueError(
                f"{method_name}: GP output dimension changed "
                f"from {raw_training_matrix.shape[1]} "
                f"to {raw_vector.shape[0]}."
            )

        if not np.all(
            np.isfinite(raw_vector)
        ):
            raise ValueError(
                f"{method_name}: GP produced "
                "non-finite features."
            )

        return raw_vector[
            keep_columns
        ]

    return extractor, keep_columns

In [136]:
# -------------------------------------------------------
# Reconstruct Restricted Original GP-8 extractor
# -------------------------------------------------------

restricted_original_pset = (
    restricted_original_smoke_raw[
        "pset"
    ]
)


restricted_original_gp_function = (
    compile_gp_program(
        restricted_original_smoke_raw[
            "program"
        ],
        restricted_original_pset,
    )
)


(
    timed_restricted_original_extractor,
    restricted_original_keep_columns,
) = make_gp_feature_extractor(
    gp_function=(
        restricted_original_gp_function
    ),

    raw_training_features=(
        restricted_original_smoke_raw[
            "train_features"
        ]
    ),

    expected_final_dimension=(
        restricted_original_result[
            "feature_dimension"
        ]
    ),

    method_name=(
        "Restricted Original GP-8"
    ),
)


sample_vector = (
    timed_restricted_original_extractor(
        X_validation[0]
    )
)


print(
    "Restricted Original GP-8 output:",
    sample_vector.shape,
)

assert (
    sample_vector.shape
    == (N_FEATURES,)
)

Restricted Original GP-8 output: (8,)


In [137]:
# -------------------------------------------------------
# Reconstruct Restricted Modified GP-8 extractor
# -------------------------------------------------------

restricted_modified_pset = (
    restricted_modified_smoke_raw[
        "pset"
    ]
)


restricted_modified_gp_function = (
    compile_gp_program(
        restricted_modified_smoke_raw[
            "program"
        ],
        restricted_modified_pset,
    )
)


(
    timed_restricted_modified_extractor,
    restricted_modified_keep_columns,
) = make_gp_feature_extractor(
    gp_function=(
        restricted_modified_gp_function
    ),

    raw_training_features=(
        restricted_modified_smoke_raw[
            "train_features"
        ]
    ),

    expected_final_dimension=(
        restricted_modified_result[
            "feature_dimension"
        ]
    ),

    method_name=(
        "Restricted Modified GP-8"
    ),
)


sample_vector = (
    timed_restricted_modified_extractor(
        X_validation[0]
    )
)


print(
    "Restricted Modified GP-8 output:",
    sample_vector.shape,
)

assert (
    sample_vector.shape
    == (N_FEATURES,)
)

Restricted Modified GP-8 output: (8,)


In [138]:
# -------------------------------------------------------
# Reconstruct Unrestricted Original GP extractor
# -------------------------------------------------------

original_gp_for_timing = load_gp_fr(
    ORIGINAL_GP_PARENT
)


original_timing_pset = (
    original_gp_for_timing
    .gp_fr_main
    .build_pset(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
    )
)


original_gp_function_for_timing = (
    compile_gp_program(
        original_gp_smoke_result[
            "program"
        ],
        original_timing_pset,
    )
)


(
    timed_original_unrestricted_extractor,
    original_unrestricted_keep_columns,
) = make_gp_feature_extractor(
    gp_function=(
        original_gp_function_for_timing
    ),

    raw_training_features=(
        original_gp_smoke_result[
            "train_features"
        ]
    ),

    expected_final_dimension=(
        original_gp_smoke_benchmark_result[
            "feature_dimension"
        ]
    ),

    method_name=(
        "Original GP — Unrestricted"
    ),
)


sample_vector = (
    timed_original_unrestricted_extractor(
        X_validation[0]
    )
)


print(
    "Original GP unrestricted usable output:",
    sample_vector.shape,
)

Original GP unrestricted usable output: (162,)


In [139]:
# -------------------------------------------------------
# Reconstruct Unrestricted Modified GP extractor
# -------------------------------------------------------

modified_gp_for_timing = load_gp_fr(
    MODIFIED_GP_PARENT
)


modified_timing_pset = (
    modified_gp_for_timing
    .gp_fr_main
    .build_pset(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
    )
)


modified_gp_function_for_timing = (
    compile_gp_program(
        unrestricted_modified_smoke_raw[
            "program"
        ],
        modified_timing_pset,
    )
)


(
    timed_modified_unrestricted_extractor,
    modified_unrestricted_keep_columns,
) = make_gp_feature_extractor(
    gp_function=(
        modified_gp_function_for_timing
    ),

    raw_training_features=(
        unrestricted_modified_smoke_raw[
            "train_features"
        ]
    ),

    expected_final_dimension=(
        unrestricted_modified_result[
            "feature_dimension"
        ]
    ),

    method_name=(
        "Modified GP — Unrestricted"
    ),
)


sample_vector = (
    timed_modified_unrestricted_extractor(
        X_validation[0]
    )
)


print(
    "Modified GP unrestricted usable output:",
    sample_vector.shape,
)

Modified GP unrestricted usable output: (266,)


In [140]:
# -------------------------------------------------------
# Unified deployment-method registry
# -------------------------------------------------------

timing_methods = {
    "Handcrafted-8": {
        "extractor":
            handcrafted_feature_extractor,

        "scaler":
            handcrafted_result[
                "scaler"
            ],

        "classifier":
            handcrafted_result[
                "classifier"
            ],

        "features":
            handcrafted_result[
                "feature_dimension"
            ],
    },

    "Restricted Original GP-8": {
        "extractor":
            timed_restricted_original_extractor,

        "scaler":
            restricted_original_result[
                "scaler"
            ],

        "classifier":
            restricted_original_result[
                "classifier"
            ],

        "features":
            restricted_original_result[
                "feature_dimension"
            ],
    },

    "Restricted Modified GP-8": {
        "extractor":
            timed_restricted_modified_extractor,

        "scaler":
            restricted_modified_result[
                "scaler"
            ],

        "classifier":
            restricted_modified_result[
                "classifier"
            ],

        "features":
            restricted_modified_result[
                "feature_dimension"
            ],
    },

    "EOH-8": {
        "extractor":
            best_eoh8_function,

        "scaler":
            eoh8_result[
                "scaler"
            ],

        "classifier":
            eoh8_result[
                "classifier"
            ],

        "features":
            eoh8_result[
                "feature_dimension"
            ],
    },

    "Original GP — Unrestricted": {
        "extractor":
            timed_original_unrestricted_extractor,

        "scaler":
            original_gp_smoke_benchmark_result[
                "scaler"
            ],

        "classifier":
            original_gp_smoke_benchmark_result[
                "classifier"
            ],

        "features":
            original_gp_smoke_benchmark_result[
                "feature_dimension"
            ],
    },

    "Modified GP — Unrestricted": {
        "extractor":
            timed_modified_unrestricted_extractor,

        "scaler":
            unrestricted_modified_result[
                "scaler"
            ],

        "classifier":
            unrestricted_modified_result[
                "classifier"
            ],

        "features":
            unrestricted_modified_result[
                "feature_dimension"
            ],
    },
}


print(
    "Timing methods registered:",
    len(timing_methods),
)

for method_name, configuration in (
    timing_methods.items()
):

    print(
        f"{method_name:<32}"
        f"{configuration['features']:>5} features"
    )

Timing methods registered: 6
Handcrafted-8                       8 features
Restricted Original GP-8            8 features
Restricted Modified GP-8            8 features
EOH-8                               8 features
Original GP — Unrestricted        162 features
Modified GP — Unrestricted        266 features


In [141]:
# -------------------------------------------------------
# Verify every inference pipeline
# -------------------------------------------------------

def validate_inference_pipeline(
    method_name,
    configuration,
    sample_image,
):
    """
    Run one complete untimed inference and verify dimensions.
    """

    features = np.asarray(
        configuration[
            "extractor"
        ](
            sample_image
        ),
        dtype=np.float64,
    ).reshape(1, -1)

    expected_features = (
        configuration[
            "features"
        ]
    )

    if (
        features.shape
        != (1, expected_features)
    ):
        raise ValueError(
            f"{method_name}: expected "
            f"(1, {expected_features}), "
            f"received {features.shape}."
        )

    scaled = configuration[
        "scaler"
    ].transform(
        features
    )

    prediction = configuration[
        "classifier"
    ].predict(
        scaled
    )

    if prediction.shape != (1,):
        raise ValueError(
            f"{method_name}: unexpected "
            "prediction shape."
        )

    return int(
        prediction[0]
    )


print("=" * 70)
print("TIMING PIPELINE PREFLIGHT")
print("=" * 70)


for method_name, configuration in (
    timing_methods.items()
):

    prediction = (
        validate_inference_pipeline(
            method_name,
            configuration,
            X_validation[0],
        )
    )

    print(
        f"{method_name:<32}"
        f"PASS  prediction={prediction}"
    )


print("=" * 70)
print(
    "All inference pipelines passed."
)

TIMING PIPELINE PREFLIGHT
Handcrafted-8                   PASS  prediction=0
Restricted Original GP-8        PASS  prediction=0
Restricted Modified GP-8        PASS  prediction=0
EOH-8                           PASS  prediction=1
Original GP — Unrestricted      PASS  prediction=0
Modified GP — Unrestricted      PASS  prediction=0
All inference pipelines passed.


In [142]:
# -------------------------------------------------------
# Serial per-image inference timer
# -------------------------------------------------------

def benchmark_inference_method(
    method_name,
    configuration,
    images,
    *,
    warmup_images=25,
    repetitions=5,
):
    """
    Measure deployment-style serial inference.

    Each image is processed individually:

        image
          -> feature extraction
          -> scaling
          -> SVM prediction

    Compilation, training and search are excluded.
    """

    images = np.asarray(
        images
    )

    extractor = configuration[
        "extractor"
    ]

    scaler = configuration[
        "scaler"
    ]

    classifier = configuration[
        "classifier"
    ]

    expected_features = int(
        configuration[
            "features"
        ]
    )

    # ---------------------------------------------------
    # Warm-up
    # ---------------------------------------------------

    warmup_count = min(
        warmup_images,
        len(images),
    )

    for image in images[
        :warmup_count
    ]:

        features = np.asarray(
            extractor(image),
            dtype=np.float64,
        ).reshape(1, -1)

        if (
            features.shape[1]
            != expected_features
        ):
            raise ValueError(
                f"{method_name}: feature dimension "
                "changed during warm-up."
            )

        scaled = scaler.transform(
            features
        )

        classifier.predict(
            scaled
        )

    # ---------------------------------------------------
    # Timed repetitions
    # ---------------------------------------------------

    feature_times = []
    scaling_times = []
    prediction_times = []
    total_times = []

    predictions = []

    for repetition in range(
        repetitions
    ):

        repetition_predictions = []

        for image in images:

            total_started = (
                time.perf_counter_ns()
            )

            # Feature extraction
            feature_started = (
                time.perf_counter_ns()
            )

            features = np.asarray(
                extractor(image),
                dtype=np.float64,
            ).reshape(1, -1)

            feature_finished = (
                time.perf_counter_ns()
            )

            if (
                features.shape[1]
                != expected_features
            ):
                raise ValueError(
                    f"{method_name}: expected "
                    f"{expected_features} features, "
                    f"received "
                    f"{features.shape[1]}."
                )

            if not np.all(
                np.isfinite(features)
            ):
                raise ValueError(
                    f"{method_name}: non-finite "
                    "inference features."
                )

            # Scaling
            scaling_started = (
                time.perf_counter_ns()
            )

            scaled = scaler.transform(
                features
            )

            scaling_finished = (
                time.perf_counter_ns()
            )

            # Prediction
            prediction_started = (
                time.perf_counter_ns()
            )

            prediction = (
                classifier.predict(
                    scaled
                )
            )

            prediction_finished = (
                time.perf_counter_ns()
            )

            total_finished = (
                prediction_finished
            )

            repetition_predictions.append(
                int(prediction[0])
            )

            feature_times.append(
                (
                    feature_finished
                    - feature_started
                )
                / 1_000_000.0
            )

            scaling_times.append(
                (
                    scaling_finished
                    - scaling_started
                )
                / 1_000_000.0
            )

            prediction_times.append(
                (
                    prediction_finished
                    - prediction_started
                )
                / 1_000_000.0
            )

            total_times.append(
                (
                    total_finished
                    - total_started
                )
                / 1_000_000.0
            )

        predictions.append(
            repetition_predictions
        )

    # ---------------------------------------------------
    # Determinism check
    # ---------------------------------------------------

    reference_predictions = np.asarray(
        predictions[0]
    )

    for repetition_predictions in (
        predictions[1:]
    ):

        if not np.array_equal(
            reference_predictions,
            np.asarray(
                repetition_predictions
            ),
        ):
            raise RuntimeError(
                f"{method_name}: predictions "
                "changed between timing repetitions."
            )

    feature_times = np.asarray(
        feature_times,
        dtype=np.float64,
    )

    scaling_times = np.asarray(
        scaling_times,
        dtype=np.float64,
    )

    prediction_times = np.asarray(
        prediction_times,
        dtype=np.float64,
    )

    total_times = np.asarray(
        total_times,
        dtype=np.float64,
    )

    mean_total_ms = float(
        np.mean(total_times)
    )

    result = {
        "method":
            method_name,

        "feature_dimension":
            expected_features,

        "n_images":
            int(len(images)),

        "repetitions":
            int(repetitions),

        "timed_inferences":
            int(
                len(total_times)
            ),

        "feature_extraction_mean_ms":
            float(
                np.mean(feature_times)
            ),

        "feature_extraction_median_ms":
            float(
                np.median(feature_times)
            ),

        "scaling_mean_ms":
            float(
                np.mean(scaling_times)
            ),

        "prediction_mean_ms":
            float(
                np.mean(prediction_times)
            ),

        "total_mean_ms":
            mean_total_ms,

        "total_median_ms":
            float(
                np.median(total_times)
            ),

        "total_std_ms":
            float(
                np.std(
                    total_times,
                    ddof=1,
                )
            ),

        "total_p95_ms":
            float(
                np.percentile(
                    total_times,
                    95,
                )
            ),

        "estimated_fps":
            float(
                1000.0
                / mean_total_ms
            )
            if mean_total_ms > 0
            else float("inf"),

        "predictions":
            reference_predictions,
    }

    return result

## 11.1 Execute the unified timing experiment

Each selected feature extractor is timed on the same validation images.

Timing runs are sequential rather than batched because the intended use case is
frame-by-frame processing of a live video stream.

The feature extractor and trained classifier are kept fixed throughout the
experiment.

In [143]:
# -------------------------------------------------------
# Execute unified timing benchmark
# -------------------------------------------------------

timing_results = {}


for method_name, configuration in (
    timing_methods.items()
):

    print("=" * 70)

    print(
        "Timing:",
        method_name,
    )

    print("=" * 70)

    started = time.perf_counter()

    timing_result = (
        benchmark_inference_method(
            method_name=method_name,
            configuration=configuration,
            images=TIMING_IMAGES,
            warmup_images=(
                TIMING_WARMUP_IMAGES
            ),
            repetitions=(
                TIMING_REPETITIONS
            ),
        )
    )

    wall_time = (
        time.perf_counter()
        - started
    )

    timing_results[
        method_name
    ] = timing_result

    print(
        "Feature extraction:",
        f"{timing_result['feature_extraction_mean_ms']:.4f}",
        "ms/image",
    )

    print(
        "Scaling:",
        f"{timing_result['scaling_mean_ms']:.4f}",
        "ms/image",
    )

    print(
        "Prediction:",
        f"{timing_result['prediction_mean_ms']:.4f}",
        "ms/image",
    )

    print(
        "Total inference:",
        f"{timing_result['total_mean_ms']:.4f}",
        "ms/image",
    )

    print(
        "Estimated FPS:",
        f"{timing_result['estimated_fps']:.2f}",
    )

    print(
        "Timing wall time:",
        f"{wall_time:.2f}",
        "seconds",
    )

    print()

Timing: Handcrafted-8
Feature extraction: 0.1938 ms/image
Scaling: 0.0755 ms/image
Prediction: 0.0905 ms/image
Total inference: 0.3635 ms/image
Estimated FPS: 2751.25
Timing wall time: 0.88 seconds

Timing: Restricted Original GP-8
Feature extraction: 0.8050 ms/image
Scaling: 0.1034 ms/image
Prediction: 0.1067 ms/image
Total inference: 1.0191 ms/image
Estimated FPS: 981.28
Timing wall time: 2.42 seconds

Timing: Restricted Modified GP-8
Feature extraction: 4.1436 ms/image
Scaling: 0.1395 ms/image
Prediction: 0.1310 ms/image
Total inference: 4.4186 ms/image
Estimated FPS: 226.32
Timing wall time: 10.44 seconds

Timing: EOH-8
Feature extraction: 0.2275 ms/image
Scaling: 0.0790 ms/image
Prediction: 0.0875 ms/image
Total inference: 0.4009 ms/image
Estimated FPS: 2494.32
Timing wall time: 0.95 seconds

Timing: Original GP — Unrestricted
Feature extraction: 2.7736 ms/image
Scaling: 0.1248 ms/image
Prediction: 0.1373 ms/image
Total inference: 3.0401 ms/image
Estimated FPS: 328.94
Timing wall 

In [144]:
# -------------------------------------------------------
# Unified timing comparison table
# -------------------------------------------------------

timing_summary = pd.DataFrame(
    [
        {
            "Method":
                result[
                    "method"
                ],

            "Features":
                result[
                    "feature_dimension"
                ],

            "Feature Extraction ms":
                result[
                    "feature_extraction_mean_ms"
                ],

            "Scaling ms":
                result[
                    "scaling_mean_ms"
                ],

            "Prediction ms":
                result[
                    "prediction_mean_ms"
                ],

            "Total Inference ms":
                result[
                    "total_mean_ms"
                ],

            "Median ms":
                result[
                    "total_median_ms"
                ],

            "Std ms":
                result[
                    "total_std_ms"
                ],

            "95th Percentile ms":
                result[
                    "total_p95_ms"
                ],

            "Estimated FPS":
                result[
                    "estimated_fps"
                ],
        }

        for result
        in timing_results.values()
    ]
)


timing_summary = (
    timing_summary
    .sort_values(
        "Total Inference ms",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    timing_summary
)

,Method,Features,Feature Extraction ms,Scaling ms,Prediction ms,Total Inference ms,Median ms,Std ms,95th Percentile ms,Estimated FPS
0,Handcrafted-8,8,0.193777,0.075545,0.090494,0.363471,0.2899,0.186589,0.72359,2751.249462
1,EOH-8,8,0.227539,0.079045,0.087543,0.400911,0.3270,0.252113,0.77096,2494.319148
2,Restricted Original GP-8,8,0.804963,0.103386,0.106709,1.019080,0.8614,0.355951,1.73632,981.277519
3,Modified GP — Unrestricted,266,2.605070,0.127938,0.170437,2.908023,2.5784,0.754265,4.60639,343.876230
4,Original GP — Unrestricted,162,2.773602,0.124799,0.137302,3.040068,2.6603,0.998438,4.77981,328.940047
5,Restricted Modified GP-8,8,4.143564,0.139461,0.130959,4.418612,4.0418,1.171209,6.44860,226.315429


In [145]:
# -------------------------------------------------------
# Prediction consistency checks
# -------------------------------------------------------

expected_validation_predictions = {
    "Handcrafted-8":
        handcrafted_result[
            "validation_predictions"
        ],

    "Restricted Original GP-8":
        restricted_original_result[
            "validation_predictions"
        ],

    "Restricted Modified GP-8":
        restricted_modified_result[
            "validation_predictions"
        ],

    "EOH-8":
        eoh8_result[
            "validation_predictions"
        ],

    "Original GP — Unrestricted":
        original_gp_smoke_benchmark_result[
            "validation_predictions"
        ],

    "Modified GP — Unrestricted":
        unrestricted_modified_result[
            "validation_predictions"
        ],
}


print("=" * 70)
print("TIMING PREDICTION CONSISTENCY")
print("=" * 70)


for method_name in timing_methods:

    timed_predictions = (
        timing_results[
            method_name
        ][
            "predictions"
        ]
    )

    expected_predictions = np.asarray(
        expected_validation_predictions[
            method_name
        ]
    )

    match = np.array_equal(
        timed_predictions,
        expected_predictions,
    )

    print(
        f"{method_name:<32}"
        f"{'PASS' if match else 'FAIL'}"
    )

    if not match:
        raise RuntimeError(
            f"{method_name}: timed predictions "
            "do not reproduce benchmark predictions."
        )


print("=" * 70)
print(
    "PASS: all timing pipelines reproduce "
    "their original validation predictions."
)

TIMING PREDICTION CONSISTENCY
Handcrafted-8                   PASS
Restricted Original GP-8        PASS
Restricted Modified GP-8        PASS
EOH-8                           PASS
Original GP — Unrestricted      PASS
Modified GP — Unrestricted      PASS
PASS: all timing pipelines reproduce their original validation predictions.


In [146]:
# -------------------------------------------------------
# Performance-versus-speed comparison
# -------------------------------------------------------

performance_lookup = {
    "Handcrafted-8":
        handcrafted_result,

    "Restricted Original GP-8":
        restricted_original_result,

    "Restricted Modified GP-8":
        restricted_modified_result,

    "EOH-8":
        eoh8_result,

    "Original GP — Unrestricted":
        original_gp_smoke_benchmark_result,

    "Modified GP — Unrestricted":
        unrestricted_modified_result,
}


performance_speed_rows = []


for method_name, timing_result in (
    timing_results.items()
):

    performance_result = (
        performance_lookup[
            method_name
        ]
    )

    performance_speed_rows.append(
        {
            "Method":
                method_name,

            "Features":
                timing_result[
                    "feature_dimension"
                ],

            "Balanced Accuracy":
                performance_result[
                    "validation_balanced_accuracy"
                ],

            "Macro F1":
                performance_result[
                    "validation_macro_f1"
                ],

            "Defect Recall":
                performance_result[
                    "validation_defect_recall"
                ],

            "Inference ms/image":
                timing_result[
                    "total_mean_ms"
                ],

            "95th Percentile ms":
                timing_result[
                    "total_p95_ms"
                ],

            "Estimated FPS":
                timing_result[
                    "estimated_fps"
                ],
        }
    )


performance_speed_comparison = (
    pd.DataFrame(
        performance_speed_rows
    )
    .sort_values(
        "Balanced Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    performance_speed_comparison
)

,Method,Features,Balanced Accuracy,Macro F1,Defect Recall,Inference ms/image,95th Percentile ms,Estimated FPS
0,EOH-8,8,0.860316,0.779643,0.816327,0.400911,0.77096,2494.319148
1,Restricted Original GP-8,8,0.835685,0.776880,0.755102,1.019080,1.73632,981.277519
2,Handcrafted-8,8,0.833854,0.793615,0.734694,0.363471,0.72359,2751.249462
3,Original GP — Unrestricted,162,0.819573,0.720851,0.775510,3.040068,4.77981,328.940047
4,Modified GP — Unrestricted,266,0.808100,0.746007,0.714286,2.908023,4.60639,343.876230
5,Restricted Modified GP-8,8,0.790084,0.738902,0.673469,4.418612,6.44860,226.315429


In [147]:
# -------------------------------------------------------
# Nominal real-time FPS thresholds
# -------------------------------------------------------

REALTIME_FPS_THRESHOLDS = [
    10,
    15,
    24,
    30,
    60,
]


realtime_rows = []


for method_name, result in (
    timing_results.items()
):

    row = {
        "Method":
            method_name,

        "Estimated FPS":
            result[
                "estimated_fps"
            ],
    }

    for threshold in (
        REALTIME_FPS_THRESHOLDS
    ):

        row[
            f">= {threshold} FPS"
        ] = (
            result[
                "estimated_fps"
            ]
            >= threshold
        )

    realtime_rows.append(
        row
    )


realtime_feasibility = pd.DataFrame(
    realtime_rows
)


display(
    realtime_feasibility
)

,Method,Estimated FPS,>= 10 FPS,>= 15 FPS,>= 24 FPS,>= 30 FPS,>= 60 FPS
0,Handcrafted-8,2751.249462,True,True,True,True,True
1,Restricted Original GP-8,981.277519,True,True,True,True,True
2,Restricted Modified GP-8,226.315429,True,True,True,True,True
3,EOH-8,2494.319148,True,True,True,True,True
4,Original GP — Unrestricted,328.940047,True,True,True,True,True
5,Modified GP — Unrestricted,343.876230,True,True,True,True,True


In [148]:
# -------------------------------------------------------
# Save timing tables
# -------------------------------------------------------

TIMING_SUMMARY_FILE = (
    OUTPUT_DIR
    / "unified_inference_timing.csv"
)

PERFORMANCE_SPEED_FILE = (
    OUTPUT_DIR
    / "performance_speed_comparison.csv"
)

REALTIME_FEASIBILITY_FILE = (
    OUTPUT_DIR
    / "realtime_fps_feasibility.csv"
)


timing_summary.to_csv(
    TIMING_SUMMARY_FILE,
    index=False,
)

performance_speed_comparison.to_csv(
    PERFORMANCE_SPEED_FILE,
    index=False,
)

realtime_feasibility.to_csv(
    REALTIME_FEASIBILITY_FILE,
    index=False,
)


print(
    "Saved timing summary:"
)

print(
    TIMING_SUMMARY_FILE
)

print()

print(
    "Saved performance/speed comparison:"
)

print(
    PERFORMANCE_SPEED_FILE
)

print()

print(
    "Saved FPS feasibility table:"
)

print(
    REALTIME_FEASIBILITY_FILE
)

Saved timing summary:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\unified_inference_timing.csv

Saved performance/speed comparison:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\performance_speed_comparison.csv

Saved FPS feasibility table:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\realtime_fps_feasibility.csv


In [149]:
# -------------------------------------------------------
# Save timing experiment metadata
# -------------------------------------------------------

TIMING_METADATA_FILE = (
    OUTPUT_DIR
    / "inference_timing_metadata.json"
)


timing_metadata = {
    "dataset":
        "KolektorSDD2",

    "benchmark_version":
        "v1",

    "image_size":
        list(
            IMAGE_SIZE
        ),

    "input_representation":
        "preprocessed grayscale float32",

    "disk_io_included":
        False,

    "feature_extraction_included":
        True,

    "scaling_included":
        True,

    "classifier_prediction_included":
        True,

    "search_time_included":
        False,

    "classifier_training_included":
        False,

    "serial_frame_processing":
        True,

    "warmup_images":
        TIMING_WARMUP_IMAGES,

    "repetitions":
        TIMING_REPETITIONS,

    "images_per_repetition":
        int(
            len(TIMING_IMAGES)
        ),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "results": [
        {
            key: value

            for key, value
            in result.items()

            if key
            != "predictions"
        }

        for result
        in timing_results.values()
    ],

    "official_test_evaluated":
        False,
}


with TIMING_METADATA_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        timing_metadata,
        file,
        indent=2,
    )


print(
    "Saved timing metadata:"
)

print(
    TIMING_METADATA_FILE
)

Saved timing metadata:
C:\Users\james\OneDrive - Lancaster University\Documents\Projects\MARS-Summer-Research\results\ksdd2_benchmark\inference_timing_metadata.json


## 11.2 Timing benchmark interpretation

The unified timing experiment measures serial CPU inference from an already
preprocessed 64 × 64 grayscale image.

The reported inference time includes:

- feature extraction;
- any fixed GP feature-column selection learned from the training set;
- feature standardisation;
- linear SVM prediction.

The timing does not include:

- image acquisition;
- disk I/O;
- RGB-to-grayscale conversion;
- resizing to 64 × 64;
- evolutionary GP search;
- EOH/LLM generation;
- classifier training.

Estimated FPS is calculated as:

`FPS = 1000 / mean inference time in milliseconds`.

The estimate represents the theoretical throughput of one sequential
inference pipeline on the machine used for this experiment. Actual live-video
throughput may be lower because camera capture, preprocessing, operating-system
scheduling and other application overheads are not included.

The official KSDD2 test set remains locked throughout this timing experiment.